In [1]:
import os
import glob
import json

print("=== KAGGLE INPUT CHECK ===")

markers = glob.glob(
    "/kaggle/input/**/pscab_q1/RESULTS_MARKER.json",
    recursive=True
)

if not markers:
    print("❌ NO previous PS-CAB output found in /kaggle/input/")
else:
    print(f"✅ Found {len(markers)} previous output marker(s):")
    
    for marker in markers:
        folder = os.path.dirname(marker)
        run_files = glob.glob(os.path.join(folder, "runs", "*.json"))
        
        print("\nOutput folder:")
        print(folder)
        print("Run records:", len(run_files))
        
        try:
            with open(marker) as f:
                print("Marker:", json.load(f))
        except Exception as e:
            print("Marker read error:", e)

        if run_files:
            print("Recorded runs:")
            for p in sorted(run_files):
                print("  -", os.path.basename(p))

=== KAGGLE INPUT CHECK ===
✅ Found 2 previous output marker(s):

Output folder:
/kaggle/input/notebooks/panjanagalaxmi/notebook80ed48e07f/pscab_q1
Run records: 15
Marker: {'updated': '2026-10-12 06:18:51'}
Recorded runs:
  - baseline_s123.json
  - baseline_s2024.json
  - baseline_s42.json
  - pscab_all_tokens_s123.json
  - pscab_all_tokens_s2024.json
  - pscab_all_tokens_s42.json
  - pscab_no_gamma_s42.json
  - pscab_no_gate_s42.json
  - pscab_random_mask_s123.json
  - pscab_random_mask_s2024.json
  - pscab_random_mask_s42.json
  - pscab_s123.json
  - pscab_s2024.json
  - pscab_s42.json
  - zeroshot_mbart50.json

Output folder:
/kaggle/input/notebooks/nagalakshmi00998y/1-10runs-pscab-zip/pscab_q1
Run records: 11
Marker: {'updated': '2026-10-11 03:28:24'}
Recorded runs:
  - baseline_s123.json
  - baseline_s2024.json
  - baseline_s42.json
  - pscab_all_tokens_s42.json
  - pscab_no_gamma_s42.json
  - pscab_no_gate_s42.json
  - pscab_random_mask_s42.json
  - pscab_s123.json
  - pscab_s2024

In [4]:
"""
================================================================================
PART 1 / 3 (v3) -- CORE LIBRARY + DATA + MAIN EXPERIMENT (baseline + PS-CAB, seed 42)
English -> Telugu NMT, facebook/mbart-large-50-many-to-many-mmt
================================================================================
v3 CHANGES (environment-drift fix + paper figures):

  E1  ENVIRONMENT PIN: transformers / sacrebleu are pinned at the very top of this
      cell (before transformers is imported). The run aborts if the pin cannot be met,
      so no two runs are ever decoded with different library versions again.
  E2  DRIFT DIAGNOSTIC (runs once, cached): output-length ratio / BP / BLEU / TER of
      every run grouped by library version, and (if the pscab_s42 .h5 exists) the
      old-environment pscab_s42 model re-decoded in the current environment.
      Figures + table go to the appendix (DIAG_*).
  E3  SUPERSEDE: completed runs produced under a different transformers version are
      moved to runs_superseded/ (kept for transparency) and re-queued automatically.
      CV records without an env field are back-filled with the session's logged version
      (they were produced in a 5.16.1 session) -- set CFG['env']['backfill_cv_env']=False
      to re-run them instead.
  E4  RESTORE picks the NEWEST previous output (marker timestamp), not the one with
      the most run records (superseding lowers the count).
  E5  Every run record stores the env AND the effective generation configuration.
  E6  New figures: dataset statistics (D_fig_*), validation figures (V_fig_*).

RUN THIS CELL FIRST IN *EVERY* KAGGLE SESSION. It defines every function used by
Parts 2 and 3. It is fully resumable: finished work is detected on disk and skipped.

What this part does (first session):
  1. Restores results from a previous session (if attached as an input).
  2. Builds a NEW frozen split (new seed -> a test set nobody has looked at),
     with group-aware splitting over exact AND near-duplicates (MinHash-LSH on
     BOTH English and Telugu, no skipped buckets), then a cross-split leakage
     audit that re-splits automatically until it passes.
  3. Freezes optional external test sets (FLORES+ devtest; your own CSVs).
  4. Zero-shot mBART-50 reference (no fine-tuning).
  5. Trains baseline_s42 and pscab_s42 with an IDENTICAL step budget
     (no wall-clock truncation: a run either completes or is not started).
  6. Validation-only analysis report (the test set is NEVER scored before Part 3;
     test predictions are generated with frozen models and stored blind).

Protocol fixes vs. the previous notebook (see review):
  - Step-matched training for every condition (was: time-truncated, queue-biased).
  - Deterministic, per-sentence random-mask control on content tokens only
    (was: re-sampled every forward pass, included special tokens).
  - New control: pscab_all_tokens (same module on every content token) to test
    whether *preposition targeting* matters or only the extra capacity.
  - Corpus-level paired bootstrap CIs + approximate-randomization p-values on
    BLEU/chrF++ (was: mean sentence-BLEU).
  - Gate analysis compared against the UNTRAINED gate (init control) + gamma and
    bias/hidden norm ratio logged; attention-mass analysis; figures saved as PDF+PNG.

KAGGLE WORKFLOW (because of 12 h / GPU limits):
  Session 1 : Part 1 cell               ("Save Version" -> "Save & Run All")
  Session 2+: attach the previous version's OUTPUT as an input to the notebook,
              run Part 1 cell (quick: everything is resumed) then Part 2 cell.
              Repeat Part 2 until it says the main queue is complete.
  Then      : Part 1 cell + Part 3 cell (cross-validation, 2 runs per session);
              when CV and the main queue are complete, Part 3 does the final
              test evaluation, statistics, tables, figures and report.
================================================================================
"""

# ==============================================================================
# 00. ENVIRONMENT PIN  (must execute before transformers is imported)
# ==============================================================================
ENV_PIN = {"transformers": "5.16.1", "sacrebleu": "2.6.0"}
import importlib.metadata as _md
import subprocess as _sp0
import sys as _sys0
for _pkg, _ver in ENV_PIN.items():
    try:
        _cur = _md.version(_pkg)
    except _md.PackageNotFoundError:
        _cur = None
    if _cur != _ver:
        if _pkg in _sys0.modules:
            raise RuntimeError(f"{_pkg} {_cur} is already imported in this kernel. Restart the kernel and run "
                               f"this cell FIRST so {_pkg}=={_ver} can be installed before import.")
        print(f"[pin] installing {_pkg}=={_ver} (found {_cur})")
        _sp0.run([_sys0.executable, "-m", "pip", "install", "-q", f"{_pkg}=={_ver}"], check=True)

# ==============================================================================
# 0. BOOTSTRAP (small packages only; never touches torch/numpy versions)
# ==============================================================================
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "0"
os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")
os.environ["TOKENIZERS_PARALLELISM"] = "false"

import importlib, importlib.util, subprocess, sys as _sys
_REQ = {"sacrebleu": "sacrebleu", "sentencepiece": "sentencepiece", "h5py": "h5py"}
_miss = [p for m, p in _REQ.items() if importlib.util.find_spec(m) is None]
if _miss:
    print(f"[bootstrap] installing {_miss}")
    _r = subprocess.run([_sys.executable, "-m", "pip", "install", "-q", *_miss])
    if _r.returncode != 0:
        print(f"[bootstrap] pip install failed for {_miss}; continuing (install them manually if needed).")
    importlib.invalidate_caches()
if importlib.util.find_spec("spacy") is not None:
    try:
        import spacy as _sp
        _sp.load("en_core_web_sm")
    except Exception:
        try:
            subprocess.run([_sys.executable, "-m", "spacy", "download", "en_core_web_sm"], check=True)
            importlib.invalidate_caches()
        except Exception as _e:
            print(f"[bootstrap] spaCy model download failed ({_e}); fallback list will be used.")

# ==============================================================================
# 1. IMPORTS
# ==============================================================================
import re, gc, json, math, time, glob, shutil, random, zlib, hashlib, unicodedata, platform
from collections import defaultdict

import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset

import sacrebleu
from sacrebleu.metrics import BLEU, CHRF, TER
from transformers import MBartForConditionalGeneration, MBart50TokenizerFast

try:
    get_ipython().run_line_magic("matplotlib", "inline")
except Exception:
    pass
try:
    import spacy
    _SPACY_OK = True
except Exception:
    _SPACY_OK = False
try:
    import h5py
    _H5_OK = True
except Exception:
    _H5_OK = False

import transformers as _tf
from transformers.utils import logging as _hf_logging
_hf_logging.set_verbosity_error()          # silences the repeated max_length/max_new_tokens notice
if _tf.__version__ != ENV_PIN["transformers"]:
    raise RuntimeError(f"transformers {_tf.__version__} != pinned {ENV_PIN['transformers']}. Restart the kernel; "
                       f"this cell must be the first thing executed in the session.")

SESSION_START = time.time()

# ==============================================================================
# 2. CONFIGURATION  (edit here only)
# ==============================================================================
CFG = {
    "paths": {
        "raw_csv": "/kaggle/input/datasets/nagalakshmi00998y/bpcccfold/bpcc_telugu_fixed100k samples.csv",
        "results_dir": "/kaggle/working/pscab_q1",
        "previous_results_dir": None,   # optional explicit path; else auto-searched in /kaggle/input
        "en_col": None, "te_col": None,
    },
    "data": {
        "max_selected_pairs": 40000,
        "select_seed": 2026,             # NEW seed -> new, never-inspected test set
        "split_seed": 2026,
        "split": {"train": 0.8, "val": 0.1, "test": 0.1},
        "near_dup_threshold": 0.8,       # Jaccard on char-5-grams, EN and TE separately
        "minhash_perms": 128, "minhash_bands": 16,
    },
    "external": {
        "try_flores_plus": True,         # needs Internet + HF_TOKEN (Kaggle secret) with access granted
        "csvs": {},                      # e.g. {"in22_gen": "/kaggle/input/.../in22_gen_en_te.csv"} (cols en,te)
    },
    "model": {"backbone": "facebook/mbart-large-50-many-to-many-mmt", "src_lang": "en_XX",
              "tgt_lang": "te_IN", "max_source_length": 128, "max_target_length": 128},
    "optim": {
        "lr": 5e-5, "pscab_lr": 1e-3, "weight_decay": 0.01, "warmup_frac": 0.10,
        "label_smoothing": 0.1, "clip": 1.0, "fp16": True,
        "train_batch_size": 4, "grad_accum": 16,        # effective batch 64
        "max_steps": 3000,                              # IDENTICAL for every condition (= 6 epochs of 32k)
        "eval_every": 250, "patience_evals": 4, "min_delta": 0.002,
        "eval_batch_size": 16,
    },
    "decoding": {"num_beams": 4, "max_new_tokens": 128},
    "experiment": {
        "seeds": [42, 123, 2024],
        "mask_seed": 13,
        "robustness_subset": 2000, "diag_subset": 1000, "attention_subset": 300,
        "save_models_for": ["pscab_s42", "baseline_s42", "pscab_random_mask_s123", "pscab_all_tokens_s123"],
        "export_h5_for": ["pscab_s42"],
    },
    "session": {"limit_hours": 11.6, "default_run_hours": 3.6},
    "env": {
        "supersede_other_env": True,     # move runs from another transformers version aside and re-run them
        "backfill_cv_env": True,         # CV records written before v3 lack 'env'; they ran under 5.16.1
        "drift_regenerate": True,        # re-decode the old pscab_s42 (.h5) in this env for the appendix (once)
    },
    "cv": {"k": 5, "models": ["baseline", "pscab"], "seed": 42, "inner_val_size": 2000},
}

R = CFG["paths"]["results_dir"]
DIRS = {k: f"{R}/{k}" for k in ["data", "runs", "preds", "logs", "figures", "tables",
                                   "analysis", "models", "cv"]}
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

# Conditions. All PS-CAB variants are the identity function at step 0.
CONDITIONS = {
    "baseline":          {"use_pscab": False, "gate": "learned", "gamma": "learned", "mask": "true"},
    "pscab":             {"use_pscab": True,  "gate": "learned", "gamma": "learned", "mask": "true"},
    "pscab_no_gate":     {"use_pscab": True,  "gate": "none",    "gamma": "learned", "mask": "true"},
    "pscab_no_gamma":    {"use_pscab": True,  "gate": "learned", "gamma": "fixed",   "mask": "true"},
    "pscab_random_mask": {"use_pscab": True,  "gate": "learned", "gamma": "learned", "mask": "random"},
    "pscab_all_tokens":  {"use_pscab": True,  "gate": "learned", "gamma": "learned", "mask": "all"},
}
ABLATIONS = ["pscab_random_mask", "pscab_all_tokens", "pscab_no_gate", "pscab_no_gamma"]


def build_main_queue():
    """Prioritise completing random-mask and all-token controls across all seeds."""
    s = CFG["experiment"]["seeds"]

    q = [("baseline", s[0]), ("pscab", s[0])]
    q += [(a, s[0]) for a in ABLATIONS]

    for sd in s[1:]:
        q += [("baseline", sd), ("pscab", sd)]

    q += [("pscab_random_mask", s[1]),
          ("pscab_all_tokens", s[1])]
    q += [("pscab_random_mask", s[2]),
          ("pscab_all_tokens", s[2])]

    q += [("pscab_no_gate", s[1]),
          ("pscab_no_gamma", s[1])]
    q += [("pscab_no_gate", s[2]),
          ("pscab_no_gamma", s[2])]

    drop = {"pscab_no_gate_s123", "pscab_no_gamma_s123",
            "pscab_no_gate_s2024", "pscab_no_gamma_s2024"}
    return [f"{c}_s{sd}" for c, sd in q if f"{c}_s{sd}" not in drop]
def parse_run_id(run_id):
    """'pscab_no_gate_s42' -> ('pscab_no_gate', 42, None); 'cv3_pscab_s42' -> ('pscab', 42, 3)."""
    fold = None
    m = re.match(r"^cv(\d+)_(.+)$", run_id)
    if m:
        fold, run_id = int(m.group(1)), m.group(2)
    cond, seed = run_id.rsplit("_s", 1)
    return cond, int(seed), fold


# ==============================================================================
# 3. PERSISTENCE ACROSS KAGGLE SESSIONS
# ==============================================================================
def _find_previous_results():
    if CFG["paths"]["previous_results_dir"]:
        return CFG["paths"]["previous_results_dir"]
    cands = []
    for depth in range(1, 6):
        pat = "/kaggle/input/" + "/".join(["*"] * depth) + "/pscab_q1/RESULTS_MARKER.json"
        cands += glob.glob(pat)
    if not cands:
        return None

    def key(marker):
        try:
            with open(marker) as f:
                t = json.load(f).get("updated", "")
        except Exception:
            t = ""
        return (t, len(glob.glob(os.path.join(os.path.dirname(marker), "runs", "*.json"))))

    best = max(cands, key=key)
    print(f"[restore] {len(cands)} previous output(s) found; newest = {os.path.dirname(best)} ({key(best)[0]})")
    return os.path.dirname(best)


def restore_previous_results():
    os.makedirs(R, exist_ok=True)
    fresh = not glob.glob(f"{R}/runs/*.json") and not glob.glob(f"{R}/runs_superseded/*.json")
    prev = _find_previous_results()
    if prev and fresh and os.path.abspath(prev) != os.path.abspath(R):
        print(f"[restore] copying previous results from {prev} "
              f"({len(glob.glob(f'{prev}/runs/*.json'))} run records) ...")
        shutil.copytree(prev, R, dirs_exist_ok=True)
    for d in DIRS.values():
        os.makedirs(d, exist_ok=True)
    os.makedirs(f"{R}/runs_superseded", exist_ok=True)
    with open(f"{R}/RESULTS_MARKER.json", "w") as f:
        json.dump({"updated": time.strftime("%Y-%m-%d %H:%M:%S")}, f)


def jdump(obj, path):
    with open(path, "w", encoding="utf-8") as f:
        json.dump(obj, f, indent=2, ensure_ascii=False, default=_json_default)


def _json_default(o):
    if isinstance(o, (np.integer,)):
        return int(o)
    if isinstance(o, (np.floating,)):
        return float(o)
    if isinstance(o, np.ndarray):
        return o.tolist()
    return str(o)


def jload(path):
    if not os.path.exists(path):
        return None
    with open(path, encoding="utf-8") as f:
        return json.load(f)


def read_csv_str(path):
    return pd.read_csv(path, dtype=str, keep_default_na=False)


def run_record_path(run_id):
    return f"{DIRS['runs']}/{run_id}.json"


def run_done(run_id):
    r = jload(run_record_path(run_id))
    return bool(r) and r.get("status") == "complete"


def session_hours_left():
    return CFG["session"]["limit_hours"] - (time.time() - SESSION_START) / 3600


def estimated_run_hours():
    hrs = [r["wall_hours_total"] for r in (jload(p) for p in glob.glob(f"{DIRS['runs']}/*.json"))
           if r and r.get("status") == "complete" and r.get("wall_hours_total") and r["kind"] != "zeroshot"]
    return (max(hrs) * 1.1) if hrs else CFG["session"]["default_run_hours"]


# ==============================================================================
# 4. SMALL UTILITIES
# ==============================================================================
def set_all_seeds(seed):
    random.seed(seed); np.random.seed(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)
    torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


def free_gpu():
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()


def amp_ctx():
    return torch.autocast(device_type="cuda", dtype=torch.float16,
                          enabled=torch.cuda.is_available() and CFG["optim"]["fp16"])


def make_scaler():
    en = torch.cuda.is_available() and CFG["optim"]["fp16"]
    try:
        return torch.amp.GradScaler("cuda", enabled=en)
    except Exception:
        return torch.cuda.amp.GradScaler(enabled=en)


def environment_info():
    info = {"python": platform.python_version(), "torch": torch.__version__,
            "cuda": torch.version.cuda, "gpu": torch.cuda.get_device_name(0) if torch.cuda.is_available() else None,
            "sacrebleu": sacrebleu.__version__, "spacy": _SPACY_OK, "env_pin": ENV_PIN}
    try:
        import transformers
        info["transformers"] = transformers.__version__
    except Exception:
        pass
    return info


def save_fig(name):
    plt.tight_layout()
    for ext in ("pdf", "png"):
        plt.savefig(f"{DIRS['figures']}/{name}.{ext}", dpi=300 if ext == "png" else None, bbox_inches="tight")
    plt.show()
    plt.close()


def save_table(df, name, caption=""):
    df.to_csv(f"{DIRS['tables']}/{name}.csv", index=False)
    try:
        with open(f"{DIRS['tables']}/{name}.tex", "w", encoding="utf-8") as f:
            f.write(df.to_latex(index=False, float_format="%.2f", caption=caption, label=f"tab:{name}"))
    except Exception:
        pass


# ==============================================================================
# 5. DATA: load, clean, exact dedup
# ==============================================================================
TELUGU_RE = re.compile(r"[\u0C00-\u0C7F]")
LATIN_RE = re.compile(r"[A-Za-z]")
URL_RE = re.compile(r"https?://|www\.")
HTML_RE = re.compile(r"<[^>]+>")
LATIN_WORD_RE = re.compile(r"\b[A-Za-z]{2,}\b")


def normalize_text(s):
    return re.sub(r"\s+", " ", unicodedata.normalize("NFC", str(s))).strip()


def normkey(s):
    return re.sub(r"\s+", " ", unicodedata.normalize("NFC", str(s)).lower()).strip()


def locate_raw_csv():
    p = CFG["paths"]["raw_csv"]
    if os.path.exists(p):
        return p
    hits = [h for h in glob.glob("/kaggle/input/**/*.csv", recursive=True)
            if "bpcc" in h.lower() and "tel" in h.lower()]
    if hits:
        print(f"[data] configured path missing; using {hits[0]}")
        return hits[0]
    raise FileNotFoundError(f"Dataset not found at {p!r}; set CFG['paths']['raw_csv'].")


def load_and_clean():
    path = locate_raw_csv()
    df = pd.read_csv(path, encoding="utf-8-sig", keep_default_na=False, na_values=[""], low_memory=False)
    print(f"[data] loaded {path}: {df.shape}, columns={list(df.columns)}")
    te_col = CFG["paths"]["te_col"] or max(
        df.columns, key=lambda c: df[c].astype(str).head(1000).apply(
            lambda x: len(TELUGU_RE.findall(x)) / max(len(x), 1)).mean())
    rest = [c for c in df.columns if c != te_col]
    en_col = CFG["paths"]["en_col"] or max(
        rest, key=lambda c: df[c].astype(str).head(1000).apply(
            lambda x: len(LATIN_RE.findall(x)) / max(len(x), 1)).mean())
    print(f"[data] English column={en_col!r}, Telugu column={te_col!r}")
    df = df.rename(columns={en_col: "en", te_col: "te"})[["en", "te"]].copy()
    n0 = len(df)
    df["en"] = df["en"].fillna("").astype(str).apply(normalize_text)
    df["te"] = df["te"].fillna("").astype(str).apply(normalize_text)
    reasons = {}
    checks = [
        ("empty", (df["en"] == "") | (df["te"] == "")),
        ("invalid_unicode", df["en"].str.contains("\ufffd") | df["te"].str.contains("\ufffd")),
        ("no_telugu_in_target", ~df["te"].apply(lambda s: bool(TELUGU_RE.search(s)))),
        ("no_latin_in_source", ~df["en"].apply(lambda s: bool(LATIN_RE.search(s)))),
        ("html", df["en"].str.contains(HTML_RE) | df["te"].str.contains(HTML_RE)),
        ("url_only", df["en"].apply(lambda s: bool(URL_RE.search(s)) and len(s.split()) <= 3)),
    ]
    keep = pd.Series(True, index=df.index)
    for name, m in checks:
        reasons[name] = int((m & keep).sum())
        keep &= ~m
    df = df[keep].reset_index(drop=True)
    n_clean = len(df)
    key = df["en"].apply(normkey) + " ||| " + df["te"].apply(normkey)
    dup = key.duplicated(keep="first")
    df = df[~dup].reset_index(drop=True)
    report = {"original_rows": n0, "removed_by_rule": reasons, "rows_after_cleaning": n_clean,
              "exact_duplicate_pairs_removed": int(dup.sum()), "rows_after_exact_dedup": len(df),
              "same_english_multiple_targets": int(df.groupby(df["en"].apply(normkey))["te"].nunique().gt(1).sum())}
    print(json.dumps(report, indent=2))
    return df, report


# ==============================================================================
# 6. NEAR-DUPLICATES: deterministic MinHash-LSH (char 5-grams), exact-Jaccard verified
# ==============================================================================
_MH_P = np.uint64((1 << 31) - 1)


def _shingle_hashes(s, n=5):
    s = normkey(s)
    grams = {s[i:i + n] for i in range(max(len(s) - n + 1, 1))}
    arr = np.fromiter((zlib.crc32(g.encode("utf-8")) % int(_MH_P) for g in grams), dtype=np.uint64,
                      count=len(grams))
    return np.unique(arr)


def minhash_near_duplicate_pairs(texts, threshold, n_perm=128, n_bands=16, seed=1):
    """Returns verified pairs (i, j, jaccard) with Jaccard >= threshold. No bucket is skipped:
    buckets above 3000 members are verified in a star pattern against their first member."""
    rng = np.random.RandomState(seed)
    A = rng.randint(1, int(_MH_P), size=n_perm).astype(np.uint64)
    B = rng.randint(0, int(_MH_P), size=n_perm).astype(np.uint64)
    sh = [_shingle_hashes(t) for t in texts]
    sigs = np.empty((len(texts), n_perm), dtype=np.uint64)
    for i, h in enumerate(sh):
        sigs[i] = ((h[:, None] * A[None, :] + B[None, :]) % _MH_P).min(axis=0)
    rows = n_perm // n_bands
    cand = set()
    for b in range(n_bands):
        buckets = defaultdict(list)
        block = sigs[:, b * rows:(b + 1) * rows]
        for i in range(len(texts)):
            buckets[block[i].tobytes()].append(i)
        for members in buckets.values():
            if len(members) < 2:
                continue
            if len(members) <= 3000:
                for x in range(len(members)):
                    for y in range(x + 1, len(members)):
                        cand.add((members[x], members[y]))
            else:
                for y in members[1:]:
                    cand.add((members[0], y))
    out = []
    for i, j in cand:
        a, b_ = sh[i], sh[j]
        inter = np.intersect1d(a, b_, assume_unique=True).size
        jac = inter / (a.size + b_.size - inter)
        if jac >= threshold:
            out.append((i, j, float(jac)))
    return out


class UnionFind:
    def __init__(self, n):
        self.p = list(range(n))

    def find(self, x):
        while self.p[x] != x:
            self.p[x] = self.p[self.p[x]]
            x = self.p[x]
        return x

    def union(self, a, b):
        ra, rb = self.find(a), self.find(b)
        if ra != rb:
            self.p[ra] = rb


def group_ids_for(df, extra_pairs=()):
    """Groups rows that share a normalized EN, a normalized TE, or are near-duplicates (EN or TE)."""
    n = len(df)
    uf = UnionFind(n)
    for col in ("en", "te"):
        first = {}
        for i, k in enumerate(df[col].apply(normkey)):
            if k in first:
                uf.union(first[k], i)
            else:
                first[k] = i
    thr = CFG["data"]["near_dup_threshold"]
    stats = {}
    for col in ("en", "te"):
        pairs = minhash_near_duplicate_pairs(df[col].tolist(), thr, CFG["data"]["minhash_perms"],
                                             CFG["data"]["minhash_bands"], seed=1)
        stats[f"near_dup_pairs_{col}"] = len(pairs)
        for i, j, _ in pairs:
            uf.union(i, j)
    for i, j in extra_pairs:
        uf.union(i, j)
    gid = np.array([uf.find(i) for i in range(n)])
    _, gid = np.unique(gid, return_inverse=True)
    return gid, stats


def split_by_groups(gid, seed):
    fr = CFG["data"]["split"]
    n = len(gid)
    groups = defaultdict(list)
    for i, g in enumerate(gid):
        groups[g].append(i)
    order = list(groups.keys())
    np.random.default_rng(seed).shuffle(order)
    t_test, t_val = int(n * fr["test"]), int(n * fr["val"])
    split = np.array(["train"] * n, dtype=object)
    cnt_test = cnt_val = 0
    for g in order:
        idx = groups[g]
        if cnt_test < t_test:
            split[idx] = "test"; cnt_test += len(idx)
        elif cnt_val < t_val:
            split[idx] = "val"; cnt_val += len(idx)
    return split


def leakage_audit(df, split):
    """Exact (normalized EN / TE / pair) + near-duplicate (Jaccard >= threshold, EN and TE,
    independent MinHash seed) overlaps between every pair of partitions."""
    thr = CFG["data"]["near_dup_threshold"]
    rep, cross_pairs = {}, []
    parts = ["train", "val", "test"]
    en_k, te_k = df["en"].apply(normkey).values, df["te"].apply(normkey).values
    for a_i in range(3):
        for b_i in range(a_i + 1, 3):
            a, b = parts[a_i], parts[b_i]
            ma, mb = split == a, split == b
            rep[f"{a}-{b}"] = {
                "exact_en": len(set(en_k[ma]) & set(en_k[mb])),
                "exact_te": len(set(te_k[ma]) & set(te_k[mb])),
                "exact_pair": len(set(zip(en_k[ma], te_k[ma])) & set(zip(en_k[mb], te_k[mb]))),
                "near_dup_en": 0, "near_dup_te": 0}
    for col in ("en", "te"):
        pairs = minhash_near_duplicate_pairs(df[col].tolist(), thr, CFG["data"]["minhash_perms"],
                                             CFG["data"]["minhash_bands"], seed=777)
        for i, j, _ in pairs:
            if split[i] != split[j]:
                a, b = sorted([split[i], split[j]], key=parts.index)
                rep[f"{a}-{b}"][f"near_dup_{col}"] += 1
                cross_pairs.append((i, j))
    total = sum(v for d in rep.values() for v in d.values())
    rep["status"] = "PASS" if total == 0 else "FAIL"
    rep["method"] = (f"exact normalized match (EN, TE, pair) + MinHash-LSH char-5-gram Jaccard >= {thr} "
                     f"on EN and TE separately ({CFG['data']['minhash_perms']} perms, independent seed), "
                     f"every candidate verified by exact Jaccard, no buckets skipped")
    return rep, cross_pairs


# ==============================================================================
# 7. PREPOSITIONS + SUBWORD MASKS
# ==============================================================================
FALLBACK_PREPS = {"in", "on", "at", "to", "from", "for", "with", "by", "into", "through", "between",
                  "among", "under", "over", "before", "after", "during", "without", "within", "about",
                  "against", "along", "around", "behind", "below", "beneath", "beside", "beyond", "near",
                  "of", "off", "onto", "since", "toward", "towards", "underneath", "until", "upon"}
_NLP = None
_SPAN_CACHE = {}


def get_nlp():
    global _NLP
    if _NLP is None:
        _NLP = False
        if _SPACY_OK:
            try:
                _NLP = spacy.load("en_core_web_sm", disable=["parser", "ner", "lemmatizer"])
            except Exception as e:
                print(f"[prep] spaCy unavailable ({e}); using fallback list.")
    return _NLP


def ensure_spans(sents):
    todo = [s for s in dict.fromkeys(sents) if s not in _SPAN_CACHE]
    if not todo:
        return
    nlp = get_nlp()
    if nlp:
        for s, doc in zip(todo, nlp.pipe(todo, batch_size=512)):
            _SPAN_CACHE[s] = [(t.idx, t.idx + len(t.text), t.text.lower()) for t in doc if t.pos_ == "ADP"]
    else:
        for s in todo:
            _SPAN_CACHE[s] = [(m.start(), m.end(), m.group(0).lower())
                              for m in re.finditer(r"\b[a-zA-Z]+\b", s) if m.group(0).lower() in FALLBACK_PREPS]


def prep_count(s):
    ensure_spans([s])
    return len(_SPAN_CACHE[s])


_MASK_CACHE = {}


def compute_masks(sents, tokenizer, mode):
    """mode: 'true' (preposition subwords), 'random' (same count, fixed per sentence, content tokens only),
    'all' (every content token). Special/pad tokens (offset s==e) are never masked."""
    L = CFG["model"]["max_source_length"]
    key = (hashlib.md5("\n".join(sents).encode("utf-8")).hexdigest(), mode)
    if key in _MASK_CACHE:
        return _MASK_CACHE[key]
    ensure_spans(sents)
    out = []
    for i in range(0, len(sents), 1000):
        part = sents[i:i + 1000]
        enc = tokenizer(part, return_offsets_mapping=True, truncation=True, max_length=L, padding="max_length")
        for sent, offs in zip(part, enc["offset_mapping"]):
            content = [k for k, (s, e) in enumerate(offs) if s != e]
            true = np.zeros(L, dtype=np.uint8)
            for k in content:
                s, e = offs[k]
                for ps, pe, _ in _SPAN_CACHE[sent]:
                    if s < pe and e > ps:
                        true[k] = 1
                        break
            if mode == "true":
                m = true
            elif mode == "all":
                m = np.zeros(L, dtype=np.uint8); m[content] = 1
            elif mode == "random":
                m = np.zeros(L, dtype=np.uint8)
                kk = min(int(true.sum()), len(content))
                if kk:
                    h = int(hashlib.md5(f"{CFG['experiment']['mask_seed']}|{sent}".encode()).hexdigest()[:8], 16)
                    m[np.random.default_rng(h).choice(content, size=kk, replace=False)] = 1
            else:
                raise ValueError(mode)
            out.append(m)
    _MASK_CACHE[key] = out
    return out


# ==============================================================================
# 8. PS-CAB MODULE + MODEL WRAPPER
# ==============================================================================
class PSCAB(nn.Module):
    """H' = H + gamma * g(H) * (W_b H) * mask   (applied to encoder outputs, i.e. the keys/values
    of every decoder cross-attention layer). Identity at step 0 in every variant:
      gamma learned : gamma=0, W_b ~ N(0, 0.02)
      gamma fixed   : gamma=1 (buffer), W_b = 0
    NOTE FOR THE PAPER: this is a gated feature modulation of the cross-attention memory,
    not an additive bias on attention logits -- describe it that way."""

    def __init__(self, d, gate="learned", gamma="learned"):
        super().__init__()
        self.gate_mode, self.gamma_mode = gate, gamma
        self.W_b = nn.Linear(d, d)
        nn.init.zeros_(self.W_b.bias)
        if gamma == "learned":
            nn.init.normal_(self.W_b.weight, 0.0, 0.02)
            self.gamma = nn.Parameter(torch.zeros(1))
        else:
            nn.init.zeros_(self.W_b.weight)
            self.register_buffer("gamma", torch.ones(1))
        if gate == "learned":
            self.W_g = nn.Linear(d, d)

    def gate(self, H):
        return torch.sigmoid(self.W_g(H)) if self.gate_mode == "learned" else torch.ones_like(H)

    def forward(self, H, mask):
        m = mask.unsqueeze(-1).to(H.dtype)
        return H + self.gamma * self.gate(H) * self.W_b(H) * m


def shift_right_mbart(labels, pad_id):
    x = labels.clone()
    x[x == -100] = pad_id
    idx = (x.ne(pad_id).sum(dim=1) - 1).unsqueeze(-1)
    start = x.gather(1, idx).squeeze(-1)
    out = x.clone()
    out[:, 1:] = x[:, :-1]
    out[:, 0] = start
    return out


class PSCABMBart(nn.Module):
    def __init__(self, cond, attn_impl=None):
        super().__init__()
        spec = CONDITIONS[cond]
        kw = {"attn_implementation": attn_impl} if attn_impl else {}
        self.backbone = MBartForConditionalGeneration.from_pretrained(CFG["model"]["backbone"], **kw)
        self.pad_id = self.backbone.config.pad_token_id
        self.use_pscab = spec["use_pscab"]
        if self.use_pscab:
            self.pscab = PSCAB(self.backbone.config.d_model, spec["gate"], spec["gamma"])

    def encode(self, input_ids, attention_mask, mask):
        enc = self.backbone.model.encoder(input_ids=input_ids, attention_mask=attention_mask, return_dict=True)
        if self.use_pscab:
            enc.last_hidden_state = self.pscab(enc.last_hidden_state, mask)
        return enc

    def forward(self, input_ids, attention_mask, mask, labels):
        enc = self.encode(input_ids, attention_mask, mask)
        out = self.backbone(attention_mask=attention_mask, encoder_outputs=enc,
                            decoder_input_ids=shift_right_mbart(labels, self.pad_id),
                            use_cache=False, return_dict=True)
        return out.logits

    @torch.no_grad()
    def generate(self, input_ids, attention_mask, mask, **kw):
        enc = self.encode(input_ids, attention_mask, mask)
        return self.backbone.generate(encoder_outputs=enc, attention_mask=attention_mask, **kw)

    def param_report(self):
        return {"total_parameters": sum(p.numel() for p in self.parameters()),
                "pscab_added_parameters": sum(p.numel() for n, p in self.named_parameters() if n.startswith("pscab."))}


# ==============================================================================
# 9. TOKENIZER / DATASET / COLLATE
# ==============================================================================
_TOK = None


def get_tokenizer():
    global _TOK
    if _TOK is None:
        _TOK = MBart50TokenizerFast.from_pretrained(CFG["model"]["backbone"])
        _TOK.src_lang, _TOK.tgt_lang = CFG["model"]["src_lang"], CFG["model"]["tgt_lang"]
    return _TOK


class PairDataset(Dataset):
    def __init__(self, df, tok, mode):
        self.en, self.te = df["en"].tolist(), df["te"].tolist()
        self.mask = compute_masks(self.en, tok, mode)
        self.true = compute_masks(self.en, tok, "true")

    def __len__(self):
        return len(self.en)

    def __getitem__(self, i):
        return self.en[i], self.te[i], self.mask[i], self.true[i]


def collate(batch, tok):
    en, te, m, t = zip(*batch)
    src = tok(list(en), max_length=CFG["model"]["max_source_length"], truncation=True, padding=True,
              return_tensors="pt")
    tgt = tok(text_target=list(te), max_length=CFG["model"]["max_target_length"], truncation=True,
              padding=True, return_tensors="pt")
    labels = tgt["input_ids"].clone()
    labels[labels == tok.pad_token_id] = -100
    L = src["input_ids"].shape[1]
    return {"input_ids": src["input_ids"], "attention_mask": src["attention_mask"],
            "mask": torch.from_numpy(np.stack(m)[:, :L].astype(np.int64)),
            "true_mask": torch.from_numpy(np.stack(t)[:, :L].astype(np.int64)),
            "labels": labels}


# ==============================================================================
# 10. METRICS + CORPUS-LEVEL PAIRED STATISTICS
# ==============================================================================
_BLEU = None


def bleu_metric():
    """spBLEU with the FLORES-200 SentencePiece tokenizer (standard for Indic MT); falls back to 13a."""
    global _BLEU
    if _BLEU is None:
        for tok in ("flores200", "flores101", "13a"):
            try:
                b = BLEU(tokenize=tok)
                b.corpus_score(["తెలుగు వాక్యం"], [["తెలుగు వాక్యం"]])
                _BLEU = b
                print(f"[metrics] BLEU tokenizer = {tok}")
                break
            except Exception as e:
                print(f"[metrics] BLEU tokenizer {tok} unavailable: {e}")
    return _BLEU


def chrf_metric():
    return CHRF(word_order=2)


def compute_metrics(hyps, refs):
    hyps = [str(h) for h in hyps]
    b, c = bleu_metric(), chrf_metric()
    res = {"bleu": round(b.corpus_score(hyps, [refs]).score, 4),
           "chrf++": round(c.corpus_score(hyps, [refs]).score, 4),
           "n": len(refs), "bleu_signature": str(b.get_signature()), "chrf_signature": str(c.get_signature())}
    try:
        t = TER()
        res["ter"] = round(t.corpus_score(hyps, [refs]).score, 4)
    except Exception as e:
        print(f"[metrics] TER failed ({e}); NaN")
        res["ter"] = float("nan")
    return res


class StatMetric:
    """Per-sentence sufficient statistics so that resampling = summing rows (exact corpus-level score)."""

    def __init__(self, name):
        self.name = name
        self.m = {"bleu": bleu_metric, "chrf++": chrf_metric}.get(name, lambda: None)()

    def stats(self, hyps, refs, seg_scores=None):
        if self.name == "comet":
            return np.asarray(seg_scores, dtype=np.float64).reshape(-1, 1)
        return np.asarray(self.m._extract_corpus_statistics([str(h) for h in hyps], [refs]), dtype=np.int64)

    def score(self, stat_sum, n):
        if self.name == "comet":
            return float(stat_sum[0] / n) * 100.0
        return float(self.m._compute_score_from_stats(list(stat_sum)).score)


def paired_test(sa, sb, metric, n_boot=2000, n_ar=10000, seed=12345):
    """A minus B. Returns observed delta, bootstrap 95% CI (Koehn 2004) and two-sided
    approximate-randomization p-value (Riezler & Maxwell 2005)."""
    rng = np.random.default_rng(seed)
    N = len(sa)
    obs = metric.score(sa.sum(0), N) - metric.score(sb.sum(0), N)
    boots = np.empty(n_boot)
    for k in range(n_boot):
        idx = rng.integers(0, N, N)
        boots[k] = metric.score(sa[idx].sum(0), N) - metric.score(sb[idx].sum(0), N)
    hits = 0
    for _ in range(n_ar):
        sw = rng.random(N) < 0.5
        a = np.where(sw[:, None], sb, sa).sum(0)
        b = np.where(sw[:, None], sa, sb).sum(0)
        if abs(metric.score(a, N) - metric.score(b, N)) >= abs(obs) - 1e-12:
            hits += 1
    lo, hi = np.percentile(boots, [2.5, 97.5])
    return {"delta": round(obs, 4), "ci95_low": round(float(lo), 4), "ci95_high": round(float(hi), 4),
            "p_ar": round((hits + 1) / (n_ar + 1), 5), "n": N}


def holm(pvals: dict, alpha=0.05):
    items = sorted(pvals.items(), key=lambda kv: kv[1])
    m, out, ok = len(items), {}, True
    for r, (k, p) in enumerate(items, 1):
        thr = alpha / (m - r + 1)
        ok = ok and p < thr
        out[k] = {"holm_threshold": round(thr, 5), "significant_holm": bool(ok)}
    return out


# ==============================================================================
# 11. GENERATION
# ==============================================================================
def gen_kwargs(tok):
    return {"num_beams": CFG["decoding"]["num_beams"], "max_new_tokens": CFG["decoding"]["max_new_tokens"],
            "forced_bos_token_id": tok.convert_tokens_to_ids(CFG["model"]["tgt_lang"])}


def generation_record(model, tok):
    """Effective decoding configuration (explicit kwargs + the model's stored generation_config)."""
    out = {"explicit_kwargs": gen_kwargs(tok)}
    try:
        gc_ = model.backbone.generation_config.to_dict()
        out["model_generation_config"] = {k: gc_.get(k) for k in (
            "max_length", "num_beams", "early_stopping", "length_penalty", "no_repeat_ngram_size",
            "forced_eos_token_id", "decoder_start_token_id", "repetition_penalty", "do_sample")}
    except Exception as e:
        out["model_generation_config"] = f"unavailable ({e})"
    return out


@torch.no_grad()
def generate(model, tok, df, mode, bs=None):
    bs = bs or CFG["optim"]["eval_batch_size"]
    order = np.argsort(-df["en"].str.len().values, kind="stable")
    sdf = df.iloc[order].reset_index(drop=True)
    loader = DataLoader(PairDataset(sdf, tok, mode), batch_size=bs, shuffle=False,
                        collate_fn=lambda b: collate(b, tok))
    tgt_id = tok.convert_tokens_to_ids(CFG["model"]["tgt_lang"])
    model.eval()
    out, t0 = [], time.time()
    for batch in loader:
        batch = {k: v.to(DEVICE) for k, v in batch.items()}
        with amp_ctx():
            g = model.generate(batch["input_ids"], batch["attention_mask"], batch["mask"], **gen_kwargs(tok))
        out.extend(tok.batch_decode(g, skip_special_tokens=True))
    el = time.time() - t0
    preds = [None] * len(df)
    for k, o in enumerate(order):
        preds[o] = out[k]
    return preds, {"sentences_per_sec": len(df) / max(el, 1e-6), "gen_seconds": el}


def save_preds(run_id, split_name, df, preds):
    d = df[["en", "te"]].copy()
    d["prediction"] = preds
    d.to_csv(f"{DIRS['preds']}/{run_id}__{split_name}.csv", index=False)


def load_preds(run_id, split_name):
    p = f"{DIRS['preds']}/{run_id}__{split_name}.csv"
    return read_csv_str(p)["prediction"].tolist() if os.path.exists(p) else None


# ==============================================================================
# 12. TRAINING (step-matched, eval every N steps, early stopping on validation NLL)
# ==============================================================================
@torch.no_grad()
def val_nll(model, loader):
    model.eval()
    tot, ntok = 0.0, 0
    for batch in loader:
        batch = {k: v.to(DEVICE) for k, v in batch.items()}
        with amp_ctx():
            logits = model(batch["input_ids"], batch["attention_mask"], batch["mask"], batch["labels"])
        l = F.cross_entropy(logits.float().view(-1, logits.size(-1)), batch["labels"].view(-1),
                            ignore_index=-100, reduction="sum")
        tot += l.item()
        ntok += int((batch["labels"] != -100).sum())
    model.train()
    return tot / max(ntok, 1)


def train_model(cond, seed, train_df, val_df, run_id):
    o = CFG["optim"]
    spec = CONDITIONS[cond]
    print(f"\n{'='*72}\nTRAIN {run_id}: {spec}\n{'='*72}")
    set_all_seeds(seed)
    tok = get_tokenizer()
    g = torch.Generator(); g.manual_seed(seed)
    tl = DataLoader(PairDataset(train_df, tok, spec["mask"]), batch_size=o["train_batch_size"], shuffle=True,
                    generator=g, collate_fn=lambda b: collate(b, tok), drop_last=False)
    vl = DataLoader(PairDataset(val_df, tok, spec["mask"]), batch_size=o["eval_batch_size"], shuffle=False,
                    collate_fn=lambda b: collate(b, tok))
    model = PSCABMBart(cond).to(DEVICE)
    init_gate = None
    if model.use_pscab and spec["gate"] == "learned":
        init_gate = {k: v.detach().cpu().clone() for k, v in model.pscab.W_g.state_dict().items()}
    base = [p for n, p in model.named_parameters() if not n.startswith("pscab.")]
    extra = [p for n, p in model.named_parameters() if n.startswith("pscab.")]
    groups = [{"params": base, "lr": o["lr"]}]
    if extra:
        groups.append({"params": extra, "lr": o["pscab_lr"], "weight_decay": 0.0})
    opt = torch.optim.AdamW(groups, weight_decay=o["weight_decay"])
    total, warm = o["max_steps"], max(1, int(o["warmup_frac"] * o["max_steps"]))
    sched = torch.optim.lr_scheduler.LambdaLR(
        opt, lambda s: s / warm if s < warm else 0.5 * (1 + math.cos(math.pi * min(1.0, (s - warm) / max(1, total - warm)))))
    scaler = make_scaler()
    params = base + extra

    log, best, best_step, best_state, bad = [], float("inf"), 0, None, 0
    step, micro, run_loss, run_n, nonfinite, stop = 0, 0, 0.0, 0, 0, "max_steps_reached"
    t0 = time.time()
    model.train()
    opt.zero_grad(set_to_none=True)
    epoch = 0
    while step < total and stop == "max_steps_reached":
        for batch in tl:
            batch = {k: v.to(DEVICE) for k, v in batch.items()}
            with amp_ctx():
                logits = model(batch["input_ids"], batch["attention_mask"], batch["mask"], batch["labels"])
                loss = F.cross_entropy(logits.view(-1, logits.size(-1)), batch["labels"].view(-1),
                                       ignore_index=-100, label_smoothing=o["label_smoothing"])
            if torch.isfinite(loss):
                scaler.scale(loss / o["grad_accum"]).backward()
                run_loss += loss.item(); run_n += 1
            else:
                nonfinite += 1
            micro += 1
            if micro % o["grad_accum"] != 0:
                continue
            scaler.unscale_(opt)
            torch.nn.utils.clip_grad_norm_(params, o["clip"])
            scaler.step(opt); scaler.update(); sched.step()
            opt.zero_grad(set_to_none=True)
            step += 1
            if step % o["eval_every"] == 0 or step == total:
                v = val_nll(model, vl)
                row = {"step": step, "epoch": round(step * o["grad_accum"] * o["train_batch_size"] / len(train_df), 3),
                       "train_loss_ls": run_loss / max(run_n, 1), "val_nll": v,
                       "lr": sched.get_last_lr()[0], "elapsed_h": (time.time() - t0) / 3600}
                if model.use_pscab and spec["gamma"] == "learned":
                    row["gamma"] = float(model.pscab.gamma.item())
                log.append(row)
                run_loss, run_n = 0.0, 0
                flag = ""
                if v < best - o["min_delta"]:
                    best, best_step, bad = v, step, 0
                    best_state = {k: t.detach().cpu().clone() for k, t in model.state_dict().items()}
                    flag = "  <- best"
                else:
                    bad += 1
                print(f"[{run_id}] step {step}/{total} ep {row['epoch']:.2f} train(ls) {row['train_loss_ls']:.4f} "
                      f"val_nll {v:.4f} lr {row['lr']:.2e} {row['elapsed_h']:.2f}h{flag}")
                if bad >= o["patience_evals"]:
                    stop = "early_stopping"
                    break
            if step >= total:
                break
        epoch += 1
    train_h = (time.time() - t0) / 3600
    model.load_state_dict(best_state)
    del opt, sched, scaler, best_state
    free_gpu()
    pd.DataFrame(log).to_csv(f"{DIRS['logs']}/{run_id}_curve.csv", index=False)
    meta = {"steps_trained": step, "best_step": best_step, "best_val_nll": best, "stop_reason": stop,
            "train_hours": train_h, "sec_per_step": train_h * 3600 / max(step, 1),
            "nonfinite_loss_batches": nonfinite, **model.param_report()}
    if model.use_pscab and spec["gamma"] == "learned":
        meta["gamma_at_best"] = float(model.pscab.gamma.item())
    print(f"[{run_id}] done: {step} steps, best step {best_step} (val_nll {best:.4f}), stop={stop}, {train_h:.2f}h")
    return model, tok, meta, init_gate


# ==============================================================================
# 13. IN-SESSION DIAGNOSTICS (need the model in memory; results saved to disk)
# ==============================================================================
@torch.no_grad()
def pscab_diagnostics(model, tok, df, cond, init_gate, run_id):
    """Gate values at preposition vs other content positions: trained gate vs the SAME gate at
    initialisation (control), gamma, and ||bias|| / ||H|| on masked positions."""
    spec = CONDITIONS[cond]
    sub = df.sample(n=min(CFG["experiment"]["diag_subset"], len(df)), random_state=0).reset_index(drop=True)
    loader = DataLoader(PairDataset(sub, tok, spec["mask"]), batch_size=CFG["optim"]["eval_batch_size"],
                        shuffle=False, collate_fn=lambda b: collate(b, tok))
    Wg0 = None
    if init_gate is not None:
        Wg0 = (init_gate["weight"].to(DEVICE).float(), init_gate["bias"].to(DEVICE).float())
    gt_p, gt_n, g0_p, g0_n, ratios = [], [], [], [], []
    pm = model.pscab
    learned_gate = spec["gate"] == "learned"
    for batch in loader:
        batch = {k: v.to(DEVICE) for k, v in batch.items()}
        with amp_ctx():
            H = model.backbone.model.encoder(input_ids=batch["input_ids"], attention_mask=batch["attention_mask"],
                                             return_dict=True).last_hidden_state
        H = H.float()
        valid = batch["attention_mask"].bool()
        tm = batch["true_mask"].bool() & valid
        content = valid.clone()
        content[:, 0] = False                                   # language-code token
        eos = batch["attention_mask"].sum(1) - 1
        content[torch.arange(len(eos), device=DEVICE), eos] = False   # </s>
        nm = content & ~tm
        G = pm.gate(H)
        if learned_gate:
            g = G.mean(-1)
            gt_p += g[tm].cpu().tolist(); gt_n += g[nm].cpu().tolist()
            if Wg0 is not None:
                g0 = torch.sigmoid(H @ Wg0[0].T + Wg0[1]).mean(-1)
                g0_p += g0[tm].cpu().tolist(); g0_n += g0[nm].cpu().tolist()
        am = batch["mask"].bool() & valid
        bias = pm.gamma * G * pm.W_b(H)
        r = bias.norm(dim=-1) / H.norm(dim=-1).clamp_min(1e-6)
        ratios += r[am].cpu().tolist()

    def auc(pos, neg):
        if not pos or not neg:
            return None
        try:
            from sklearn.metrics import roc_auc_score
            return float(roc_auc_score([1] * len(pos) + [0] * len(neg), pos + neg))
        except Exception:
            return None

    rep = {"gamma": float(model.pscab.gamma.item()),
           "bias_to_hidden_norm_ratio_mean": float(np.mean(ratios)) if ratios else None,
           "bias_to_hidden_norm_ratio_median": float(np.median(ratios)) if ratios else None,
           "n_prep_positions": len(gt_p), "n_other_positions": len(gt_n)}
    if gt_p:
        rep.update({"gate_trained_prep_mean": float(np.mean(gt_p)), "gate_trained_other_mean": float(np.mean(gt_n)),
                    "gate_trained_auc_prep_vs_other": auc(gt_p, gt_n)})
    if g0_p:
        rep.update({"gate_init_prep_mean": float(np.mean(g0_p)), "gate_init_other_mean": float(np.mean(g0_n)),
                    "gate_init_auc_prep_vs_other": auc(g0_p, g0_n)})
    rng = np.random.default_rng(0)
    samp = lambda x: rng.choice(np.array(x), size=min(20000, len(x)), replace=False) if x else np.array([])
    np.savez_compressed(f"{DIRS['analysis']}/{run_id}_gate_samples.npz", trained_prep=samp(gt_p),
                        trained_other=samp(gt_n), init_prep=samp(g0_p), init_other=samp(g0_n))
    return rep


@torch.no_grad()
def attention_mass(model, tok, df, cond, run_id):
    """Teacher-forced cross-attention mass on preposition source tokens (mean over heads and target
    positions), last layer and mean over layers, normalised by the uniform expectation n_prep/n_src."""
    try:
        eager = PSCABMBart(cond, attn_impl="eager")
        eager.load_state_dict(model.state_dict())
        eager.to(DEVICE).eval()
    except Exception as e:
        print(f"[attention] eager model unavailable ({e}); skipped.")
        return None
    sents = df["en"].tolist()
    ensure_spans(sents)
    cand = [i for i, s in enumerate(sents) if _SPAN_CACHE[s]]
    if not cand:
        del eager
        free_gpu()
        return None
    rng = np.random.default_rng(0)
    idx = sorted(rng.choice(cand, size=min(CFG["experiment"]["attention_subset"], len(cand)), replace=False))
    sub = df.iloc[idx].reset_index(drop=True)
    true = compute_masks(sub["en"].tolist(), tok, "true")
    applied = compute_masks(sub["en"].tolist(), tok, CONDITIONS[cond]["mask"])
    last, allL, ratio_last = [], [], []
    for k in range(len(sub)):
        src = tok(sub["en"][k], return_tensors="pt", truncation=True, max_length=CFG["model"]["max_source_length"]).to(DEVICE)
        tgt = tok(text_target=sub["te"][k], return_tensors="pt", truncation=True,
                  max_length=CFG["model"]["max_target_length"]).to(DEVICE)
        S = src["input_ids"].shape[1]
        pm = torch.from_numpy(applied[k][:S].astype(np.int64)).unsqueeze(0).to(DEVICE)
        tm = torch.from_numpy(true[k][:S].astype(bool)).to(DEVICE)
        if tm.sum() == 0:
            continue
        enc = eager.encode(src["input_ids"], src["attention_mask"], pm)
        out = eager.backbone(attention_mask=src["attention_mask"], encoder_outputs=enc,
                             decoder_input_ids=shift_right_mbart(tgt["input_ids"], eager.pad_id),
                             output_attentions=True, use_cache=False, return_dict=True)
        A = torch.stack([a[0].float().mean(0) for a in out.cross_attentions])  # (layers, T, S)
        mass = A[:, :, tm].sum(-1).mean(-1)                                     # (layers,)
        last.append(float(mass[-1])); allL.append(float(mass.mean()))
        ratio_last.append(float(mass[-1]) / (float(tm.sum()) / S))
    del eager
    free_gpu()
    return {"n_sentences": len(last), "prep_mass_last_layer": float(np.mean(last)),
            "prep_mass_mean_layers": float(np.mean(allL)),
            "prep_mass_last_layer_over_uniform": float(np.mean(ratio_last))}


PERTURBATIONS = {
    "punct_spacing": lambda s: re.sub(r"\s*([,.;:!?])", r" \1 ", s).strip(),
    "lowercase": lambda s: s.lower(),
    "drop_final_punct": lambda s: s[:-1].rstrip() if s and s[-1] in ".!?" else s,
}


def robustness(model, tok, df, cond):
    sub = df.sample(n=min(CFG["experiment"]["robustness_subset"], len(df)), random_state=1).reset_index(drop=True)
    refs = sub["te"].tolist()
    base, _ = generate(model, tok, sub, CONDITIONS[cond]["mask"])
    b = compute_metrics(base, refs)
    rows = [{"perturbation": "none", "bleu": b["bleu"], "chrf++": b["chrf++"], "delta_bleu": 0.0, "delta_chrf++": 0.0}]
    for name, fn in PERTURBATIONS.items():
        p = sub.copy(); p["en"] = p["en"].apply(fn)
        preds, _ = generate(model, tok, p, CONDITIONS[cond]["mask"])
        m = compute_metrics(preds, refs)
        rows.append({"perturbation": name, "bleu": m["bleu"], "chrf++": m["chrf++"],
                     "delta_bleu": round(m["bleu"] - b["bleu"], 4), "delta_chrf++": round(m["chrf++"] - b["chrf++"], 4)})
    return rows


def save_model_bundle(model, tok, run_id, record):
    d = f"{DIRS['models']}/{run_id}"
    os.makedirs(d, exist_ok=True)
    model.backbone.half().save_pretrained(d, safe_serialization=True)
    model.backbone.float()
    tok.save_pretrained(d)
    if model.use_pscab:
        torch.save({k: v.half() for k, v in model.pscab.state_dict().items()}, f"{d}/pscab_module_fp16.pt")
    jdump({"condition": record["condition"], "seed": record["seed"], "config": CFG,
           "reload": "PSCABMBart(cond); backbone=MBartForConditionalGeneration.from_pretrained(dir); "
                     "pscab.load_state_dict(torch.load('pscab_module_fp16.pt')) (cast to float)"},
          f"{d}/bundle_info.json")
    if run_id in CFG["experiment"]["export_h5_for"] and _H5_OK:
        with h5py.File(f"{d}/{run_id}_fp16_state_dict.h5", "w") as f:
            f.attrs["note"] = "PyTorch state_dict tensors (fp16) of PSCABMBart; NOT a Keras model."
            for k, v in model.state_dict().items():
                f.create_dataset(k, data=v.detach().half().cpu().numpy(), compression="gzip")
    print(f"[save] model bundle -> {d}")


# ==============================================================================
# 14. RUN EXECUTORS
# ==============================================================================
def load_splits():
    return tuple(read_csv_str(f"{DIRS['data']}/{n}.csv") for n in ("train", "val", "test"))


def external_sets():
    man = jload(f"{DIRS['data']}/external_manifest.json") or {}
    return {k: read_csv_str(v) for k, v in man.items() if os.path.exists(v)}


def execute_main_run(run_id):
    """Train one (condition, seed) on the frozen split. Generates validation predictions (scored now),
    and TEST / external predictions (stored blind, scored only in Part 3)."""
    if run_done(run_id):
        print(f"[skip] {run_id} already complete.")
        return jload(run_record_path(run_id))
    need = estimated_run_hours()
    if session_hours_left() < need:
        print(f"[time] {run_id} needs ~{need:.1f}h but only {session_hours_left():.1f}h left in this "
              f"session -> NOT started (never truncated). Run it next session.")
        return None
    cond, seed, _ = parse_run_id(run_id)
    t_all = time.time()
    train, val, test = load_splits()
    model, tok, meta, init_gate = train_model(cond, seed, train, val, run_id)
    mode = CONDITIONS[cond]["mask"]
    rec = {"run_id": run_id, "kind": "main", "condition": cond, "seed": seed, "spec": CONDITIONS[cond],
           "status": "complete", "env": environment_info(), **meta}

    rec["generation"] = generation_record(model, tok)
    vp, vinfo = generate(model, tok, val, mode)
    save_preds(run_id, "val", val, vp)
    rec["val_metrics"] = compute_metrics(vp, val["te"].tolist())
    rec["inference_sentences_per_sec"] = vinfo["sentences_per_sec"]
    print(f"[{run_id}] VAL {rec['val_metrics']['bleu']} BLEU / {rec['val_metrics']['chrf++']} chrF++")

    tp, _ = generate(model, tok, test, mode)
    save_preds(run_id, "test", test, tp)                    # blind: not scored here
    for name, edf in external_sets().items():
        ep, _ = generate(model, tok, edf, mode)
        save_preds(run_id, f"ext_{name}", edf, ep)

    if model.use_pscab:
        rec["diagnostics"] = pscab_diagnostics(model, tok, val, cond, init_gate, run_id)
        print(f"[{run_id}] diagnostics: {rec['diagnostics']}")
    if seed == CFG["experiment"]["seeds"][0]:
        rec["attention"] = attention_mass(model, tok, val, cond, run_id)
        rec["robustness_val"] = robustness(model, tok, val, cond)
    if run_id in CFG["experiment"]["save_models_for"]:
        save_model_bundle(model, tok, run_id, rec)
    del model
    free_gpu()
    rec["wall_hours_total"] = (time.time() - t_all) / 3600
    jdump(rec, run_record_path(run_id))
    print(f"[{run_id}] COMPLETE in {rec['wall_hours_total']:.2f}h")
    return rec


def execute_zeroshot():
    rid = "zeroshot_mbart50"
    if run_done(rid):
        print(f"[skip] {rid} already complete.")
        return
    t0 = time.time()
    train, val, test = load_splits()
    tok = get_tokenizer()
    model = PSCABMBart("baseline").to(DEVICE).eval()
    vp, info = generate(model, tok, val, "true")
    save_preds(rid, "val", val, vp)
    tp, _ = generate(model, tok, test, "true")
    save_preds(rid, "test", test, tp)
    for name, edf in external_sets().items():
        ep, _ = generate(model, tok, edf, "true")
        save_preds(rid, f"ext_{name}", edf, ep)
    rec = {"run_id": rid, "kind": "zeroshot", "condition": "zeroshot_mbart50", "seed": None, "status": "complete",
           "env": environment_info(), "generation": generation_record(model, tok),
           "val_metrics": compute_metrics(vp, val["te"].tolist()), "inference_sentences_per_sec": info["sentences_per_sec"],
           "wall_hours_total": (time.time() - t0) / 3600, **model.param_report()}
    jdump(rec, run_record_path(rid))
    print(f"[zeroshot] VAL {rec['val_metrics']['bleu']} BLEU / {rec['val_metrics']['chrf++']} chrF++")
    del model
    free_gpu()


def run_next_main_runs(max_runs=2):
    q = build_main_queue()
    pending = [r for r in q if not run_done(r)]
    print(f"[queue] main: {len(q) - len(pending)}/{len(q)} complete; pending next: {pending[:max_runs]}")
    started = 0
    for rid in pending:
        if started >= max_runs:
            break
        rec = execute_main_run(rid)
        if rec is None:
            break
        started += 1
    left = [r for r in q if not run_done(r)]
    print(f"[queue] main queue after this session: {len(q) - len(left)}/{len(q)} complete. "
          + ("MAIN QUEUE COMPLETE." if not left else f"Next session will run: {left[:2]}"))


# ==============================================================================
# 15. DATA PREPARATION (frozen once)
# ==============================================================================
def prepare_flores_plus(train_all):
    try:
        try:
            from kaggle_secrets import UserSecretsClient
            os.environ.setdefault("HF_TOKEN", UserSecretsClient().get_secret("HF_TOKEN"))
        except Exception:
            pass
        from datasets import load_dataset
        en = load_dataset("openlanguagedata/flores_plus", "eng_Latn", split="devtest").to_pandas()
        te = load_dataset("openlanguagedata/flores_plus", "tel_Telu", split="devtest").to_pandas()
        df = en[["id", "text"]].rename(columns={"text": "en"}).merge(
            te[["id", "text"]].rename(columns={"text": "te"}), on="id")[["en", "te"]]
        return df
    except Exception as e:
        print(f"[external] FLORES+ not available ({e}). Accept the dataset terms on HF and add HF_TOKEN "
              f"as a Kaggle secret, or supply CSVs in CFG['external']['csvs'].")
        return None


def freeze_external_sets(all_df):
    man_p = f"{DIRS['data']}/external_manifest.json"
    if os.path.exists(man_p):
        return jload(man_p)
    os.makedirs(f"{DIRS['data']}/external", exist_ok=True)
    known_en, known_te = set(all_df["en"].apply(normkey)), set(all_df["te"].apply(normkey))
    sets = {}
    if CFG["external"]["try_flores_plus"]:
        f = prepare_flores_plus(all_df)
        if f is not None:
            sets["flores_plus_devtest"] = f
    for name, p in CFG["external"]["csvs"].items():
        if os.path.exists(p):
            d = read_csv_str(p)
            sets[name] = d[["en", "te"]]
    man, rep = {}, {}
    for name, d in sets.items():
        d = d.copy()
        d["en"], d["te"] = d["en"].apply(normalize_text), d["te"].apply(normalize_text)
        ov = d["en"].apply(normkey).isin(known_en) | d["te"].apply(normkey).isin(known_te)
        rep[name] = {"n_raw": len(d), "overlap_with_our_data_removed": int(ov.sum())}
        d = d[~ov].reset_index(drop=True)
        p = f"{DIRS['data']}/external/{name}.csv"
        d.to_csv(p, index=False)
        man[name] = p
        rep[name]["n_used"] = len(d)
    jdump(man, man_p)
    jdump(rep, f"{DIRS['data']}/external_report.json")
    print(f"[external] frozen external test sets: {rep if rep else 'none'} "
          f"(external sets are frozen now; adding one later would leave earlier runs without predictions)")
    return man


def stage_data_prep():
    names = ["train", "val", "test"]
    summ_p = f"{DIRS['data']}/data_summary.json"
    if all(os.path.exists(f"{DIRS['data']}/{n}.csv") for n in names) and os.path.exists(summ_p):
        print("[data] frozen split found -- reusing it.")
        summ = jload(summ_p)
        print(json.dumps(summ["leakage_audit"], indent=2))
        return summ
    df, clean_rep = load_and_clean()
    rng = np.random.default_rng(CFG["data"]["select_seed"])
    n = min(CFG["data"]["max_selected_pairs"], len(df))
    sel = df.iloc[np.sort(rng.choice(len(df), size=n, replace=False))].reset_index(drop=True)
    sel["row_id"] = np.arange(len(sel))
    print(f"[data] selected {len(sel)} pairs (seed={CFG['data']['select_seed']}); building near-duplicate groups ...")
    t0 = time.time()
    extra = []
    for it in range(5):
        gid, nd = group_ids_for(sel, extra)
        split = split_by_groups(gid, CFG["data"]["split_seed"])
        audit, cross = leakage_audit(sel, split)
        print(f"[data] split iteration {it}: audit {audit['status']} "
              f"({len(cross)} cross-split near-dup pairs) [{time.time() - t0:.0f}s]")
        if audit["status"] == "PASS":
            break
        extra += cross
    if audit["status"] != "PASS":
        raise RuntimeError("Leakage audit failed after 5 re-splits: " + json.dumps(audit))
    sel["group_id"], sel["split"] = gid, split
    for nm in names:
        sel[sel["split"] == nm][["row_id", "group_id", "en", "te"]].to_csv(f"{DIRS['data']}/{nm}.csv", index=False)
    sizes = {nm: int((split == nm).sum()) for nm in names}
    ensure_spans(sel["en"].tolist())
    prep_share = {nm: float(np.mean([len(_SPAN_CACHE[s]) > 0 for s in sel.loc[sel["split"] == nm, "en"]]))
                  for nm in names}
    summ = {"cleaning": clean_rep, "selected_pairs": len(sel), "split_sizes": sizes, "near_dup_grouping": nd,
            "n_groups": int(gid.max() + 1), "largest_group": int(np.bincount(gid).max()),
            "share_sentences_with_preposition": prep_share, "leakage_audit": audit,
            "preposition_detector": "spaCy en_core_web_sm ADP" if get_nlp() else "fallback list"}
    jdump(summ, summ_p)
    print(json.dumps(summ, indent=2))
    freeze_external_sets(sel)
    return summ


# ==============================================================================
# 16. PROGRESS REPORT (VALIDATION ONLY -- run after every part)
# ==============================================================================
def all_records(kind=None):
    recs = [jload(p) for p in sorted(glob.glob(f"{DIRS['runs']}/*.json"))]
    recs = [r for r in recs if r and r.get("status") == "complete"]
    return [r for r in recs if kind is None or r.get("kind") == kind]


def progress_report():
    print(f"\n{'#'*72}\nPROGRESS REPORT -- VALIDATION SET ONLY (test set not scored)\n{'#'*72}")
    recs = [r for r in all_records() if r.get("kind") in ("main", "zeroshot")]
    if not recs:
        print("no completed runs yet.")
        return
    train, val, test = load_splits()
    refs = val["te"].tolist()

    # --- Table P1: per-run validation results + training facts
    rows = []
    for r in recs:
        vm = r["val_metrics"]
        rows.append({"run": r["run_id"], "condition": r["condition"], "seed": r.get("seed"),
                     "steps": r.get("steps_trained"), "best_step": r.get("best_step"),
                     "stop": r.get("stop_reason"), "val_nll": r.get("best_val_nll"),
                     "BLEU": vm["bleu"], "chrF++": vm["chrf++"], "TER": vm["ter"],
                     "train_h": r.get("train_hours"), "transformers": rec_tf_version(r)})
    t1 = pd.DataFrame(rows)
    print("\n[P1] Per-run validation results\n" + t1.to_string(index=False))
    save_table(t1, "P1_validation_per_run", "Validation results per run")
    steps = t1.dropna(subset=["steps"])
    if len(steps) and steps["steps"].nunique() > 1:
        print("[P1 note] runs differ in steps_trained ONLY because of early stopping on validation NLL "
              "(identical max_steps budget); report best_step in the paper.")

    # --- Table P2: condition mean +- SD over seeds (validation)
    main = t1[t1["condition"].isin(CONDITIONS)]
    if len(main):
        agg = main.groupby("condition").agg(n_seeds=("seed", "nunique"), BLEU_mean=("BLEU", "mean"),
                                            BLEU_sd=("BLEU", "std"), chrF_mean=("chrF++", "mean"),
                                            chrF_sd=("chrF++", "std")).reset_index()
        print("\n[P2] Validation, mean +- SD over seeds\n" + agg.to_string(index=False))
        save_table(agg, "P2_validation_by_condition")

    # --- Table P3: paired corpus-level tests on validation (development statistics)
    preds = {r["run_id"]: load_preds(r["run_id"], "val") for r in recs}
    comps = []
    for r in recs:
        c, s = r["condition"], r.get("seed")
        if c == "pscab":
            comps.append((f"pscab_s{s}", f"baseline_s{s}"))
        elif c in ABLATIONS:
            comps.append((f"pscab_s{s}", f"{c}_s{s}"))
    comps = [(a, b) for a, b in comps if preds.get(a) and preds.get(b)]
    if comps:
        rows = []
        for mname in ("bleu", "chrf++"):
            M = StatMetric(mname)
            cache = {k: M.stats(v, refs) for k, v in preds.items() if v}
            pv = {}
            for a, b in comps:
                res = paired_test(cache[a], cache[b], M, n_boot=1000, n_ar=2000)
                rows.append({"metric": mname, "A": a, "B": b, **res})
                pv[f"{mname}:{a}-{b}"] = res["p_ar"]
            h = holm(pv)
            for row in rows:
                k = f"{row['metric']}:{row['A']}-{row['B']}"
                if k in h:
                    row.update(h[k])
        t3 = pd.DataFrame(rows)
        print("\n[P3] Paired tests on VALIDATION (A - B; AR p-value, bootstrap 95% CI, Holm per metric)\n"
              + t3.to_string(index=False))
        save_table(t3, "P3_validation_paired_tests")

    # --- Table P4: preposition subsets on validation (+ noise floor)
    ensure_spans(val["en"].tolist())
    has = np.array([len(_SPAN_CACHE[s]) > 0 for s in val["en"]])
    rows = []
    for rid, p in preds.items():
        if not p:
            continue
        P = np.array(p, dtype=object); Rf = np.array(refs, dtype=object)
        for name, m in [("with_prep", has), ("no_prep", ~has)]:
            if m.sum():
                mm = compute_metrics(list(P[m]), list(Rf[m]))
                rows.append({"run": rid, "subset": name, "n": int(m.sum()), "BLEU": mm["bleu"], "chrF++": mm["chrf++"]})
    t4 = pd.DataFrame(rows)
    print("\n[P4] Validation by preposition subset\n" + t4.to_string(index=False))
    print("[P4 note] On no_prep sentences the true-mask PS-CAB module is an exact no-op; any baseline-vs-pscab "
          "gap there measures training variance only (use it as a noise floor for H2).")
    save_table(t4, "P4_validation_preposition_subsets")

    # --- Table P5: module diagnostics
    rows = [{"run": r["run_id"], **r["diagnostics"]} for r in recs if r.get("diagnostics")]
    if rows:
        t5 = pd.DataFrame(rows)
        print("\n[P5] PS-CAB diagnostics (gate trained vs init control; gamma; bias/hidden ratio)\n" + t5.to_string(index=False))
        save_table(t5, "P5_pscab_diagnostics")
    rows = [{"run": r["run_id"], **r["attention"]} for r in recs if r.get("attention")]
    if rows:
        t6 = pd.DataFrame(rows)
        print("\n[P6] Cross-attention mass on preposition tokens (validation, teacher-forced)\n" + t6.to_string(index=False))
        save_table(t6, "P6_attention_mass")
    rows = [{"run": r["run_id"], **x} for r in recs if r.get("robustness_val") for x in r["robustness_val"]]
    if rows:
        t7 = pd.DataFrame(rows)
        print("\n[P7] Robustness to source perturbations (validation subset)\n" + t7.to_string(index=False))
        save_table(t7, "P7_robustness_validation")

    # --- Figures (validation only)
    try:
        validation_figures()
    except Exception as e:
        print(f"[figures] validation_figures failed: {e}")
    for r in recs:
        f = f"{DIRS['analysis']}/{r['run_id']}_gate_samples.npz"
        if os.path.exists(f) and r["run_id"].endswith(f"_s{CFG['experiment']['seeds'][0]}"):
            z = np.load(f)
            if z["trained_prep"].size:
                plt.figure(figsize=(7, 3.8))
                bins = np.linspace(0, 1, 60)
                plt.hist(z["trained_other"], bins, alpha=.5, density=True, label="trained: other")
                plt.hist(z["trained_prep"], bins, alpha=.5, density=True, label="trained: preposition")
                if z["init_prep"].size:
                    plt.hist(z["init_prep"], bins, histtype="step", density=True, label="init: preposition")
                    plt.hist(z["init_other"], bins, histtype="step", density=True, label="init: other")
                plt.xlabel("mean gate value"); plt.legend(fontsize=7); plt.title(f"Gate distribution ({r['run_id']}, validation)")
                save_fig(f"P_fig_gate_{r['run_id']}")
    q = build_main_queue()
    done = [x for x in q if run_done(x)]
    print(f"\n[status] main queue {len(done)}/{len(q)} complete; zero-shot "
          f"{'done' if run_done('zeroshot_mbart50') else 'pending'}; session time left {session_hours_left():.1f}h")



# ==============================================================================
# 16b. PAPER STYLE, ENVIRONMENT DRIFT DIAGNOSTIC, SUPERSEDING, DATASET / VALIDATION FIGURES
# ==============================================================================
COND_LABEL = {"zeroshot_mbart50": "Zero-shot mBART-50", "baseline": "Baseline", "pscab": "PS-CAB",
              "pscab_no_gate": "PS-CAB −gate", "pscab_no_gamma": "PS-CAB −γ (fixed)",
              "pscab_random_mask": "Random mask", "pscab_all_tokens": "All tokens"}
COND_COLOR = {"zeroshot_mbart50": "#7f7f7f", "baseline": "#1f77b4", "pscab": "#d62728",
              "pscab_no_gate": "#ff7f0e", "pscab_no_gamma": "#9467bd",
              "pscab_random_mask": "#2ca02c", "pscab_all_tokens": "#8c564b"}
COND_ORDER = ["baseline", "pscab", "pscab_no_gate", "pscab_no_gamma", "pscab_random_mask", "pscab_all_tokens"]


def apply_paper_style():
    plt.rcParams.update({"font.size": 9, "axes.titlesize": 10, "axes.labelsize": 9, "legend.fontsize": 7.5,
                         "xtick.labelsize": 8, "ytick.labelsize": 8, "figure.dpi": 110,
                         "axes.spines.top": False, "axes.spines.right": False, "axes.grid": True,
                         "grid.alpha": 0.25, "savefig.bbox": "tight"})


def rec_tf_version(r):
    return (r.get("env") or {}).get("transformers")


def _records_incl_superseded(kinds=("main", "zeroshot")):
    out = []
    for folder, sup in ((DIRS["runs"], False), (f"{R}/runs_superseded", True)):
        for p in sorted(glob.glob(f"{folder}/*.json")):
            r = jload(p)
            if r and r.get("status") == "complete" and r.get("kind") in kinds:
                r["_superseded"] = sup
                out.append(r)
    return out


def env_drift_diagnostic():
    """E2: one-time appendix diagnostic. Groups every run by transformers version and compares
    output length ratio / BP / BLEU / chrF++ / TER / val NLL; optionally re-decodes the old-environment
    pscab_s42 model (.h5) in the current environment."""
    cache = f"{DIRS['analysis']}/env_drift_diagnostic.json"
    if os.path.exists(cache):
        c = jload(cache)
        print(f"\n[drift] cached diagnostic (versions seen: {c.get('versions')}); re-decode: {c.get('redecode')}")
        return c
    recs = [r for r in _records_incl_superseded() if not r["_superseded"]]
    versions = sorted({str(rec_tf_version(r) or "unknown") for r in recs})
    if not recs or versions == [ENV_PIN["transformers"]]:
        return None
    print(f"\n{'#'*72}\nENVIRONMENT DRIFT DIAGNOSTIC (appendix) -- versions present: {versions}\n{'#'*72}")
    _, val, _ = load_splits()
    refs = val["te"].tolist()
    B = bleu_metric()
    rows = []
    for r in recs:
        pr = load_preds(r["run_id"], "val")
        if pr is None:
            continue
        sc = B.corpus_score(pr, [refs])
        rows.append({"run": r["run_id"], "condition": r["condition"], "transformers": str(rec_tf_version(r) or "unknown"),
                     "val_nll": r.get("best_val_nll"), "BLEU": round(sc.score, 2), "BP": round(sc.bp, 4),
                     "len_ratio": round(sc.sys_len / sc.ref_len, 4), "chrF++": r["val_metrics"]["chrf++"],
                     "TER": r["val_metrics"]["ter"], "mean_hyp_words": round(float(np.mean([len(x.split()) for x in pr])), 2)})
    d = pd.DataFrame(rows)
    print(d.to_string(index=False))
    save_table(d, "DIAG_env_drift_per_run", "Validation outputs grouped by transformers version (appendix)")

    redecode = None
    ps = jload(run_record_path("pscab_s42"))
    h5s = glob.glob(f"{DIRS['models']}/pscab_s42/*.h5")
    if (CFG["env"]["drift_regenerate"] and torch.cuda.is_available() and ps and h5s
            and rec_tf_version(ps) != ENV_PIN["transformers"]):
        try:
            print(f"[drift] re-decoding pscab_s42 (trained under transformers {rec_tf_version(ps)}) "
                  f"with transformers {ENV_PIN['transformers']} ...")
            m = PSCABMBart("pscab")
            with h5py.File(h5s[0], "r") as f:
                sd = {k: torch.from_numpy(f[k][()]).float() for k in f.keys()}
            miss, unexp = m.load_state_dict(sd, strict=False)
            m.to(DEVICE).eval()
            new, _ = generate(m, get_tokenizer(), val, "true")
            old = load_preds("pscab_s42", "val")
            so, sn = B.corpus_score(old, [refs]), B.corpus_score(new, [refs])
            mn = compute_metrics(new, refs)
            redecode = {"old_env": rec_tf_version(ps), "new_env": ENV_PIN["transformers"],
                        "missing_keys": len(miss), "unexpected_keys": len(unexp),
                        "old_bleu": round(so.score, 2), "new_bleu": round(sn.score, 2),
                        "old_len_ratio": round(so.sys_len / so.ref_len, 4), "new_len_ratio": round(sn.sys_len / sn.ref_len, 4),
                        "old_chrf": ps["val_metrics"]["chrf++"], "new_chrf": mn["chrf++"],
                        "old_ter": ps["val_metrics"]["ter"], "new_ter": mn["ter"],
                        "identical_outputs_share": round(float(np.mean([a == b for a, b in zip(old, new)])), 4),
                        "note": "same weights (fp16 .h5 snapshot), only the decoding environment differs"}
            redecode["verdict"] = ("DRIFT CONFIRMED" if abs(redecode["new_bleu"] - redecode["old_bleu"]) > 0.5
                                   else "no material drift (<0.5 BLEU)")
            print("[drift] re-decode:", json.dumps(redecode, indent=2))
            pd.DataFrame({"en": val["en"], "te": val["te"], "prediction": new}).to_csv(
                f"{DIRS['analysis']}/pscab_s42_REDECODED_val.csv", index=False)
            del m
            free_gpu()
        except Exception as e:
            redecode = {"error": str(e)}
            print(f"[drift] re-decode failed: {e}")
    else:
        print("[drift] re-decode skipped (needs GPU + models/pscab_s42/*.h5 + an old-env pscab_s42 record).")

    if len(d):
        fig, ax = plt.subplots(1, 2, figsize=(10, 3.6))
        vs = sorted(d["transformers"].unique())
        cmap = {v: c for v, c in zip(vs, ["#1f77b4", "#d62728", "#2ca02c", "#9467bd"])}
        ax[0].bar(range(len(d)), d["len_ratio"], color=[cmap[v] for v in d["transformers"]])
        ax[0].set_xticks(range(len(d))); ax[0].set_xticklabels(d["run"], rotation=40, ha="right", fontsize=7)
        ax[0].axhline(1.0, ls="--", c="gray", lw=.8); ax[0].set_ylabel("hyp/ref length (spBLEU tokens)")
        ax[0].set_title("(a) Output length by run")
        ax[0].set_ylim(min(d["len_ratio"].min(), 1) - 0.03, max(d["len_ratio"].max(), 1) + 0.03)
        for v in vs:
            sub = d[d["transformers"] == v]
            ax[1].scatter(sub["val_nll"], sub["BLEU"], c=cmap[v], label=f"transformers {v}", s=40)
        if redecode and "new_bleu" in redecode:
            ax[1].annotate("pscab_s42 re-decoded", xy=(ps["best_val_nll"], redecode["new_bleu"]), fontsize=7)
            ax[1].scatter([ps["best_val_nll"]], [redecode["new_bleu"]], marker="*", s=140, c="k")
        ax[1].set_xlabel("best validation NLL (teacher-forced)"); ax[1].set_ylabel("validation spBLEU")
        ax[1].set_title("(b) Decoding vs training quality"); ax[1].legend()
        save_fig("DIAG_fig_env_drift")
    out = {"versions": versions, "per_run": rows, "redecode": redecode, "pinned": ENV_PIN}
    jdump(out, cache)
    return out


def supersede_other_env_runs():
    """E3: move completed runs from a different transformers version out of the active set."""
    if not CFG["env"]["supersede_other_env"]:
        return []
    for d in ("preds_superseded", "logs_superseded", "runs_superseded"):
        os.makedirs(f"{R}/{d}", exist_ok=True)
    moved = []
    for p in sorted(glob.glob(f"{DIRS['runs']}/*.json")):
        r = jload(p)
        if not r or r.get("status") != "complete":
            continue
        v = rec_tf_version(r)
        if v is None and r.get("kind") == "cv" and CFG["env"]["backfill_cv_env"]:
            r.setdefault("env", {})
            r["env"].update({"transformers": ENV_PIN["transformers"], "backfilled_from_session_log": True})
            jdump(r, p)
            print(f"[supersede] {r['run_id']}: env back-filled ({ENV_PIN['transformers']}, from the session log)")
            continue
        if v == ENV_PIN["transformers"]:
            continue
        rid = r["run_id"]
        os.replace(p, f"{R}/runs_superseded/{rid}.json")
        for f in glob.glob(f"{DIRS['preds']}/{rid}__*.csv"):
            os.replace(f, f"{R}/preds_superseded/{os.path.basename(f)}")
        for f in glob.glob(f"{DIRS['logs']}/{rid}_curve.csv"):
            os.replace(f, f"{R}/logs_superseded/{os.path.basename(f)}")
        moved.append({"run_id": rid, "transformers": v, "moved": time.strftime("%Y-%m-%d %H:%M:%S")})
    if moved:
        log_p = f"{DIRS['analysis']}/superseded_runs.json"
        jdump((jload(log_p) or []) + moved, log_p)
        print(f"[supersede] {len(moved)} run(s) from another environment re-queued: {[m['run_id'] for m in moved]}")
    return moved


def dataset_figures():
    """Descriptive statistics of the frozen splits (no model output involved)."""
    if os.path.exists(f"{DIRS['figures']}/D_fig_lengths.pdf"):
        return
    print(f"\n{'#'*72}\nDATASET FIGURES\n{'#'*72}")
    splits = dict(zip(["train", "val", "test"], load_splits()))
    rows, cnts = [], {}
    for name, d in splits.items():
        en_l, te_l = d["en"].str.split().str.len(), d["te"].str.split().str.len()
        ensure_spans(d["en"].tolist())
        c = np.array([len(_SPAN_CACHE[x]) for x in d["en"]])
        cnts[name] = c
        rows.append({"split": name, "pairs": len(d), "EN_words_mean": round(en_l.mean(), 2), "EN_words_median": int(en_l.median()),
                     "TE_words_mean": round(te_l.mean(), 2), "TE_words_median": int(te_l.median()),
                     "TE/EN_word_ratio": round(te_l.sum() / en_l.sum(), 3),
                     "%_with_preposition": round(100 * (c > 0).mean(), 2), "prepositions_per_sentence": round(c.mean(), 3)})
    t = pd.DataFrame(rows)
    print(t.to_string(index=False))
    save_table(t, "D1_split_statistics", "Statistics of the frozen splits")

    fig, ax = plt.subplots(1, 2, figsize=(9, 3.2))
    bins = np.arange(0, 61, 2)
    for name, d in splits.items():
        ax[0].hist(d["en"].str.split().str.len().clip(upper=60), bins, histtype="step", density=True, lw=1.4, label=name)
        ax[1].hist(d["te"].str.split().str.len().clip(upper=60), bins, histtype="step", density=True, lw=1.4, label=name)
    ax[0].set_title("(a) English source length"); ax[1].set_title("(b) Telugu target length")
    for a in ax:
        a.set_xlabel("words (clipped at 60)"); a.set_ylabel("density"); a.legend()
    save_fig("D_fig_lengths")

    fig, ax = plt.subplots(1, 2, figsize=(9.5, 3.4))
    labels = ["0", "1", "2", "3", "4+"]
    w = 0.27
    for i, (name, c) in enumerate(cnts.items()):
        share = [100 * np.mean(np.minimum(c, 4) == k) for k in range(5)]
        ax[0].bar(np.arange(5) + (i - 1) * w, share, w, label=name)
    ax[0].set_xticks(range(5)); ax[0].set_xticklabels(labels)
    ax[0].set_xlabel("prepositions per source sentence"); ax[0].set_ylabel("% of sentences")
    ax[0].set_title("(a) Preposition density"); ax[0].legend()
    from collections import Counter
    words = Counter(w_ for s_ in splits["train"]["en"] for _, _, w_ in _SPAN_CACHE[s_])
    top = words.most_common(20)[::-1]
    ax[1].barh([k for k, _ in top], [v for _, v in top], color="#d62728")
    ax[1].set_xlabel("occurrences (train)"); ax[1].set_title("(b) 20 most frequent prepositions")
    save_fig("D_fig_prepositions")

    d = splits["train"]
    plt.figure(figsize=(4.6, 4))
    plt.hexbin(d["en"].str.split().str.len().clip(upper=60), d["te"].str.split().str.len().clip(upper=60),
               gridsize=35, mincnt=1, bins="log", cmap="viridis")
    plt.plot([0, 60], [0, 60], ls="--", c="w", lw=.8)
    plt.colorbar(label="log10(count)")
    plt.xlabel("English words"); plt.ylabel("Telugu words")
    plt.title("Telugu is agglutinative: fewer words per sentence")
    save_fig("D_fig_en_te_length_hexbin")


def _curve(rid, folder=None):
    p = f"{folder or DIRS['logs']}/{rid}_curve.csv"
    return pd.read_csv(p) if os.path.exists(p) else None


def validation_figures():
    """V_fig_*: validation-only figures over all completed main runs (current environment)."""
    recs = all_records("main")
    if not recs:
        return
    by_c = defaultdict(list)
    for r in recs:
        by_c[r["condition"]].append(r)
    conds = [c for c in COND_ORDER if c in by_c]

    # V1 learning curves: mean over seeds with min-max band
    plt.figure(figsize=(7.5, 4))
    for c in conds:
        cs = [x for x in (_curve(r["run_id"]) for r in by_c[c]) if x is not None]
        if not cs:
            continue
        L = min(len(x) for x in cs)
        M = np.stack([x["val_nll"].values[:L] for x in cs])
        st = cs[0]["step"].values[:L]
        plt.plot(st, M.mean(0), marker="o", ms=3, c=COND_COLOR[c], label=f"{COND_LABEL[c]} (n={len(cs)})")
        if len(cs) > 1:
            plt.fill_between(st, M.min(0), M.max(0), color=COND_COLOR[c], alpha=.15)
    plt.xlabel("optimizer step"); plt.ylabel("validation NLL / token")
    plt.title("Validation NLL (identical step budget; band = seed range)"); plt.legend(ncol=2)
    save_fig("V_fig_learning_curves")

    # V2 validation metrics with seed dots
    fig, ax = plt.subplots(1, 2, figsize=(10, 3.6))
    for a, key, lab in ((ax[0], "chrf++", "chrF++"), (ax[1], "bleu", "spBLEU")):
        for i, c in enumerate(conds):
            v = [r["val_metrics"][key] for r in by_c[c]]
            a.bar(i, np.mean(v), color=COND_COLOR[c], alpha=.55)
            a.scatter([i] * len(v), v, c="k", s=14, zorder=3)
        a.set_xticks(range(len(conds))); a.set_xticklabels([COND_LABEL[c] for c in conds], rotation=25, ha="right")
        allv = [r["val_metrics"][key] for r in recs]
        a.set_ylim(min(allv) - 1.5, max(allv) + 1); a.set_ylabel(f"validation {lab}")
    ax[0].set_title("(a) chrF++ (bars = mean, dots = seeds)"); ax[1].set_title("(b) spBLEU")
    save_fig("V_fig_validation_metrics")

    # V3 module diagnostics
    dg = [(r["run_id"], r["condition"], r["diagnostics"]) for r in recs if r.get("diagnostics")]
    if dg:
        fig, ax = plt.subplots(1, 3, figsize=(12, 3.6))
        x = np.arange(len(dg))
        names = [d[0] for d in dg]
        ax[0].bar(x, [d[2].get("gamma", np.nan) for d in dg], color=[COND_COLOR[d[1]] for d in dg])
        ax[0].axhline(0, c="gray", lw=.8); ax[0].set_title("(a) learned scale γ")
        ax[1].bar(x, [d[2].get("bias_to_hidden_norm_ratio_mean", np.nan) for d in dg], color=[COND_COLOR[d[1]] for d in dg])
        ax[1].set_title("(b) ‖γ·g·W_bH‖ / ‖H‖ on masked tokens")
        tr = [d[2].get("gate_trained_auc_prep_vs_other", np.nan) or np.nan for d in dg]
        it = [d[2].get("gate_init_auc_prep_vs_other", np.nan) or np.nan for d in dg]
        ax[2].bar(x - .2, it, .4, label="gate at init", color="#bbbbbb")
        ax[2].bar(x + .2, tr, .4, label="gate trained", color="#d62728")
        ax[2].axhline(.5, ls="--", c="k", lw=.8); ax[2].set_ylim(0, 1)
        ax[2].set_title("(c) gate AUC: preposition vs other tokens"); ax[2].legend()
        for a in ax:
            a.set_xticks(x); a.set_xticklabels(names, rotation=40, ha="right", fontsize=7)
        save_fig("V_fig_module_diagnostics")

    # V4 attention mass, V5 robustness (seed-0 runs)
    at = [(r["run_id"], r["condition"], r["attention"]) for r in recs if r.get("attention")]
    if at:
        fig, ax = plt.subplots(1, 2, figsize=(10, 3.4))
        x = np.arange(len(at))
        ax[0].bar(x, [a[2]["prep_mass_last_layer_over_uniform"] for a in at], color=[COND_COLOR[a[1]] for a in at])
        ax[0].axhline(1, ls="--", c="k", lw=.8)
        ax[0].set_title("(a) last-layer attention on prepositions ÷ uniform share")
        ax[1].bar(x, [a[2]["prep_mass_mean_layers"] for a in at], color=[COND_COLOR[a[1]] for a in at])
        ax[1].set_title("(b) mean-over-layers attention mass on prepositions")
        for a in ax:
            a.set_xticks(x); a.set_xticklabels([q[0] for q in at], rotation=35, ha="right", fontsize=7)
        save_fig("V_fig_attention_mass")
    rb = [(r["run_id"], r["condition"], r["robustness_val"]) for r in recs if r.get("robustness_val")]
    if rb:
        perts = [p_ for p_ in PERTURBATIONS]
        plt.figure(figsize=(8, 3.6))
        w = 0.8 / len(rb)
        for i, (rid, c, rows) in enumerate(rb):
            dmap = {q["perturbation"]: q["delta_chrf++"] for q in rows}
            plt.bar(np.arange(len(perts)) + i * w, [dmap.get(p_, np.nan) for p_ in perts], w,
                    color=COND_COLOR[c], label=rid)
        plt.axhline(0, c="gray", lw=.8)
        plt.xticks(np.arange(len(perts)) + w * (len(rb) - 1) / 2, perts)
        plt.ylabel("Δ chrF++ vs unperturbed"); plt.title("Robustness to source perturbations (validation)")
        plt.legend(fontsize=6.5, ncol=2)
        save_fig("V_fig_robustness")

    # V6 output length ratio per run (validation)
    _, val, _ = load_splits()
    refs = val["te"].tolist()
    B = bleu_metric()
    rows = []
    for r in all_records():
        if r.get("kind") not in ("main", "zeroshot"):
            continue
        pr = load_preds(r["run_id"], "val")
        if pr is None:
            continue
        sc = B.corpus_score(pr, [refs])
        rows.append((r["run_id"], r["condition"], sc.sys_len / sc.ref_len))
    if rows:
        plt.figure(figsize=(8, 3.4))
        plt.bar(range(len(rows)), [q[2] for q in rows], color=[COND_COLOR.get(q[1], "#777") for q in rows])
        plt.axhline(1, ls="--", c="k", lw=.8)
        plt.xticks(range(len(rows)), [q[0] for q in rows], rotation=40, ha="right", fontsize=7)
        vals = [q[2] for q in rows]
        plt.ylim(min(vals + [1]) - .03, max(vals + [1]) + .03)
        plt.ylabel("hyp/ref length (spBLEU tokens)"); plt.title("Output length ratio on validation")
        save_fig("V_fig_output_length_ratio")


# ==============================================================================
# 17. PART 1 DRIVER
# ==============================================================================
apply_paper_style()
restore_previous_results()
jdump(CFG, f"{R}/config_used.json")
print("[env]", environment_info())
bleu_metric()
DATA_SUMMARY = stage_data_prep()
DRIFT = env_drift_diagnostic()            # E2 (once, cached; must run before superseding)
SUPERSEDED = supersede_other_env_runs()   # E3
try:
    dataset_figures()
except Exception as _e:
    print(f"[figures] dataset_figures failed: {_e}")

# "auto": run Part 1 training only while its 3 items (zero-shot, baseline_s42, pscab_s42) are incomplete.
# In later sessions this cell then just loads the library + restored results (fast) before Part 2/3.
RUN_PART1 = "auto"
_part1_items = ["zeroshot_mbart50"] + build_main_queue()[:2]
if RUN_PART1 is True or (RUN_PART1 == "auto" and not all(run_done(x) for x in _part1_items)):
    execute_zeroshot()
    for _rid in build_main_queue()[:2]:          # baseline_s42, pscab_s42
        if execute_main_run(_rid) is None:
            break
    progress_report()
    print("\nPART 1 finished. Save the version; next session attach its output as input, run Part 1 then Part 2.")
else:
    print("[part1] core runs already complete -- library loaded, results restored. Now run Part 2 or Part 3.")

[pin] installing sacrebleu==2.6.0 (found None)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 100.8/100.8 kB 2.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 129.6/129.6 kB 3.5 MB/s eta 0:00:00
[restore] 2 previous output(s) found; newest = /kaggle/input/notebooks/panjanagalaxmi/notebook80ed48e07f/pscab_q1 (2026-10-12 06:18:51)
[restore] copying previous results from /kaggle/input/notebooks/panjanagalaxmi/notebook80ed48e07f/pscab_q1 (15 run records) ...
[env] {'python': '3.13.15', 'torch': '2.11.0+cpu', 'cuda': None, 'gpu': None, 'sacrebleu': '2.6.0', 'spacy': True, 'env_pin': {'transformers': '5.16.1', 'sacrebleu': '2.6.0'}, 'transformers': '5.16.1'}
[metrics] BLEU tokenizer = flores200
[data] frozen split found -- reusing it.
{
  "train-val": {
    "exact_en": 0,
    "exact_te": 0,
    "exact_pair": 0,
    "near_dup_en": 0,
    "near_dup_te": 0
  },
  "train-test": {
    "exact_en": 0,
    "exact_te": 0,
    "exact_pair": 0,
    "near_dup_en": 0,
    "near_dup_te": 0
  }

In [8]:
"""
================================================================================
PART 3 / 3 (v3) -- GROUPED K-FOLD CROSS-VALIDATION + FINAL TEST EVALUATION & REPORT
================================================================================
v3 CHANGES:
  V3-1  CV records store env + generation config; the final evaluation refuses to mix
        transformers versions (all runs must come from the pinned environment of Part 1).
  V3-2  PAPER FIGURES FIG01-FIG17 (architecture, main results, effect forest, per-seed deltas,
        H2, per-preposition heatmap, density/length, sentence-level win/tie/loss, output
        length, metric agreement, gates, module magnitude, attention + robustness, CV,
        errors + cost, learning curves, external sets), a qualitative-examples table
        (LaTeX/Markdown/CSV) and PAPER_ASSETS.md with a draft caption for every figure.
  V3-3  PAPER_FIGURES_ONLY: regenerate all figures from saved results without re-scoring.

Run the Part 1 cell first in the session (with completed runs restored it does not retrain).
Each execution of this cell:
  1. trains up to 2 pending cross-validation runs (baseline / PS-CAB per fold),
  2. prints the CV analysis for the folds completed so far,
  3. if the main queue (Parts 1-2) is complete, performs the FINAL held-out test evaluation.

Cross-validation design: pool = train + validation (test split never used), K group-folds
(near-duplicate groups never straddle folds). Fold f: held-out = fold f, early-stopping set =
2,000 sentences from the other folds, training = the rest. Same step budget and seed.
The folds are five partitions of ONE dataset -- report them as "five-fold grouped
cross-validation", not as five independent experiments.

What changed vs the previous Part 3 (reporting / statistics only; no training change):
  P3-1  SD / CI shown as "N/A" (never 0) when a condition has a single seed.
  P3-2  FORCE_FINAL output is stamped PRELIMINARY and written to a different file; the final
        summary is only written when the whole main queue is complete.
  P3-3  External benchmarks reported twice: contamination-FILTERED (what Part 1 froze, with the
        number removed) and OFFICIAL FULL sets (scored with the saved baseline_s42 / pscab_s42).
  P3-4  Hierarchical bootstrap relabelled "seed-aware uncertainty characterisation"; new per-seed
        consistency table (deltas per seed, #seeds better, paired t over seeds) is the primary
        seed-level evidence; Holm-ADJUSTED p added to the per-seed paired tests.
  P3-5  CV wording/notes corrected (folds are not independent replications).
  P3-6  Human-evaluation agreement (Cohen's kappa) computed once two annotated sheets exist.
  P3-7  COMET can be switched off (USE_COMET) and is never installed during CV-only sessions;
        the evidence report states dataset-selection rationale and the pretraining-contamination limit.
================================================================================
"""
assert "run_next_main_runs" in globals(), "Run the Part 1 cell first in this session."

CV_RUNS_THIS_SESSION = 2
DO_CV = False             # set False to skip CV training this session
FORCE_FINAL = False      # keep False for the paper. True = preliminary look only (stamped PRELIMINARY)
USE_COMET = True         # False = skip COMET (score saved prediction CSVs separately later)
RUN_FULL_EXTERNAL = True                       # score OFFICIAL full external sets with the saved models
PAPER_FIGURES = True     # build FIG01-FIG17 + qualitative table + PAPER_ASSETS.md after the final evaluation
FULL_EXTERNAL_FOR = ["baseline_s42", "pscab_s42"]   # model bundles Part 1 saved to disk

from scipy import stats as sps


# ==============================================================================
# 0. SMALL HELPERS
# ==============================================================================
def holm_adjust(pvals: dict):
    items = sorted(pvals.items(), key=lambda kv: kv[1])
    m, out, running = len(items), {}, 0.0
    for r, (k, p) in enumerate(items):
        running = max(running, min(1.0, (m - r) * p))
        out[k] = running
    return out


def na_df(df):
    """Display/save copy: NaN -> 'N/A' (SD with a single observation is undefined, not zero)."""
    return df.astype(object).where(df.notna(), "N/A")


# ==============================================================================
# A. CROSS-VALIDATION
# ==============================================================================
def build_cv_folds():
    p = f"{DIRS['cv']}/cv_folds.csv"
    if os.path.exists(p):
        return read_csv_str(p)
    train, val, _ = load_splits()
    pool = pd.concat([train, val], ignore_index=True)
    k = CFG["cv"]["k"]
    sizes = pool.groupby("group_id").size()
    order = list(sizes.index)
    np.random.default_rng(CFG["data"]["split_seed"] + 1).shuffle(order)
    fold_size, assign = [0] * k, {}
    for gname in order:                                   # greedy: next group -> smallest fold
        f = int(np.argmin(fold_size))
        assign[gname] = f
        fold_size[f] += int(sizes[gname])
    pool["fold"] = pool["group_id"].map(assign).astype(int)
    pool.to_csv(p, index=False)
    print(f"[cv] built {k} group-folds over {len(pool)} pairs: sizes={fold_size}")
    return read_csv_str(p)


def cv_queue():
    k, seed = CFG["cv"]["k"], CFG["cv"]["seed"]
    return [f"cv{f}_{m}_s{seed}" for f in range(k) for m in CFG["cv"]["models"]]


def cv_fold_data(fold):
    pool = build_cv_folds()
    pool["fold"] = pool["fold"].astype(int)
    held = pool[pool["fold"] == fold].reset_index(drop=True)
    rest = pool[pool["fold"] != fold].reset_index(drop=True)
    groups = rest["group_id"].unique().tolist()
    np.random.default_rng(1000 + fold).shuffle(groups)
    chosen, n = set(), 0
    gsize = rest.groupby("group_id").size()
    for gname in groups:
        if n >= CFG["cv"]["inner_val_size"]:
            break
        chosen.add(gname); n += int(gsize[gname])
    inner_val = rest[rest["group_id"].isin(chosen)].reset_index(drop=True)
    tr = rest[~rest["group_id"].isin(chosen)].reset_index(drop=True)
    assert not (set(tr["group_id"]) & set(held["group_id"])) and not (set(inner_val["group_id"]) & set(held["group_id"]))
    assert len(tr) > 0, "CV training set is empty -- reduce CFG['cv']['inner_val_size']."
    return tr, inner_val, held


def execute_cv_run(run_id):
    if run_done(run_id):
        print(f"[skip] {run_id} already complete.")
        return jload(run_record_path(run_id))
    need = estimated_run_hours()
    if session_hours_left() < need:
        print(f"[time] {run_id} needs ~{need:.1f}h, {session_hours_left():.1f}h left -> not started.")
        return None
    cond, seed, fold = parse_run_id(run_id)
    t0 = time.time()
    tr, iv, held = cv_fold_data(fold)
    print(f"[cv] {run_id}: train={len(tr)} early-stop={len(iv)} held-out={len(held)}")
    model, tok, meta, _ = train_model(cond, seed, tr, iv, run_id)
    gen_rec = generation_record(model, tok)
    hp, info = generate(model, tok, held, CONDITIONS[cond]["mask"])
    save_preds(run_id, "heldout", held, hp)
    rec = {"run_id": run_id, "kind": "cv", "condition": cond, "seed": seed, "fold": fold, "status": "complete",
           "n_train": len(tr), "n_early_stop": len(iv), "n_heldout": len(held),
           "heldout_metrics": compute_metrics(hp, held["te"].tolist()), "env": environment_info(),
           "generation": gen_rec, **meta}
    rec["val_metrics"] = rec["heldout_metrics"]          # for the generic progress table
    del model
    free_gpu()
    rec["wall_hours_total"] = (time.time() - t0) / 3600
    jdump(rec, run_record_path(run_id))
    print(f"[cv] {run_id} held-out {rec['heldout_metrics']['bleu']} BLEU / {rec['heldout_metrics']['chrf++']} chrF++")
    return rec


def run_next_cv_runs(max_runs=2):
    q = cv_queue()
    pending = [r for r in q if not run_done(r)]
    print(f"[cv-queue] {len(q) - len(pending)}/{len(q)} complete; next: {pending[:max_runs]}")
    started = 0
    for rid in pending:
        if started >= max_runs:
            break
        if execute_cv_run(rid) is None:
            break
        started += 1


def cv_report():
    print(f"\n{'#'*72}\nCROSS-VALIDATION REPORT (five-fold grouped CV; held-out folds; test split untouched)\n{'#'*72}")
    k, seed = CFG["cv"]["k"], CFG["cv"]["seed"]
    rows, pairs = [], []
    for f in range(k):
        b, p = f"cv{f}_baseline_s{seed}", f"cv{f}_pscab_s{seed}"
        for rid in (b, p):
            if run_done(rid):
                r = jload(run_record_path(rid))
                rows.append({"fold": f, "model": r["condition"], "n_heldout": r["n_heldout"], "n_train": r["n_train"],
                             "best_step": r["best_step"], "BLEU": r["heldout_metrics"]["bleu"],
                             "chrF++": r["heldout_metrics"]["chrf++"], "TER": r["heldout_metrics"]["ter"]})
        if run_done(b) and run_done(p):
            pairs.append(f)
    if not rows:
        print("no CV run completed yet.")
        return None
    df = pd.DataFrame(rows)
    print(df.to_string(index=False))
    save_table(df, "F14a_cv_per_fold")
    if len(pairs) < 2:
        print(f"[cv] {len(pairs)} complete fold pair(s); statistics need >= 2.")
        return df
    out = []
    for m in ("BLEU", "chrF++"):
        b = df[df["model"] == "baseline"].set_index("fold").loc[pairs, m].values
        p = df[df["model"] == "pscab"].set_index("fold").loc[pairs, m].values
        d = p - b
        n_te = df[df["fold"].isin(pairs)]["n_heldout"].mean()
        n_tr = df[df["fold"].isin(pairs)]["n_train"].mean()
        kk = len(d)
        var = np.var(d, ddof=1) if kk > 1 else np.nan
        t_nb = (d.mean() / math.sqrt((1 / kk + n_te / n_tr) * var)
                if (n_tr and n_tr > 0 and np.isfinite(var) and var > 0) else np.nan)
        p_nb = 2 * sps.t.sf(abs(t_nb), kk - 1) if np.isfinite(t_nb) else np.nan
        try:
            p_w = sps.wilcoxon(d).pvalue if kk >= 2 and np.any(d != 0) else np.nan
        except Exception:
            p_w = np.nan
        out.append({"metric": m, "folds": kk, "baseline_mean": b.mean(), "baseline_sd": b.std(ddof=1),
                    "pscab_mean": p.mean(), "pscab_sd": p.std(ddof=1), "mean_delta": d.mean(), "sd_delta": d.std(ddof=1),
                    "folds_pscab_better": int((d > 0).sum()), "p_corrected_resampled_t": p_nb, "p_wilcoxon": p_w})
    # pooled out-of-fold approximate randomization
    pa, pb, rr = [], [], []
    for f in pairs:
        A = read_csv_str(f"{DIRS['preds']}/cv{f}_pscab_s{seed}__heldout.csv")
        B = read_csv_str(f"{DIRS['preds']}/cv{f}_baseline_s{seed}__heldout.csv")
        pa += A["prediction"].tolist(); pb += B["prediction"].tolist(); rr += A["te"].tolist()
    for mn in ("bleu", "chrf++"):
        M = StatMetric(mn)
        res = paired_test(M.stats(pa, rr), M.stats(pb, rr), M, n_boot=1000, n_ar=5000)
        for o in out:
            if o["metric"].lower().startswith(mn[:4]):
                o.update({"pooled_oof_delta": res["delta"], "pooled_oof_ci": f"[{res['ci95_low']}, {res['ci95_high']}]",
                          "pooled_oof_p_ar": res["p_ar"], "pooled_n": res["n"]})
    s = pd.DataFrame(out)
    print("\n" + s.to_string(index=False))
    print("[cv note] These are the folds of ONE dataset with one training seed, not independent experiments: "
          "write 'five-fold grouped cross-validation'. Wilcoxon with k=5 cannot go below p=0.0625 (two-sided); "
          "rely on the corrected resampled t-test and the pooled out-of-fold AR, and report per-fold values + mean ± SD.")
    save_table(s, "F14b_cv_summary", "Five-fold grouped cross-validation (PS-CAB minus baseline)")
    plt.figure(figsize=(6, 4))
    for f in pairs:
        b = df[(df.fold == f) & (df.model == "baseline")]["chrF++"].values[0]
        p = df[(df.fold == f) & (df.model == "pscab")]["chrF++"].values[0]
        plt.plot([0, 1], [b, p], marker="o", color="tab:green" if p > b else "tab:red")
    plt.xticks([0, 1], ["baseline", "PS-CAB"]); plt.ylabel("held-out chrF++"); plt.title(f"{len(pairs)}-fold CV (paired by fold)")
    save_fig("F_fig_cv_paired")
    return s

# ==============================================================================
# B. COMET (installed without touching torch / numpy; only called from the final evaluation)
# ==============================================================================
_COMET = None


def get_comet():
    global _COMET
    if _COMET is not None:
        return _COMET
    _COMET = False
    if not USE_COMET:
        print("[comet] disabled (USE_COMET=False).")
        return _COMET
    tv = torch.__version__
    if importlib.util.find_spec("comet") is None:
        print("[comet] installing unbabel-comet (--no-deps, keeps torch/numpy unchanged) ...")
        subprocess.run([_sys.executable, "-m", "pip", "install", "-q", "--no-deps", "unbabel-comet", "entmax", "jsonargparse"])
        for pkg, mod in (("pytorch-lightning", "pytorch_lightning"), ("torchmetrics", "torchmetrics")):
            if importlib.util.find_spec(mod) is None:
                subprocess.run([_sys.executable, "-m", "pip", "install", "-q", pkg])
        importlib.invalidate_caches()
    try:
        import types
        from comet import download_model, load_from_checkpoint
        cm = load_from_checkpoint(download_model("Unbabel/wmt22-comet-da"))

        # compatibility patch for transformers 5.x: read outputs by name, not by tuple position
        def _xlmr_forward(self, input_ids, attention_mask, **kwargs):
            out = self.model(input_ids=input_ids, attention_mask=attention_mask,
                             output_hidden_states=True, return_dict=True)
            last = out.last_hidden_state
            return {"sentemb": last[:, 0, :], "wordemb": last,
                    "all_layers": out.hidden_states, "attention_mask": attention_mask}
        cm.encoder.forward = types.MethodType(_xlmr_forward, cm.encoder)

        _COMET = cm
        print(f"[comet] Unbabel/wmt22-comet-da loaded + transformers-5 patch applied (torch {tv})")
    except Exception as e:
        print(f"[comet] unavailable ({e}). COMET reported as null. Score the saved prediction CSVs "
              f"in a separate notebook with `comet-score`; no training run needs to be repeated.")
    return _COMET


def comet_segments(run_id, split_name, df_src):
    global _COMET
    path = f"{DIRS['analysis']}/comet__{run_id}__{split_name}.npy"
    if os.path.exists(path):
        return np.load(path)
    cm = get_comet()
    if not cm:
        return None
    preds = load_preds(run_id, split_name)
    data = [{"src": s, "mt": h, "ref": r} for s, h, r in zip(df_src["en"], preds, df_src["te"])]
    try:
        out = cm.predict(data, batch_size=64, gpus=1 if torch.cuda.is_available() else 0, progress_bar=False)
    except Exception as e:
        print(f"[comet] predict failed ({type(e).__name__}: {e}). COMET switched off for this session; "
              f"score the saved prediction CSVs separately with comet-score.")
        _COMET = False
        return None
    arr = np.asarray(out.scores, dtype=np.float64)
    np.save(path, arr)
    return arr


# ==============================================================================
# C. SEED-LEVEL AND SUBSET STATISTICS
# ==============================================================================
def hier_bootstrap(A_list, B_list, metric, N, n=2000, seed=7):
    """Paired hierarchical bootstrap over seeds AND sentences (mean over seeds of A_s - B_s).
    With only K=3 seeds this is a seed-aware uncertainty characterisation, NOT 3 independent
    replications multiplied by N sentences."""
    rng = np.random.default_rng(seed)
    K = len(A_list)
    obs = np.mean([metric.score(a.sum(0), N) - metric.score(b.sum(0), N) for a, b in zip(A_list, B_list)])
    d = np.empty(n)
    for i in range(n):
        ks = rng.integers(0, K, K)
        idx = rng.integers(0, N, N)
        d[i] = np.mean([metric.score(A_list[k][idx].sum(0), N) - metric.score(B_list[k][idx].sum(0), N) for k in ks])
    lo, hi = np.percentile(d, [2.5, 97.5])
    p = min(1.0, 2 * min((d <= 0).mean(), (d >= 0).mean()))
    return {"delta_mean_over_seeds": round(float(obs), 4), "ci95_low": round(float(lo), 4),
            "ci95_high": round(float(hi), 4), "bootstrap_sign_p": round(max(p, 1 / n), 5), "n_seeds": K}


def seed_consistency(A_list, B_list, metric, N):
    """Primary seed-level evidence: per-seed deltas (corpus-level, same seed paired), mean, SD,
    number of seeds where A is better, and a paired t-test over seeds (low power with 3 seeds)."""
    d = np.array([metric.score(a.sum(0), N) - metric.score(b.sum(0), N) for a, b in zip(A_list, B_list)])
    k = len(d)
    sd = float(d.std(ddof=1)) if k > 1 else np.nan
    try:
        p_t = float(sps.ttest_1samp(d, 0.0).pvalue) if k >= 2 and sd > 0 else np.nan
    except Exception:
        p_t = np.nan
    return {"n_seeds": k, "seed_deltas": [round(float(x), 3) for x in d], "mean_delta": round(float(d.mean()), 4),
            "sd_delta": round(sd, 4) if np.isfinite(sd) else np.nan, "n_seeds_pscab_better": int((d > 0).sum()),
            "p_paired_t_over_seeds": round(p_t, 4) if np.isfinite(p_t) else np.nan}


def did_bootstrap(A_list, B_list, metric, prep_idx, nop_idx, n=2000, seed=11):
    """
    H2 difference-in-differences with hierarchical bootstrap over
    BOTH seeds and sentences.

    DiD =
        [PS-CAB - baseline on preposition sentences]
        -
        [PS-CAB - baseline on no-preposition sentences]

    Bootstrap resamples:
      1. seeds with replacement
      2. sentences within each selected seed
    """
    rng = np.random.default_rng(seed)

    K = len(A_list)

    def did_for_indices(seed_ids, prep_samples, noprep_samples):
        prep_deltas = []
        noprep_deltas = []

        for k, ip, iq in zip(seed_ids, prep_samples, noprep_samples):
            dp = (
                metric.score(A_list[k][ip].sum(0), len(ip))
                - metric.score(B_list[k][ip].sum(0), len(ip))
            )

            dn = (
                metric.score(A_list[k][iq].sum(0), len(iq))
                - metric.score(B_list[k][iq].sum(0), len(iq))
            )

            prep_deltas.append(dp)
            noprep_deltas.append(dn)

        return float(np.mean(prep_deltas)), float(np.mean(noprep_deltas))

    # Observed effect
    observed_seed_ids = list(range(K))
    observed_prep = [prep_idx] * K
    observed_noprep = [nop_idx] * K

    dp, dn = did_for_indices(
        observed_seed_ids,
        observed_prep,
        observed_noprep
    )

    observed_did = dp - dn

    # Hierarchical bootstrap over seeds AND sentences
    boots = np.empty(n, dtype=np.float64)

    for i in range(n):

        # Resample seeds with replacement
        seed_ids = rng.integers(0, K, size=K)

        ip_shared = rng.choice(prep_idx, size=len(prep_idx), replace=True)
        iq_shared = rng.choice(nop_idx, size=len(nop_idx), replace=True)
        prep_samples = [ip_shared] * K
        noprep_samples = [iq_shared] * K

        bp, bn = did_for_indices(
            seed_ids,
            prep_samples,
            noprep_samples
        )

        boots[i] = bp - bn

    lo, hi = np.percentile(boots, [2.5, 97.5])

    p = min(
        1.0,
        2 * min(
            float((boots <= 0).mean()),
            float((boots >= 0).mean())
        )
    )

    return {
        "delta_prep": round(float(dp), 4),
        "delta_noprep": round(float(dn), 4),
        "DiD": round(float(observed_did), 4),
        "DiD_ci95_low": round(float(lo), 4),
        "DiD_ci95_high": round(float(hi), 4),
        "bootstrap_sign_p": round(max(p, 1 / n), 5),
        "n_seeds": K
    }

# ==============================================================================
# D. EXTERNAL BENCHMARKS: contamination-filtered (frozen in Part 1) AND official full sets
# ==============================================================================
def raw_external_sets():
    """Official, UNFILTERED external sets (normalised only). Needs HF access for FLORES+ / your CSVs."""
    sets = {}
    if CFG["external"]["try_flores_plus"]:
        f = prepare_flores_plus(None)
        if f is not None:
            sets["flores_plus_devtest"] = f
    for name, p in CFG["external"]["csvs"].items():
        if os.path.exists(p):
            sets[name] = read_csv_str(p)[["en", "te"]]
    out = {}
    for name, d in sets.items():
        d = d.copy()
        d["en"], d["te"] = d["en"].apply(normalize_text), d["te"].apply(normalize_text)
        out[name] = d.reset_index(drop=True)
    return out


def _load_saved_model(run_id):
    cond, _, _ = parse_run_id(run_id)
    d = f"{DIRS['models']}/{run_id}"
    model = PSCABMBart(cond)
    model.backbone = MBartForConditionalGeneration.from_pretrained(d).float()
    if model.use_pscab:
        sd = torch.load(f"{d}/pscab_module_fp16.pt", map_location="cpu")
        model.pscab.load_state_dict({k: v.float() for k, v in sd.items()})
    tok = MBart50TokenizerFast.from_pretrained(d)
    tok.src_lang, tok.tgt_lang = CFG["model"]["src_lang"], CFG["model"]["tgt_lang"]
    return model.to(DEVICE).eval(), tok


def external_full_report(comet_ok):
    """Scores the OFFICIAL full external sets with the saved models (full sets were not generated in
    Part 1, which only decoded the contamination-filtered copies)."""
    print(f"\n{'#'*72}\nEXTERNAL BENCHMARKS -- OFFICIAL FULL SETS (no filtering)\n{'#'*72}")
    if not RUN_FULL_EXTERNAL:
        print("[external-full] disabled.")
        return None
    raw = raw_external_sets()
    if not raw:
        print("[external-full] no official external set available (FLORES+ needs Internet + HF_TOKEN with access; "
              "or add CSVs to CFG['external']['csvs']). Nothing scored.")
        return None
    ids = [r for r in FULL_EXTERNAL_FOR if os.path.isdir(f"{DIRS['models']}/{r}")]
    if not ids:
        print("[external-full] saved model bundles not found; nothing scored.")
        return None
    for rid in ids:
        cond, _, _ = parse_run_id(rid)
        todo = [n for n in raw if load_preds(rid, f"extfull_{n}") is None]
        if not todo:
            continue
        model, tok = _load_saved_model(rid)
        for n in todo:
            p, _ = generate(model, tok, raw[n], CONDITIONS[cond]["mask"])
            save_preds(rid, f"extfull_{n}", raw[n], p)
            print(f"[external-full] {rid} on {n}: {len(raw[n])} sentences decoded")
        del model
        free_gpu()
    rows = []
    for n, d in raw.items():
        er = d["te"].tolist()
        for rid in ids:
            p = load_preds(rid, f"extfull_{n}")
            if p is None:
                continue
            m = compute_metrics(p, er)
            row = {"set": n, "version": "official-full", "n": len(er), "run": rid,
                   "BLEU": m["bleu"], "chrF++": m["chrf++"], "TER": m["ter"]}
            if comet_ok:
                cs = comet_segments(rid, f"extfull_{n}", d)
                row["COMET"] = round(float(np.mean(cs)) * 100, 2) if cs is not None else None
            rows.append(row)
        if all(load_preds(r, f"extfull_{n}") for r in ("baseline_s42", "pscab_s42")):
            M = StatMetric("chrf++")
            res = paired_test(M.stats(load_preds("pscab_s42", f"extfull_{n}"), er),
                              M.stats(load_preds("baseline_s42", f"extfull_{n}"), er), M)
            print(f"[external-full] {n}: chrF++ pscab_s42 - baseline_s42 = {res}")
    if rows:
        t = pd.DataFrame(rows)
        print(t.to_string(index=False))
        save_table(t, "F13b_external_official_full", "External benchmarks, official full sets (no filtering)")
        return t
    return None


# ==============================================================================
# E. FINAL TEST EVALUATION
# ==============================================================================
def final_test_evaluation():
    print(f"\n{'#'*72}\nFINAL HELD-OUT TEST EVALUATION\n{'#'*72}")
    q = build_main_queue()
    missing = [r for r in q if not run_done(r)]
    if missing and not FORCE_FINAL:
        print(f"[final] main queue incomplete ({len(missing)} missing: {missing[:4]}...). "
              f"Finish Part 2 first. Do NOT set FORCE_FINAL=True for the paper: looking at test results "
              f"while experimental decisions are still open invalidates the held-out test.")
        return
    prelim = bool(missing)
    if prelim:
        print("!" * 72 + f"\n[final] PRELIMINARY MODE: {len(missing)} main runs missing. These numbers are NOT for the paper.\n" + "!" * 72)
    train, val, test = load_splits()
    refs = test["te"].tolist()
    N = len(refs)
    recs = {r["run_id"]: r for r in all_records() if r["kind"] in ("main", "zeroshot")}
    runs = [rid for rid in recs if load_preds(rid, "test") is not None]
    if not runs:
        print("[final] no test predictions found.")
        return
    vers = {str(rec_tf_version(recs[r])) for r in runs}
    if len(vers) > 1 or vers != {ENV_PIN["transformers"]}:
        print(f"[final] transformers versions among the runs: {vers} (pinned {ENV_PIN['transformers']}).")
        if not FORCE_FINAL:
            print("[final] refusing to compare runs decoded under different environments. Run Part 1 (it supersedes "
                  "and re-queues them) and finish the re-runs first.")
            return
    preds = {rid: load_preds(rid, "test") for rid in runs}
    seeds = CFG["experiment"]["seeds"]
    f7 = f8 = f11 = pd.DataFrame()

    # ---- F1 dataset table
    ds = jload(f"{DIRS['data']}/data_summary.json")
    f1 = pd.DataFrame([{"original": ds["cleaning"]["original_rows"], "after_cleaning": ds["cleaning"]["rows_after_cleaning"],
                        "after_exact_dedup": ds["cleaning"]["rows_after_exact_dedup"], "selected": ds["selected_pairs"],
                        **{f"n_{k}": v for k, v in ds["split_sizes"].items()}, "near_dup_groups": ds["n_groups"],
                        "leakage_audit": ds["leakage_audit"]["status"]}])
    print("\n[F1] Data\n" + f1.to_string(index=False))
    save_table(f1, "F1_dataset")

    # ---- F2 per-run test metrics (+COMET)
    comet_ok = USE_COMET and (get_comet() is not False)
    seg = {}
    rows = []
    for rid in runs:
        m = compute_metrics(preds[rid], refs)
        row = {"run": rid, "condition": recs[rid]["condition"], "seed": recs[rid].get("seed"),
               "BLEU": m["bleu"], "chrF++": m["chrf++"], "TER": m["ter"]}
        if comet_ok:
            seg[rid] = comet_segments(rid, "test", test)
            row["COMET"] = round(float(np.mean(seg[rid])) * 100, 2) if seg[rid] is not None else None
        rows.append(row)
    f2 = pd.DataFrame(rows)
    f2["seed"] = f2["seed"].astype("Int64")
    print("\n[F2] Test results per run\n" + f2.to_string(index=False))
    sig = compute_metrics(preds[runs[0]], refs)
    print(f"BLEU signature: {sig['bleu_signature']}\nchrF signature: {sig['chrf_signature']}")
    save_table(f2, "F2_test_per_run")

    # ---- F3 condition mean +- SD, 95% t-CI over seeds (SD / CI = N/A for a single seed)
    metrics_cols = [c for c in ("BLEU", "chrF++", "TER", "COMET") if c in f2.columns]
    rows = []
    for cond in ["zeroshot_mbart50"] + list(CONDITIONS):
        sub = f2[f2["condition"] == cond]
        if not len(sub):
            continue
        row = {"condition": cond, "n_seeds": len(sub)}
        for c in metrics_cols:
            v = sub[c].dropna().astype(float).values
            if not len(v):
                continue
            sd = v.std(ddof=1) if len(v) > 1 else np.nan
            half = sps.t.ppf(0.975, len(v) - 1) * sd / math.sqrt(len(v)) if len(v) > 1 else np.nan
            row[f"{c}_mean"] = round(v.mean(), 2)
            row[f"{c}_sd"] = round(sd, 2) if np.isfinite(sd) else np.nan
            row[f"{c}_ci95"] = round(half, 2) if np.isfinite(half) else np.nan
        rows.append(row)
    f3 = pd.DataFrame(rows)
    print("\n[F3] Test, mean ± SD (95% t-CI half-width) over seeds; N/A = single seed\n" + na_df(f3).to_string(index=False))
    save_table(na_df(f3), "F3_test_by_condition", "Test results, mean ± SD over seeds (N/A = single seed)")

    # ---- F4 per-seed paired tests (AR p + bootstrap CI), Holm thresholds + Holm-ADJUSTED p per metric
    mets = ["bleu", "chrf++"] + (["comet"] if comet_ok and all(seg.get(r) is not None for r in runs) else [])
    comps = []
    for s in seeds:
        if f"pscab_s{s}" in preds and f"baseline_s{s}" in preds:
            comps.append((f"pscab_s{s}", f"baseline_s{s}"))
        for a in ABLATIONS:
            if f"pscab_s{s}" in preds and f"{a}_s{s}" in preds:
                comps.append((f"pscab_s{s}", f"{a}_s{s}"))
    rows, stat_cache = [], {}
    for mn in mets:
        M = StatMetric(mn)
        stat_cache[mn] = {rid: (M.stats(preds[rid], refs, seg.get(rid)) if mn != "comet" else M.stats(None, None, seg[rid]))
                          for rid in runs}
        pv = {}
        for a, b in comps:
            res = paired_test(stat_cache[mn][a], stat_cache[mn][b], M)
            rows.append({"metric": mn, "A": a, "B": b, **res})
            pv[f"{mn}|{a}|{b}"] = res["p_ar"]
        h = holm(pv)
        adj = holm_adjust(pv) if pv else {}
        for r in rows:
            k = f"{r['metric']}|{r['A']}|{r['B']}"
            if k in h:
                r.update(h[k])
                r["p_holm_adjusted"] = round(adj[k], 5)
    f4 = pd.DataFrame(rows)
    print("\n[F4] Paired tests on TEST (A - B, same seed): AR p (10k), bootstrap 95% CI (2k), Holm per metric\n" + f4.to_string(index=False))
    save_table(f4, "F4_test_paired_tests")

    # ---- F5 seed-level evidence: (a) per-seed consistency [primary], (b) hierarchical bootstrap [characterisation]
    rows5b, rows5 = [], []
    for mn in mets:
        M = StatMetric(mn)
        for other in ["baseline"] + ABLATIONS:
            ss_ = [s for s in seeds if f"pscab_s{s}" in preds and f"{other}_s{s}" in preds]
            if not ss_:
                continue
            A = [stat_cache[mn][f"pscab_s{s}"] for s in ss_]
            B = [stat_cache[mn][f"{other}_s{s}"] for s in ss_]
            rows5b.append({"metric": mn, "comparison": f"pscab - {other}", **seed_consistency(A, B, M, N)})
            rows5.append({"metric": mn, "comparison": f"pscab - {other}", **hier_bootstrap(A, B, M, N)})
    f5b, f5 = pd.DataFrame(rows5b), pd.DataFrame(rows5)
    print("\n[F5a] PRIMARY seed-level evidence: per-seed deltas (same-seed pairing), mean, SD, #seeds better, "
          "paired t over seeds (3 seeds = very low power)\n" + na_df(f5b).to_string(index=False))
    save_table(na_df(f5b.assign(seed_deltas=f5b["seed_deltas"].astype(str))) if len(f5b) else f5b,
               "F5a_test_seed_consistency", "PS-CAB effect per seed")
    print("\n[F5b] Seed-aware uncertainty characterisation (hierarchical bootstrap over seeds and sentences). "
          "With only K seeds this is NOT K independent replications x N sentences; do not present it as "
          "population-level proof.\n" + f5.to_string(index=False))
    save_table(f5, "F5b_test_hierarchical_bootstrap", "Seed-aware uncertainty characterisation (hierarchical bootstrap)")

    # ---- F6 H2: preposition subsets + difference-in-differences
    ensure_spans(test["en"].tolist())
    counts = np.array([len(_SPAN_CACHE[s]) for s in test["en"]])
    prep_idx, nop_idx = np.where(counts > 0)[0], np.where(counts == 0)[0]
    rows = []
    ss = [s for s in seeds if f"pscab_s{s}" in preds and f"baseline_s{s}" in preds]
    if ss and len(nop_idx) >= 30:
        for mn in mets:
            M = StatMetric(mn)
            rows.append({"metric": mn, "n_prep": len(prep_idx), "n_noprep": len(nop_idx),
                         **did_bootstrap([stat_cache[mn][f"pscab_s{s}"] for s in ss],
                                         [stat_cache[mn][f"baseline_s{s}"] for s in ss], M, prep_idx, nop_idx)})
    f6 = pd.DataFrame(rows)
    if len(f6):
        print("\n[F6] H2 difference-in-differences (PS-CAB - baseline on prep sentences minus on no-prep sentences)\n"
              + f6.to_string(index=False))
        print("[F6 note] On no-prep sentences the true-mask module is an exact no-op, so delta_noprep estimates "
              "pure training variance; H2 is supported only if the DiD CI excludes 0 and is positive. "
              f"Only {len(nop_idx)} no-prep sentences exist, so H2 has limited power.")
        save_table(f6, "F6_H2_difference_in_differences")

    # ---- F7 preposition density / type, F8 length buckets (mean over seeds of PS-CAB - baseline)
    def subset_delta(mask, mn="chrf++"):
        M = StatMetric(mn)
        v = [M.score(stat_cache[mn][f"pscab_s{s}"][mask].sum(0), mask.sum())
             - M.score(stat_cache[mn][f"baseline_s{s}"][mask].sum(0), mask.sum()) for s in ss]
        return round(float(np.mean(v)), 3), (round(float(np.std(v, ddof=1)), 3) if len(v) > 1 else np.nan)

    rows = []
    if ss:
        bucket = np.array(["0" if c == 0 else "1" if c == 1 else "2" if c == 2 else "3+" for c in counts])
        for b in ["0", "1", "2", "3+"]:
            m = bucket == b
            if m.sum() >= 30:
                d, sd = subset_delta(m)
                rows.append({"group": f"#prep={b}", "n": int(m.sum()), "chrF++_delta_mean": d, "sd_over_seeds": sd})
        words = [[w for _, _, w in _SPAN_CACHE[s]] for s in test["en"]]
        for pw in ["of", "in", "to", "for", "on", "with", "from", "by", "at", "into", "about", "after", "over", "during"]:
            m = np.array([pw in w for w in words])
            if m.sum() >= 30:
                d, sd = subset_delta(m)
                rows.append({"group": f"prep='{pw}'", "n": int(m.sum()), "chrF++_delta_mean": d, "sd_over_seeds": sd})
        f7 = pd.DataFrame(rows)
        print("\n[F7] PS-CAB - baseline by preposition density / type (test, chrF++, mean over seeds; N/A = single seed)\n"
              + na_df(f7).to_string(index=False))
        save_table(na_df(f7), "F7_preposition_density_type")
        lens = test["en"].apply(lambda s: len(s.split())).values
        rows = []
        for lo, hi, lab in [(1, 10, "1-10"), (11, 20, "11-20"), (21, 30, "21-30"), (31, 10**6, ">30")]:
            m = (lens >= lo) & (lens <= hi)
            if m.sum() >= 30:
                d, sd = subset_delta(m)
                row = {"length": lab, "n": int(m.sum()), "chrF++_delta_mean": d, "sd_over_seeds": sd}
                for cond in ("baseline", "pscab"):
                    M = StatMetric("chrf++")
                    row[f"{cond}_chrF++"] = round(np.mean([M.score(stat_cache["chrf++"][f"{cond}_s{s}"][m].sum(0), m.sum()) for s in ss]), 2)
                rows.append(row)
        f8 = pd.DataFrame(rows)
        print("\n[F8] By source length (test; N/A = single seed)\n" + na_df(f8).to_string(index=False))
        save_table(na_df(f8), "F8_length_buckets")

    # ---- F9 error heuristics (automatic flags, NOT a linguistic taxonomy)
    rows = []
    for rid in runs:
        p = preds[rid]
        ratio = np.array([len(h) / max(len(r), 1) for h, r in zip(p, refs)])
        rows.append({"run": rid,
                     "untranslated_latin_words_%": 100 * np.mean([bool(LATIN_WORD_RE.search(h)) for h in p]),
                     "repeated_trigram_%": 100 * np.mean([_has_repeat(h) for h in p]),
                     "empty_%": 100 * np.mean([h.strip() == "" for h in p]),
                     "len_ratio_<0.5_%": 100 * np.mean(ratio < 0.5), "len_ratio_>2_%": 100 * np.mean(ratio > 2)})
    f9 = pd.DataFrame(rows).round(3)
    print("\n[F9] Automatic error flags (test)\n" + f9.to_string(index=False))
    save_table(f9, "F9_error_flags")

    # ---- F10 cost, F11 diagnostics
    rows = []
    for rid, r in recs.items():
        if r["kind"] != "main":
            continue
        rows.append({"run": rid, "total_params": r.get("total_parameters"), "added_params": r.get("pscab_added_parameters"),
                     "steps": r.get("steps_trained"), "best_step": r.get("best_step"), "stop": r.get("stop_reason"),
                     "sec_per_step": round(r.get("sec_per_step", float("nan")), 3), "train_h": round(r.get("train_hours", 0), 2),
                     "infer_sent_per_s": round(r.get("inference_sentences_per_sec", float("nan")), 2)})
    f10 = pd.DataFrame(rows)
    print("\n[F10] Cost (identical step budget; differences in steps come from early stopping only)\n" + f10.to_string(index=False))
    save_table(f10, "F10_cost")
    rows = [{"run": rid, **r["diagnostics"]} for rid, r in recs.items() if r.get("diagnostics")]
    if rows:
        f11 = pd.DataFrame(rows)
        print("\n[F11] Module diagnostics (validation). Attention mass / gate statistics are mechanistic "
              "DIAGNOSTICS of source-token utilisation, not proof of mechanism.\n" + f11.to_string(index=False))
        save_table(f11, "F11_module_diagnostics")

    # ---- F13 external test sets: (a) contamination-FILTERED (frozen in Part 1), (b) OFFICIAL FULL
    ext = external_sets()
    rep = jload(f"{DIRS['data']}/external_report.json") or {}
    if rep:
        print("\n[F13 note] contamination filtering applied in Part 1 (n_raw -> n_used):\n" + json.dumps(rep, indent=2))
    rows = []
    for name, edf in ext.items():
        er = edf["te"].tolist()
        for rid in runs:
            p = load_preds(rid, f"ext_{name}")
            if p is None:
                continue
            m = compute_metrics(p, er)
            row = {"set": name, "version": "contamination-filtered", "n": len(er), "run": rid,
                   "BLEU": m["bleu"], "chrF++": m["chrf++"], "TER": m["ter"]}
            if comet_ok:
                cs = comet_segments(rid, f"ext_{name}", edf)
                row["COMET"] = round(float(np.mean(cs)) * 100, 2) if cs is not None else None
            rows.append(row)
        for s in seeds:
            a, b = load_preds(f"pscab_s{s}", f"ext_{name}"), load_preds(f"baseline_s{s}", f"ext_{name}")
            if a and b:
                M = StatMetric("chrf++")
                res = paired_test(M.stats(a, er), M.stats(b, er), M)
                print(f"[F13] {name} (filtered) seed {s}: chrF++ pscab-baseline {res}")
    if rows:
        f13 = pd.DataFrame(rows)
        print("\n[F13] External test sets -- CONTAMINATION-FILTERED (do not call these the official benchmark score)\n"
              + f13.to_string(index=False))
        save_table(f13, "F13_external_contamination_filtered")
    else:
        print("\n[F13] no filtered external predictions (no external set was frozen in Part 1).")
    external_full_report(comet_ok)

    # ---- Figures
    if len(f3) and "chrF++_mean" in f3:
        plt.figure(figsize=(8, 4))
        x = range(len(f3))
        plt.bar(x, f3["chrF++_mean"], yerr=f3["chrF++_sd"].fillna(0), capsize=4)
        plt.xticks(x, f3["condition"], rotation=25, ha="right")
        plt.ylim(max(0, f3["chrF++_mean"].min() - 5), f3["chrF++_mean"].max() + 2)
        plt.ylabel("test chrF++"); plt.title("Test chrF++ (mean; error bar = SD, none for a single seed)")
        save_fig("F_fig_test_chrf_by_condition")
    if len(f5):
        sub = f5[f5["metric"] == "chrf++"]
        if len(sub):
            plt.figure(figsize=(7, 3.6))
            y = np.arange(len(sub))
            plt.errorbar(sub["delta_mean_over_seeds"], y, xerr=[sub["delta_mean_over_seeds"] - sub["ci95_low"],
                                                                 sub["ci95_high"] - sub["delta_mean_over_seeds"]], fmt="o", capsize=4)
            plt.axvline(0, ls="--", c="gray"); plt.yticks(y, sub["comparison"])
            plt.xlabel("chrF++ difference (seed-aware 95% interval)"); plt.title("Effect sizes on test")
            save_fig("F_fig_forest_chrf")
    if len(f6):
        r = f6[f6["metric"] == "chrf++"].iloc[0]
        plt.figure(figsize=(5, 3.6))
        plt.bar(["with prepositions", "no prepositions\n(module inactive)"], [r["delta_prep"], r["delta_noprep"]])
        plt.axhline(0, c="gray", lw=.8); plt.ylabel("chrF++: PS-CAB - baseline")
        plt.title(f"H2: DiD = {r['DiD']} [{r['DiD_ci95_low']}, {r['DiD_ci95_high']}]")
        save_fig("F_fig_H2_did")

    export_blinded_human_eval(test, counts)
    final_evidence_report(f3, f4, f5, f5b, f6, prelim)
    if PAPER_FIGURES:
        ctx = dict(test=test, refs=refs, N=N, preds=preds, recs=recs, runs=runs, seeds=seeds, stat_cache=stat_cache,
                   seg=seg, mets=mets, comet_ok=comet_ok, counts=counts, f2=f2, f3=f3, f4=f4, f5=f5, f5b=f5b, f6=f6,
                   f7=f7, f8=f8, f9=f9, f10=f10, f11=f11, prelim=prelim)
        paper_figures(ctx)


def _has_repeat(s, n=3):
    t = s.split()
    grams = [tuple(t[i:i + n]) for i in range(len(t) - n + 1)]
    return len(grams) != len(set(grams))


# ==============================================================================
# F. HUMAN EVALUATION: blinded sheet export + agreement once annotated
# ==============================================================================
def export_blinded_human_eval(test, counts, n_prep=150, n_noprep=50):
    a_id, b_id = "baseline_s42", "pscab_s42"
    pa, pb = load_preds(a_id, "test"), load_preds(b_id, "test")
    if not pa or not pb:
        return
    rng = np.random.default_rng(2026)
    pi = np.where(counts > 0)[0]; ni = np.where(counts == 0)[0]
    idx = np.concatenate([rng.choice(pi, min(n_prep, len(pi)), replace=False),
                          rng.choice(ni, min(n_noprep, len(ni)), replace=False)])
    rng.shuffle(idx)
    sheet, key = [], []
    for k, i in enumerate(idx):
        flip = bool(rng.integers(0, 2))
        A, B = (pb[i], pa[i]) if flip else (pa[i], pb[i])
        sheet.append({"item": k, "source_en": test["en"].iloc[i], "reference_te": test["te"].iloc[i],
                      "translation_A": A, "translation_B": B, "adequacy_A_1to5": "", "adequacy_B_1to5": "",
                      "fluency_A_1to5": "", "fluency_B_1to5": "", "preposition_relation_correct_A_yes_no": "",
                      "preposition_relation_correct_B_yes_no": "", "preference_A_B_tie": ""})
        key.append({"item": k, "test_row": int(i), "A": b_id if flip else a_id, "B": a_id if flip else b_id,
                    "has_preposition": bool(counts[i] > 0)})
    pd.DataFrame(sheet).to_csv(f"{DIRS['analysis']}/HUMAN_EVAL_SHEET_blinded.csv", index=False)
    pd.DataFrame(key).to_csv(f"{DIRS['analysis']}/HUMAN_EVAL_KEY_do_not_share.csv", index=False)
    print(f"\n[human-eval] blinded sheet ({len(idx)} items, A/B randomised) + separate key written to {DIRS['analysis']}. "
          f"A blank sheet is NOT evidence: have >= 2 native Telugu annotators fill copies saved as "
          f"HUMAN_EVAL_ANNOTATOR1.csv / HUMAN_EVAL_ANNOTATOR2.csv in that folder, then re-run this cell.")


def human_eval_agreement():
    """Cohen's kappa (quadratic-weighted for 1-5 scores) between two annotators + unblinded preference counts."""
    a_dir = DIRS["analysis"]
    p1, p2 = f"{a_dir}/HUMAN_EVAL_ANNOTATOR1.csv", f"{a_dir}/HUMAN_EVAL_ANNOTATOR2.csv"
    kp = f"{a_dir}/HUMAN_EVAL_KEY_do_not_share.csv"
    if not (os.path.exists(p1) and os.path.exists(p2) and os.path.exists(kp)):
        print("\n[human-eval] annotator sheets not found -> no human-evaluation claim can be made yet.")
        return
    from sklearn.metrics import cohen_kappa_score
    a1, a2, key = read_csv_str(p1), read_csv_str(p2), read_csv_str(kp)
    ordinal = ["adequacy_A_1to5", "adequacy_B_1to5", "fluency_A_1to5", "fluency_B_1to5"]
    nominal = ["preposition_relation_correct_A_yes_no", "preposition_relation_correct_B_yes_no", "preference_A_B_tie"]
    rows = []
    for c in ordinal + nominal:
        x, y = a1[c].str.strip().str.lower(), a2[c].str.strip().str.lower()
        ok = (x != "") & (y != "")
        if ok.sum() < 10:
            continue
        kappa = (cohen_kappa_score(x[ok].astype(int), y[ok].astype(int), weights="quadratic")
                 if c in ordinal else cohen_kappa_score(x[ok], y[ok]))
        rows.append({"item_type": c, "n_rated_by_both": int(ok.sum()), "cohens_kappa": round(float(kappa), 3),
                     "weighting": "quadratic" if c in ordinal else "none"})
    if rows:
        t = pd.DataFrame(rows)
        print("\n[human-eval] inter-annotator agreement\n" + t.to_string(index=False))
        save_table(t, "F15_human_eval_agreement")
    for nm, a in (("annotator1", a1), ("annotator2", a2)):
        pref = a["preference_A_B_tie"].str.strip().str.lower()
        cnt = {"pscab_s42": 0, "baseline_s42": 0, "tie": 0}
        for k_, v in zip(key["item"].astype(int), key["A"]):
            pv = pref.iloc[k_] if k_ < len(pref) else ""
            if pv == "a":
                cnt[v] += 1
            elif pv == "b":
                cnt[key.loc[key["item"].astype(int) == k_, "B"].iloc[0]] += 1
            elif pv == "tie":
                cnt["tie"] += 1
        print(f"[human-eval] {nm} unblinded preference counts: {cnt}")


# ==============================================================================
# G. FINAL EVIDENCE REPORT
# ==============================================================================
def final_evidence_report(f3, f4, f5, f5b, f6, prelim=False):
    L = ["=" * 72,
         ("PRELIMINARY EVIDENCE SUMMARY -- NOT FOR THE PAPER (main queue incomplete)" if prelim else
          "FINAL EVIDENCE SUMMARY (generated from saved results; no claim is auto-asserted)"), "=" * 72]
    ds = jload(f"{DIRS['data']}/data_summary.json")
    L.append(f"Data: {ds['selected_pairs']} pairs, split {ds['split_sizes']}, leakage audit {ds['leakage_audit']['status']} "
             f"({ds['leakage_audit']['method']}).")
    L.append(f"Dataset-selection rationale: the cleaned BPCC corpus had {ds['cleaning']['rows_after_exact_dedup']} unique valid "
             f"pairs; a fixed {ds['selected_pairs']}-pair subset was drawn at random with seed {CFG['data']['select_seed']} before any "
             f"training, to keep baseline, ablations and seeds under one identical, T4-feasible regime. Selection used no "
             f"validation/test score, sentence difficulty or preposition frequency.")
    L.append(f"Protocol: identical max_steps={CFG['optim']['max_steps']} for all conditions, early stopping on validation NLL "
             f"(patience {CFG['optim']['patience_evals']} evals), seeds {CFG['experiment']['seeds']}, beam {CFG['decoding']['num_beams']}.")
    checks = {}
    if len(f5b):
        for mn in f5b["metric"].unique():
            r = f5b[(f5b["metric"] == mn) & (f5b["comparison"] == "pscab - baseline")]
            h = f5[(f5["metric"] == mn) & (f5["comparison"] == "pscab - baseline")] if len(f5) else f5
            if len(r) and len(h):
                r, h = r.iloc[0], h.iloc[0]
                excl = h["ci95_low"] > 0 or h["ci95_high"] < 0
                consistent = int(r["n_seeds_pscab_better"]) == int(r["n_seeds"])
                checks[f"main_{mn}"] = bool(h["ci95_low"] > 0 and consistent)
                L.append(f"PS-CAB - baseline [{mn}]: per-seed deltas {r['seed_deltas']}, mean {r['mean_delta']}, "
                         f"better in {r['n_seeds_pscab_better']}/{r['n_seeds']} seeds; seed-aware 95% interval "
                         f"{h['ci95_low']}..{h['ci95_high']} -> "
                         f"{'interval excludes 0' if excl else 'interval includes 0: no reliable difference'}"
                         f"{'' if consistent else '; NOT consistent across seeds'}")
        for abl in ABLATIONS:
            r = f5b[(f5b["metric"] == "chrf++") & (f5b["comparison"] == f"pscab - {abl}")]
            h = f5[(f5["metric"] == "chrf++") & (f5["comparison"] == f"pscab - {abl}")] if len(f5) else f5
            if len(r) and len(h):
                r, h = r.iloc[0], h.iloc[0]
                checks[f"beats_{abl}"] = bool(h["ci95_low"] > 0 and int(r["n_seeds_pscab_better"]) == int(r["n_seeds"]))
                L.append(f"PS-CAB - {abl} [chrF++]: per-seed {r['seed_deltas']}, mean {r['mean_delta']}, "
                         f"seed-aware interval {h['ci95_low']}..{h['ci95_high']}")
    if len(f6):
        r = f6[f6["metric"] == "chrf++"].iloc[0]
        checks["H2"] = bool(r["DiD_ci95_low"] > 0)
        L.append(f"H2 (DiD, chrF++): {r['DiD']} (CI {r['DiD_ci95_low']}..{r['DiD_ci95_high']}) -> "
                 f"{'supported' if checks['H2'] else 'not supported'}")
    L.append("")
    need = ["main_chrf++", "H2", "beats_pscab_random_mask", "beats_pscab_all_tokens"]
    L.append("Criteria for a preposition-specific claim: " + ", ".join(
        f"{k}={'PASS' if checks.get(k) else ('FAIL' if k in checks else 'MISSING')}" for k in need))
    if all(checks.get(k) for k in need):
        L.append("EVIDENCE PATTERN: consistent with a preposition-specific improvement (main effect > 0 and consistent "
                 "across seeds, H2 > 0, beats random-mask and all-token controls). Still confirm with COMET, CV and human evaluation.")
    elif checks.get("main_chrf++"):
        L.append("EVIDENCE PATTERN: an overall improvement exists but is NOT shown to be preposition-specific "
                 "(see FAIL/MISSING above) -> do not claim the preposition hypothesis; frame accordingly.")
    else:
        L.append("EVIDENCE PATTERN: no reliable improvement over the baseline. Report as a controlled negative result "
                 "(the design, controls and noise-floor analysis are the contribution), do not claim gains.")
    L += ["", "Paper checklist:",
          "  - report sacreBLEU signatures (printed in F2), COMET model name, seeds, steps, GPU",
          "  - describe PS-CAB as gated modulation of encoder states (cross-attention memory), not a logit bias",
          "  - gate statistics and attention mass are mechanistic diagnostics of source-token utilisation, not proof of mechanism",
          "  - report the no-preposition noise floor next to every subset result",
          "  - seeds: lead with per-seed results, mean ± SD (N/A for one seed), same-seed paired tests; call the hierarchical "
          "bootstrap a seed-aware uncertainty characterisation (only 3 seeds)",
          "  - CV: 'five-fold grouped cross-validation' (folds of one dataset, not independent experiments)",
          "  - external sets: report BOTH official full scores and contamination-filtered scores, with the number removed",
          "  - limitation: zero overlap between our splits does not imply mBART-50 never saw these sentences during pretraining",
          "  - human evaluation: only claim it after >= 2 annotators filled the sheets and kappa is reported"]
    text = "\n".join(L)
    fname = "PRELIMINARY_EVIDENCE_SUMMARY.txt" if prelim else "FINAL_EVIDENCE_SUMMARY.txt"
    with open(f"{R}/{fname}", "w", encoding="utf-8") as f:
        f.write(text)
    print("\n" + text)


# ==============================================================================
# G2. PAPER FIGURES (FIG01-FIG21), QUALITATIVE EXAMPLES, ASSET INDEX
#     Every figure is built only from saved predictions / records (no new decoding),
#     saved as PDF (vector, for LaTeX) + PNG (300 dpi), and listed in PAPER_ASSETS.md.
# ==============================================================================
FIG_CAPTIONS = {}


def _safe(fn):
    def wrap(*a, **k):
        try:
            return fn(*a, **k)
        except Exception as e:
            print(f"[paper-fig] {fn.__name__} skipped: {type(e).__name__}: {e}")
    wrap.__name__ = fn.__name__
    return wrap


def _cap(name, text):
    FIG_CAPTIONS[name] = text


def _runs_of(ctx, cond):
    return [f"{cond}_s{s}" for s in ctx["seeds"] if f"{cond}_s{s}" in ctx["preds"]]


def _conds(ctx):
    return [c for c in COND_ORDER if _runs_of(ctx, c)]


def _sent_chrf(ctx, rid):
    cache = ctx.setdefault("_sent", {})
    if rid not in cache:
        M = StatMetric("chrf++")
        cache[rid] = np.array([M.score(row, 1) for row in ctx["stat_cache"]["chrf++"][rid]])
    return cache[rid]


def _subset_score(ctx, rid, mask, mn="chrf++"):
    M = StatMetric(mn)
    st = ctx["stat_cache"][mn][rid][mask]
    return M.score(st.sum(0), int(mask.sum()))


def _cond_subset(ctx, cond, mask, mn="chrf++"):
    v = [_subset_score(ctx, r, mask, mn) for r in _runs_of(ctx, cond)]
    return (float(np.mean(v)), float(np.std(v, ddof=1)) if len(v) > 1 else np.nan) if v else (np.nan, np.nan)


def _cond_delta(ctx, cond, mask, mn="chrf++", ref="baseline"):
    v = [_subset_score(ctx, f"{cond}_s{s}", mask, mn) - _subset_score(ctx, f"{ref}_s{s}", mask, mn)
         for s in ctx["seeds"] if f"{cond}_s{s}" in ctx["preds"] and f"{ref}_s{s}" in ctx["preds"]]
    return (float(np.mean(v)), float(np.std(v, ddof=1)) if len(v) > 1 else np.nan, len(v)) if v else (np.nan, np.nan, 0)


@_safe
def fig01_architecture(ctx):
    from matplotlib.patches import FancyBboxPatch
    fig, ax = plt.subplots(figsize=(10, 3.4))
    ax.set_xlim(0, 10); ax.set_ylim(0, 3.4); ax.axis("off")

    def box(x, y, w, h, txt, fc):
        ax.add_patch(FancyBboxPatch((x, y), w, h, boxstyle="round,pad=0.04", fc=fc, ec="k", lw=1))
        ax.text(x + w / 2, y + h / 2, txt, ha="center", va="center", fontsize=8)

    def arrow(x0, y0, x1, y1):
        ax.annotate("", xy=(x1, y1), xytext=(x0, y0), arrowprops=dict(arrowstyle="->", lw=1.2))

    box(0.1, 1.4, 1.4, 0.7, "English source\n(en_XX)", "#eeeeee")
    box(1.9, 1.4, 1.7, 0.7, "mBART-50 encoder\n(12 layers) → H", "#cfe2f3")
    box(4.0, 1.15, 2.5, 1.2, "PS-CAB\n g = σ(W_g H)\n H' = H + γ · g ⊙ (W_b H) ⊙ m", "#f4cccc")
    box(4.0, 2.6, 2.5, 0.6, "preposition mask m\n(spaCy ADP → subword offsets)", "#fff2cc")
    box(6.9, 1.4, 1.7, 0.7, "mBART-50 decoder\ncross-attn K,V = H'", "#cfe2f3")
    box(8.9, 1.4, 1.0, 0.7, "Telugu\n(te_IN)", "#eeeeee")
    arrow(1.5, 1.75, 1.9, 1.75); arrow(3.6, 1.75, 4.0, 1.75); arrow(5.25, 2.6, 5.25, 2.35)
    arrow(6.5, 1.75, 6.9, 1.75); arrow(8.6, 1.75, 8.9, 1.75)
    ax.text(5.25, 0.55, "Controls: −gate (g≡1) · −γ (γ≡1, W_b zero-init) · random mask (same #tokens) · all tokens (m≡1)\n"
                        "All variants are the identity map at step 0; ~2.1M added parameters (0.34% of 611M).",
            ha="center", va="center", fontsize=7.5, style="italic")
    save_fig("FIG01_architecture")
    _cap("FIG01_architecture", "PS-CAB inserts a gated, preposition-masked residual on the encoder states that serve "
         "as keys/values of every decoder cross-attention layer. Controls used in the ablation are listed below.")


@_safe
def fig02_main_results(ctx):
    f2 = ctx["f2"]
    mets = [m for m in ("chrF++", "BLEU", "COMET", "TER") if m in f2.columns and f2[m].notna().any()]
    conds = _conds(ctx)
    fig, ax = plt.subplots(1, len(mets), figsize=(3.4 * len(mets), 3.6))
    ax = np.atleast_1d(ax)
    zs = f2[f2["condition"] == "zeroshot_mbart50"]
    for a, m in zip(ax, mets):
        allv = []
        for i, c in enumerate(conds):
            v = f2[f2["condition"] == c][m].dropna().astype(float).values
            allv += list(v)
            a.bar(i, v.mean(), color=COND_COLOR[c], alpha=.6, yerr=v.std(ddof=1) if len(v) > 1 else 0, capsize=3)
            a.scatter([i] * len(v), v, c="k", s=12, zorder=3)
        a.set_xticks(range(len(conds))); a.set_xticklabels([COND_LABEL[c] for c in conds], rotation=35, ha="right")
        lo, hi = min(allv), max(allv)
        pad = max(0.5, (hi - lo) * 0.6)
        a.set_ylim(lo - pad, hi + pad)
        zt = f" (zero-shot: {float(zs[m].iloc[0]):.1f})" if len(zs) and pd.notna(zs[m].iloc[0]) else ""
        a.set_title(f"{m}{' ↓' if m == 'TER' else ' ↑'}{zt}", fontsize=9)
    save_fig("FIG02_main_results")
    _cap("FIG02_main_results", "Held-out test results. Bars: mean over seeds; error bars: SD; dots: individual seeds. "
         "The y-axis is zoomed to the fine-tuned systems; the zero-shot mBART-50 score is given in each title.")


@_safe
def fig03_forest(ctx):
    f5 = ctx["f5"]
    if not len(f5):
        return
    mets = [m for m in ("chrf++", "bleu", "comet") if m in set(f5["metric"])]
    fig, ax = plt.subplots(1, len(mets), figsize=(3.8 * len(mets), 3.4), sharey=True)
    ax = np.atleast_1d(ax)
    for a, m in zip(ax, mets):
        sub = f5[f5["metric"] == m].reset_index(drop=True)
        y = np.arange(len(sub))[::-1]
        for yi, (_, r) in zip(y, sub.iterrows()):
            other = r["comparison"].split(" - ")[1]
            a.errorbar(r["delta_mean_over_seeds"], yi, xerr=[[r["delta_mean_over_seeds"] - r["ci95_low"]],
                                                              [r["ci95_high"] - r["delta_mean_over_seeds"]]],
                       fmt="o", capsize=3, color=COND_COLOR.get(other, "k"))
        a.axvline(0, ls="--", c="gray")
        a.set_yticks(y); a.set_yticklabels([f"PS-CAB − {COND_LABEL.get(c.split(' - ')[1], c)}" for c in sub["comparison"]])
        a.set_title({"chrf++": "chrF++", "bleu": "spBLEU", "comet": "COMET×100"}[m]); a.set_xlabel("difference")
    save_fig("FIG03_effect_forest")
    _cap("FIG03_effect_forest", "Effect of PS-CAB relative to the baseline and each control on the test set: mean "
         "difference over seeds with a seed-and-sentence (hierarchical) bootstrap 95% interval. Intervals crossing 0 "
         "indicate no reliable difference.")


@_safe
def fig04_seed_deltas(ctx):
    f5b = ctx["f5b"]
    if not len(f5b):
        return
    mets = [m for m in ("chrf++", "bleu", "comet") if m in set(f5b["metric"])]
    fig, ax = plt.subplots(1, len(mets), figsize=(3.8 * len(mets), 3.4))
    ax = np.atleast_1d(ax)
    mk = ["o", "s", "^", "D"]
    for a, m in zip(ax, mets):
        sub = f5b[f5b["metric"] == m].reset_index(drop=True)
        for i, (_, r) in enumerate(sub.iterrows()):
            d = list(r["seed_deltas"]) if not isinstance(r["seed_deltas"], str) else json.loads(r["seed_deltas"])
            for j, v in enumerate(d):
                a.scatter(i + (j - 1) * 0.12, v, marker=mk[j % 4], color=COND_COLOR.get(r["comparison"].split(" - ")[1], "k"),
                          s=26, label=f"seed {ctx['seeds'][j]}" if i == 0 else None)
            a.hlines(np.mean(d), i - .3, i + .3, color="k")
        a.axhline(0, ls="--", c="gray")
        a.set_xticks(range(len(sub)))
        a.set_xticklabels([COND_LABEL.get(c.split(" - ")[1], c) for c in sub["comparison"]], rotation=30, ha="right")
        a.set_ylabel(f"PS-CAB minus condition"); a.set_title({"chrf++": "chrF++", "bleu": "spBLEU", "comet": "COMET×100"}[m])
    ax[0].legend(fontsize=7)
    save_fig("FIG04_per_seed_deltas")
    _cap("FIG04_per_seed_deltas", "Same-seed paired differences on the test set (markers = seeds, line = mean). "
         "Consistency of sign across seeds is the primary seed-level evidence.")


@_safe
def fig05_h2(ctx):
    counts = ctx["counts"]
    prep, nop = counts > 0, counts == 0
    if nop.sum() < 30:
        return
    fig, ax = plt.subplots(1, 2, figsize=(10, 3.6))
    rng = np.random.default_rng(3)
    M = StatMetric("chrf++")
    rows = []
    for j, s in enumerate(ctx["seeds"]):
        a_, b_ = f"pscab_s{s}", f"baseline_s{s}"
        if a_ not in ctx["preds"] or b_ not in ctx["preds"]:
            continue
        for k, (nm, mask) in enumerate((("with prepositions", prep), ("no prepositions", nop))):
            idx = np.where(mask)[0]
            A, B = ctx["stat_cache"]["chrf++"][a_], ctx["stat_cache"]["chrf++"][b_]
            obs = M.score(A[idx].sum(0), len(idx)) - M.score(B[idx].sum(0), len(idx))
            bt = []
            for _ in range(400):
                ii = rng.choice(idx, len(idx))
                bt.append(M.score(A[ii].sum(0), len(ii)) - M.score(B[ii].sum(0), len(ii)))
            lo, hi = np.percentile(bt, [2.5, 97.5])
            rows.append((s, nm, obs, lo, hi))
            ax[0].errorbar(k + (j - 1) * .15, obs, yerr=[[obs - lo], [hi - obs]], fmt="o", capsize=3,
                           color=["#1f77b4", "#ff7f0e", "#2ca02c"][j % 3], label=f"seed {s}" if k == 0 else None)
    ax[0].axhline(0, ls="--", c="gray")
    ax[0].set_xticks([0, 1]); ax[0].set_xticklabels([f"with prepositions\n(n={prep.sum()})",
                                                     f"no prepositions\n(n={nop.sum()}, module inactive)"])
    ax[0].set_ylabel("chrF++: PS-CAB − baseline"); ax[0].set_title("(a) Subset effects per seed (95% CI)"); ax[0].legend()
    f6 = ctx["f6"]
    if len(f6):
        sub = f6.reset_index(drop=True)
        y = np.arange(len(sub))
        ax[1].errorbar(sub["DiD"], y, xerr=[sub["DiD"] - sub["DiD_ci95_low"], sub["DiD_ci95_high"] - sub["DiD"]],
                       fmt="o", capsize=4, color="#d62728")
        ax[1].axvline(0, ls="--", c="gray"); ax[1].set_yticks(y); ax[1].set_yticklabels(sub["metric"])
        ax[1].set_xlabel("difference-in-differences"); ax[1].set_title("(b) H2: Δ(prep) − Δ(no-prep), seed-aware CI")
    save_fig("FIG05_H2_preposition_effect")
    _cap("FIG05_H2_preposition_effect", "Hypothesis H2. (a) PS-CAB − baseline on sentences with and without prepositions; "
         "on the latter the module is an exact no-op, so the gap measures training variance (noise floor). "
         "(b) Difference-in-differences with a seed-and-sentence bootstrap 95% interval.")


@_safe
def fig06_prep_type_heatmap(ctx):
    test = ctx["test"]
    words = [[w for _, _, w in _SPAN_CACHE[s]] for s in test["en"]]
    types = ["of", "in", "to", "for", "on", "with", "from", "by", "at", "as", "into", "about", "after", "over",
             "during", "between", "through", "under", "against", "without", "before", "within"]
    types = [t for t in types if sum(t in w for w in words) >= 30]
    conds = [c for c in _conds(ctx) if c != "baseline"]
    if not types or not conds:
        return
    Mx = np.full((len(types), len(conds)), np.nan)
    for i, t in enumerate(types):
        mask = np.array([t in w for w in words])
        for j, c in enumerate(conds):
            Mx[i, j] = _cond_delta(ctx, c, mask)[0]
    v = np.nanmax(np.abs(Mx)) or 1
    plt.figure(figsize=(1.3 * len(conds) + 2.5, 0.32 * len(types) + 1.4))
    plt.imshow(Mx, cmap="RdBu", vmin=-v, vmax=v, aspect="auto")
    plt.colorbar(label="Δ chrF++ vs baseline")
    for i in range(len(types)):
        for j in range(len(conds)):
            plt.text(j, i, f"{Mx[i, j]:+.2f}", ha="center", va="center", fontsize=6.5)
    ns = [sum(t in w for w in words) for t in types]
    plt.yticks(range(len(types)), [f"{t} (n={n})" for t, n in zip(types, ns)])
    plt.xticks(range(len(conds)), [COND_LABEL[c] for c in conds], rotation=30, ha="right")
    plt.grid(False); plt.title("Per-preposition effect (test, mean over seeds)")
    save_fig("FIG06_preposition_type_heatmap")
    _cap("FIG06_preposition_type_heatmap", "chrF++ difference to the baseline on test sentences containing each "
         "preposition (n ≥ 30), averaged over seeds. Sentences can contain several prepositions, so rows overlap.")


@_safe
def fig07_density_length(ctx):
    counts = ctx["counts"]
    test = ctx["test"]
    lens = test["en"].str.split().str.len().values
    groups = [("#prep", [("0", counts == 0), ("1", counts == 1), ("2", counts == 2), ("3", counts == 3), ("4+", counts >= 4)]),
              ("source words", [("1-10", lens <= 10), ("11-20", (lens > 10) & (lens <= 20)),
                                ("21-30", (lens > 20) & (lens <= 30)), (">30", lens > 30)])]
    fig, ax = plt.subplots(2, 2, figsize=(10, 6.2))
    for col, (gname, buckets) in enumerate(groups):
        buckets = [(b, m) for b, m in buckets if m.sum() >= 30]
        x = np.arange(len(buckets))
        for c in _conds(ctx):
            ys = [_cond_subset(ctx, c, m)[0] for _, m in buckets]
            ax[0, col].plot(x, ys, marker="o", color=COND_COLOR[c], label=COND_LABEL[c])
            if c != "baseline":
                ds = [_cond_delta(ctx, c, m) for _, m in buckets]
                ax[1, col].errorbar(x, [d[0] for d in ds], yerr=[0 if np.isnan(d[1]) else d[1] for d in ds],
                                    marker="o", capsize=3, color=COND_COLOR[c], label=COND_LABEL[c])
        for r in (0, 1):
            ax[r, col].set_xticks(x); ax[r, col].set_xticklabels([f"{b}\n(n={m.sum()})" for b, m in buckets], fontsize=7)
        ax[1, col].axhline(0, ls="--", c="gray")
        ax[0, col].set_ylabel("chrF++"); ax[1, col].set_ylabel("Δ chrF++ vs baseline (± SD seeds)")
        ax[1, col].set_xlabel(gname)
    ax[0, 0].set_title("(a) by preposition density"); ax[0, 1].set_title("(b) by source length")
    ax[0, 0].legend(fontsize=6.5)
    save_fig("FIG07_density_and_length")
    _cap("FIG07_density_and_length", "Test chrF++ by number of prepositions in the source (left) and by source "
         "length (right). Top: absolute scores (mean over seeds); bottom: difference to the baseline (± SD over seeds).")


@_safe
def fig08_sentence_level(ctx):
    b_runs, p_runs = _runs_of(ctx, "baseline"), _runs_of(ctx, "pscab")
    pairs = [(f"pscab_s{s}", f"baseline_s{s}") for s in ctx["seeds"]
             if f"pscab_s{s}" in ctx["preds"] and f"baseline_s{s}" in ctx["preds"]]
    if not pairs:
        return
    d = np.mean([_sent_chrf(ctx, a) - _sent_chrf(ctx, b) for a, b in pairs], axis=0)
    base = np.mean([_sent_chrf(ctx, b) for _, b in pairs], axis=0)
    ps = np.mean([_sent_chrf(ctx, a) for a, _ in pairs], axis=0)
    fig, ax = plt.subplots(1, 3, figsize=(13, 3.7))
    ax[0].hist(np.clip(d, -30, 30), bins=np.arange(-30, 31, 1), color="#d62728", alpha=.8)
    w, l, t = (d > .5).mean(), (d < -.5).mean(), (np.abs(d) <= .5).mean()
    ax[0].set_title(f"(a) per-sentence Δ chrF++ (PS-CAB − baseline)\nwin {w:.1%} · tie {t:.1%} · loss {l:.1%}")
    ax[0].set_xlabel("Δ chrF++ (mean over seeds, clipped ±30)"); ax[0].set_ylabel("sentences")
    hb = ax[1].hexbin(base, ps, gridsize=40, mincnt=1, bins="log", cmap="viridis")
    ax[1].plot([0, 100], [0, 100], "w--", lw=.8)
    ax[1].set_xlabel("baseline sentence chrF++"); ax[1].set_ylabel("PS-CAB sentence chrF++")
    ax[1].set_title("(b) sentence-level agreement"); fig.colorbar(hb, ax=ax[1], label="log10 count")
    conds = [c for c in _conds(ctx) if c != "baseline"]
    W, T, L = [], [], []
    for c in conds:
        pr = [(f"{c}_s{s}", f"baseline_s{s}") for s in ctx["seeds"] if f"{c}_s{s}" in ctx["preds"] and f"baseline_s{s}" in ctx["preds"]]
        dd = np.mean([_sent_chrf(ctx, a) - _sent_chrf(ctx, b) for a, b in pr], axis=0)
        W.append((dd > .5).mean() * 100); T.append((np.abs(dd) <= .5).mean() * 100); L.append((dd < -.5).mean() * 100)
    x = np.arange(len(conds))
    ax[2].bar(x, W, color="#2ca02c", label="win"); ax[2].bar(x, T, bottom=W, color="#cccccc", label="tie (|Δ|≤0.5)")
    ax[2].bar(x, L, bottom=np.array(W) + np.array(T), color="#d62728", label="loss")
    ax[2].set_xticks(x); ax[2].set_xticklabels([COND_LABEL[c] for c in conds], rotation=30, ha="right")
    ax[2].set_ylabel("% of test sentences"); ax[2].set_title("(c) vs baseline"); ax[2].legend(fontsize=6.5)
    save_fig("FIG08_sentence_level")
    _cap("FIG08_sentence_level", "Sentence-level analysis on the test set. (a) Distribution of per-sentence chrF++ "
         "differences, PS-CAB − baseline (averaged over seeds). (b) Sentence-level scores of the two systems. "
         "(c) Win/tie/loss rates of every condition against the baseline.")


@_safe
def fig09_output_length(ctx):
    refs = ctx["refs"]
    B = bleu_metric()
    conds = ["zeroshot_mbart50"] + _conds(ctx)
    fig, ax = plt.subplots(1, 2, figsize=(11, 3.6))
    data, ratios = [], []
    for c in conds:
        rids = [r for r in ctx["runs"] if ctx["recs"][r]["condition"] == c]
        rr = []
        for rid in rids:
            sc = B.corpus_score(ctx["preds"][rid], [refs])
            rr.append(sc.sys_len / sc.ref_len)
        ratios.append(rr)
        data.append(np.mean([[len(h) / max(len(r), 1) for h, r in zip(ctx["preds"][rid], refs)] for rid in rids], axis=0))
    ax[0].boxplot([np.clip(x, 0, 3) for x in data], showfliers=False)
    ax[0].axhline(1, ls="--", c="gray")
    ax[0].set_xticks(range(1, len(conds) + 1)); ax[0].set_xticklabels([COND_LABEL[c] for c in conds], rotation=30, ha="right")
    ax[0].set_ylabel("hyp/ref characters"); ax[0].set_title("(a) per-sentence length ratio")
    for i, (c, rr) in enumerate(zip(conds, ratios)):
        ax[1].bar(i, np.mean(rr), color=COND_COLOR[c], alpha=.6)
        ax[1].scatter([i] * len(rr), rr, c="k", s=12, zorder=3)
    ax[1].axhline(1, ls="--", c="gray")
    ax[1].set_xticks(range(len(conds))); ax[1].set_xticklabels([COND_LABEL[c] for c in conds], rotation=30, ha="right")
    allr = [v for rr in ratios for v in rr]
    ax[1].set_ylim(min(allr + [1]) - .03, max(allr + [1]) + .03)
    ax[1].set_ylabel("corpus hyp/ref (spBLEU tokens)"); ax[1].set_title("(b) corpus length ratio (dots = seeds)")
    save_fig("FIG09_output_length")
    _cap("FIG09_output_length", "Output-length behaviour on the test set. Systematic length differences inflate or deflate "
         "BLEU through the brevity penalty without a change in chrF++; this figure documents that all compared systems "
         "decode with the same length behaviour (single pinned environment).")


@_safe
def fig10_metric_agreement(ctx):
    f2 = ctx["f2"]
    f = f2[f2["condition"] != "zeroshot_mbart50"]
    pairs = [("chrF++", "BLEU")] + ([("chrF++", "COMET")] if "COMET" in f and f["COMET"].notna().all() else []) + [("chrF++", "TER")]
    fig, ax = plt.subplots(1, len(pairs), figsize=(4 * len(pairs), 3.5))
    ax = np.atleast_1d(ax)
    for a, (x, y) in zip(ax, pairs):
        for c in _conds(ctx):
            sub = f[f["condition"] == c]
            a.scatter(sub[x], sub[y], color=COND_COLOR[c], label=COND_LABEL[c], s=30)
        tau = sps.kendalltau(f[x].astype(float), f[y].astype(float))
        a.set_xlabel(x); a.set_ylabel(y); a.set_title(f"Kendall τ = {tau.statistic:.2f} (p={tau.pvalue:.2f}, n={len(f)})")
    ax[0].legend(fontsize=6.5)
    save_fig("FIG10_metric_agreement")
    _cap("FIG10_metric_agreement", "Agreement between automatic metrics across all fine-tuned runs (test set). "
         "Low rank correlation between metrics means differences between systems are within metric noise.")


@_safe
def fig11_gates(ctx):
    s0 = ctx["seeds"][0]
    items = []
    for c in COND_ORDER:
        f = f"{DIRS['analysis']}/{c}_s{s0}_gate_samples.npz"
        if os.path.exists(f):
            z = np.load(f)
            if z["trained_prep"].size:
                items.append((c, z))
    if not items:
        return
    fig, ax = plt.subplots(1, len(items), figsize=(3.4 * len(items), 3.2), sharey=True)
    ax = np.atleast_1d(ax)
    bins = np.linspace(0, 1, 50)
    recs = ctx["recs"]
    for a, (c, z) in zip(ax, items):
        a.hist(z["trained_other"], bins, density=True, alpha=.45, color="#1f77b4", label="trained · other")
        a.hist(z["trained_prep"], bins, density=True, alpha=.45, color="#d62728", label="trained · preposition")
        if z["init_prep"].size:
            a.hist(z["init_other"], bins, density=True, histtype="step", color="#1f77b4", ls="--", label="init · other")
            a.hist(z["init_prep"], bins, density=True, histtype="step", color="#d62728", ls="--", label="init · preposition")
        dg = (recs.get(f"{c}_s{s0}") or {}).get("diagnostics", {})
        a.set_title(f"{COND_LABEL[c]}\nAUC trained {dg.get('gate_trained_auc_prep_vs_other', float('nan')):.2f} "
                    f"/ init {dg.get('gate_init_auc_prep_vs_other', float('nan')):.2f}", fontsize=8)
        a.set_xlabel("mean gate value")
    ax[0].legend(fontsize=6)
    save_fig("FIG11_gate_distributions")
    _cap("FIG11_gate_distributions", "Gate values g = σ(W_g H) at preposition vs other content tokens (validation, "
         f"seed {s0}), trained vs the same gate at initialisation. AUC = 0.5 means the gate does not distinguish prepositions.")


@_safe
def fig12_module_magnitude(ctx):
    rows = [(r["condition"], r["seed"], r["diagnostics"]) for r in ctx["recs"].values() if r.get("diagnostics")]
    if not rows:
        return
    conds = [c for c in COND_ORDER if any(r[0] == c for r in rows)]
    fig, ax = plt.subplots(1, 2, figsize=(9.5, 3.4))
    for k, (key, title) in enumerate((("gamma", "(a) learned scale γ"),
                                      ("bias_to_hidden_norm_ratio_mean", "(b) ‖γ·g·W_bH‖ / ‖H‖ on masked tokens"))):
        for i, c in enumerate(conds):
            v = [r[2].get(key) for r in rows if r[0] == c and r[2].get(key) is not None]
            ax[k].bar(i, np.mean(v), color=COND_COLOR[c], alpha=.6)
            ax[k].scatter([i] * len(v), v, c="k", s=14, zorder=3)
        ax[k].axhline(0, c="gray", lw=.8)
        ax[k].set_xticks(range(len(conds))); ax[k].set_xticklabels([COND_LABEL[c] for c in conds], rotation=30, ha="right")
        ax[k].set_title(title)
    save_fig("FIG12_module_magnitude")
    _cap("FIG12_module_magnitude", "Magnitude of the PS-CAB intervention (validation; dots = seeds). A large relative "
         "perturbation of the encoder states combined with unchanged translation quality indicates that the decoder "
         "learns to discount the added signal.")


@_safe
def fig13_attention_robustness(ctx):
    recs = [r for r in ctx["recs"].values() if r.get("kind") == "main" and r.get("attention")]
    rob = [r for r in ctx["recs"].values() if r.get("kind") == "main" and r.get("robustness_val")]
    if not recs and not rob:
        return
    fig, ax = plt.subplots(1, 2, figsize=(11, 3.6))
    conds = [c for c in COND_ORDER if any(r["condition"] == c for r in recs)]
    for i, c in enumerate(conds):
        v = [r["attention"]["prep_mass_last_layer_over_uniform"] for r in recs if r["condition"] == c]
        ax[0].bar(i, np.mean(v), color=COND_COLOR[c], alpha=.7)
        ax[0].scatter([i] * len(v), v, c="k", s=12, zorder=3)
    ax[0].axhline(1, ls="--", c="k", lw=.8)
    ax[0].set_xticks(range(len(conds))); ax[0].set_xticklabels([COND_LABEL[c] for c in conds], rotation=30, ha="right")
    ax[0].set_ylabel("attention mass ÷ uniform share"); ax[0].set_title("(a) last-layer cross-attention on prepositions")
    perts = list(PERTURBATIONS)
    rc = [c for c in COND_ORDER if any(r["condition"] == c for r in rob)]
    w = 0.8 / max(len(rc), 1)
    for i, c in enumerate(rc):
        vals = []
        for p_ in perts:
            v = [q["delta_chrf++"] for r in rob if r["condition"] == c for q in r["robustness_val"] if q["perturbation"] == p_]
            vals.append(np.mean(v) if v else np.nan)
        ax[1].bar(np.arange(len(perts)) + i * w, vals, w, color=COND_COLOR[c], label=COND_LABEL[c])
    ax[1].axhline(0, c="gray", lw=.8)
    ax[1].set_xticks(np.arange(len(perts)) + w * (len(rc) - 1) / 2); ax[1].set_xticklabels(perts)
    ax[1].set_ylabel("Δ chrF++ vs unperturbed"); ax[1].set_title("(b) robustness to source perturbations"); ax[1].legend(fontsize=6.5)
    save_fig("FIG13_attention_and_robustness")
    _cap("FIG13_attention_and_robustness", "(a) Teacher-forced cross-attention mass on preposition tokens in the last "
         "decoder layer, divided by their uniform share (validation). (b) chrF++ change under source perturbations "
         "(validation subset).")


@_safe
def fig14_cv(ctx):
    k, seed = CFG["cv"]["k"], CFG["cv"]["seed"]
    rows = []
    for f in range(k):
        for m in CFG["cv"]["models"]:
            r = jload(run_record_path(f"cv{f}_{m}_s{seed}"))
            if r and r.get("status") == "complete":
                rows.append({"fold": f, "model": m, **{x: r["heldout_metrics"][y] for x, y in (("chrF++", "chrf++"), ("BLEU", "bleu"))}})
    df = pd.DataFrame(rows)
    if not len(df) or df["fold"].nunique() < 2:
        return
    folds = sorted(df.groupby("fold").filter(lambda g: len(g) == 2)["fold"].unique())
    fig, ax = plt.subplots(1, 3, figsize=(12, 3.5))
    for a, m in zip(ax[:2], ("chrF++", "BLEU")):
        for f in folds:
            b = df[(df.fold == f) & (df.model == "baseline")][m].values[0]
            p = df[(df.fold == f) & (df.model == "pscab")][m].values[0]
            a.plot([0, 1], [b, p], marker="o", color="#2ca02c" if p > b else "#d62728")
            a.annotate(f"f{f}", (1.03, p), fontsize=7)
        a.set_xticks([0, 1]); a.set_xticklabels(["Baseline", "PS-CAB"]); a.set_ylabel(f"held-out {m}")
    d = [df[(df.fold == f) & (df.model == "pscab")]["chrF++"].values[0] - df[(df.fold == f) & (df.model == "baseline")]["chrF++"].values[0]
         for f in folds]
    ax[2].bar([f"fold {f}" for f in folds], d, color=["#2ca02c" if x > 0 else "#d62728" for x in d])
    ax[2].axhline(0, c="gray"); ax[2].axhline(np.mean(d), ls="--", c="k", label=f"mean {np.mean(d):+.2f}")
    ax[2].set_ylabel("Δ chrF++ (PS-CAB − baseline)"); ax[2].legend()
    ax[0].set_title("(a) chrF++ per fold"); ax[1].set_title("(b) spBLEU per fold"); ax[2].set_title("(c) paired differences")
    save_fig("FIG14_cross_validation")
    _cap("FIG14_cross_validation", f"{len(folds)}-fold grouped cross-validation over the train+validation pool "
         "(folds of one dataset, one seed). Lines connect the two systems on the same held-out fold.")


@_safe
def fig15_errors_cost(ctx):
    f9, f10 = ctx["f9"], ctx["f10"]
    fig, ax = plt.subplots(1, 3, figsize=(13, 3.5))
    if len(f9):
        f9 = f9.copy()
        f9["condition"] = f9["run"].map(lambda r: ctx["recs"][r]["condition"])
        cols = [c for c in f9.columns if c.endswith("%")]
        g = f9.groupby("condition")[cols].mean()
        conds = [c for c in ["zeroshot_mbart50"] + COND_ORDER if c in g.index and c != "zeroshot_mbart50"]
        w = 0.8 / max(len(conds), 1)
        for i, c in enumerate(conds):
            ax[0].bar(np.arange(len(cols)) + i * w, g.loc[c, cols].values, w, color=COND_COLOR[c], label=COND_LABEL[c])
        ax[0].set_xticks(np.arange(len(cols)) + w * (len(conds) - 1) / 2)
        ax[0].set_xticklabels([c.replace("_%", "").replace("_", " ") for c in cols], rotation=25, ha="right", fontsize=7)
        ax[0].set_ylabel("% of test outputs"); ax[0].set_title("(a) automatic error flags"); ax[0].legend(fontsize=6)
    if len(f10):
        f10 = f10.copy()
        f10["condition"] = f10["run"].map(lambda r: ctx["recs"][r]["condition"])
        g = f10.groupby("condition")[["sec_per_step", "infer_sent_per_s", "added_params"]].mean()
        conds = [c for c in COND_ORDER if c in g.index]
        ax[1].bar(range(len(conds)), g.loc[conds, "sec_per_step"], color=[COND_COLOR[c] for c in conds])
        ax[1].set_title("(b) training time per optimizer step (s)")
        ax[2].bar(range(len(conds)), g.loc[conds, "infer_sent_per_s"], color=[COND_COLOR[c] for c in conds])
        ax[2].set_title("(c) inference speed (sentences/s, beam 4)")
        for a in ax[1:]:
            a.set_xticks(range(len(conds))); a.set_xticklabels([COND_LABEL[c] for c in conds], rotation=30, ha="right")
        for i, c in enumerate(conds):
            ax[1].annotate(f"+{g.loc[c, 'added_params'] / 1e6:.2f}M", (i, g.loc[c, "sec_per_step"]), ha="center",
                           va="bottom", fontsize=6.5)
    save_fig("FIG15_errors_and_cost")
    _cap("FIG15_errors_and_cost", "(a) Automatic error flags on the test set (heuristics, not a linguistic taxonomy). "
         "(b) Training cost per optimizer step with the number of added parameters. (c) Inference throughput.")


@_safe
def fig16_learning_curves(ctx):
    plt.figure(figsize=(7.5, 4))
    for c in _conds(ctx):
        cs = [x for x in (_curve(r) for r in _runs_of(ctx, c)) if x is not None]
        if not cs:
            continue
        L = min(len(x) for x in cs)
        Mx = np.stack([x["val_nll"].values[:L] for x in cs])
        st = cs[0]["step"].values[:L]
        plt.plot(st, Mx.mean(0), marker="o", ms=3, color=COND_COLOR[c], label=f"{COND_LABEL[c]} (n={len(cs)})")
        if len(cs) > 1:
            plt.fill_between(st, Mx.min(0), Mx.max(0), color=COND_COLOR[c], alpha=.15)
    plt.axvline(2000, ls=":", c="k", lw=.8)
    plt.xlabel("optimizer step (effective batch 64)"); plt.ylabel("validation NLL / token"); plt.legend(ncol=2)
    plt.title("Learning curves (identical step budget; band = range over seeds)")
    save_fig("FIG16_learning_curves")
    _cap("FIG16_learning_curves", "Validation NLL during training for every condition (mean over seeds; band = min–max "
         "over seeds). All runs share the same 3,000-step budget and checkpoint selection by validation NLL.")


@_safe
def fig17_external(ctx):
    tabs = [p for p in (f"{DIRS['tables']}/F13_external_contamination_filtered.csv",
                        f"{DIRS['tables']}/F13b_external_official_full.csv") if os.path.exists(p)]
    if not tabs:
        return
    df = pd.concat([pd.read_csv(p) for p in tabs], ignore_index=True)
    df["condition"] = df["run"].map(lambda r: (ctx["recs"].get(r) or {}).get("condition", parse_run_id(r)[0] if "_s" in r else r))
    sets = df[["set", "version"]].drop_duplicates().values.tolist()
    fig, ax = plt.subplots(1, len(sets), figsize=(4.5 * len(sets), 3.5))
    ax = np.atleast_1d(ax)
    for a, (st, ver) in zip(ax, sets):
        sub = df[(df["set"] == st) & (df["version"] == ver)]
        g = sub.groupby("condition")["chrF++"]
        conds = [c for c in ["zeroshot_mbart50"] + COND_ORDER if c in g.groups]
        a.bar(range(len(conds)), [g.mean()[c] for c in conds], color=[COND_COLOR[c] for c in conds],
              yerr=[g.std().fillna(0)[c] for c in conds], capsize=3)
        a.set_xticks(range(len(conds))); a.set_xticklabels([COND_LABEL[c] for c in conds], rotation=30, ha="right")
        a.set_title(f"{st} ({ver})", fontsize=8); a.set_ylabel("chrF++")
    save_fig("FIG17_external_benchmarks")
    _cap("FIG17_external_benchmarks", "Out-of-domain evaluation on external benchmarks, reported both as the official "
         "full set and after removing sentences that overlap with our corpus.")


@_safe
def qualitative_examples(ctx, n_each=3):
    s0 = ctx["seeds"][0]
    a_, b_ = f"pscab_s{s0}", f"baseline_s{s0}"
    if a_ not in ctx["preds"] or b_ not in ctx["preds"]:
        return
    test = ctx["test"]
    sa, sb = _sent_chrf(ctx, a_), _sent_chrf(ctx, b_)
    d = sa - sb
    has = ctx["counts"] > 0
    cand = np.where(has & (test["en"].str.split().str.len().values <= 25))[0]
    order = cand[np.argsort(d[cand])]
    rng = np.random.default_rng(11)
    pick = [("PS-CAB better", i) for i in order[::-1][:n_each]] + [("baseline better", i) for i in order[:n_each]] + \
           [("random", i) for i in rng.choice(cand, n_each, replace=False)]
    rows = []
    for kind, i in pick:
        preps = ", ".join(w for _, _, w in _SPAN_CACHE[test["en"].iloc[i]])
        rows.append({"type": kind, "prepositions": preps, "source_en": test["en"].iloc[i], "reference_te": test["te"].iloc[i],
                     "baseline": ctx["preds"][b_][i], "pscab": ctx["preds"][a_][i],
                     "chrF++_baseline": round(sb[i], 1), "chrF++_pscab": round(sa[i], 1)})
    t = pd.DataFrame(rows)
    t.to_csv(f"{DIRS['tables']}/T_qualitative_examples.csv", index=False)
    with open(f"{DIRS['tables']}/T_qualitative_examples.md", "w", encoding="utf-8") as f:
        f.write(t.to_markdown(index=False) if hasattr(t, "to_markdown") else t.to_string(index=False))
    try:
        with open(f"{DIRS['tables']}/T_qualitative_examples.tex", "w", encoding="utf-8") as f:
            f.write("% Requires a Telugu-capable font, e.g. \\usepackage{fontspec}\\newfontfamily\\telugufont{Noto Sans Telugu}\n")
            f.write(t.to_latex(index=False, escape=True, caption="Qualitative examples (test, seed %d)" % s0,
                               label="tab:qualitative"))
    except Exception:
        pass
    print(f"\n[qualitative] {len(t)} examples (largest gains, largest losses, random) -> tables/T_qualitative_examples.*")


def write_assets_index():
    figs = sorted(set(os.path.splitext(os.path.basename(p))[0] for p in glob.glob(f"{DIRS['figures']}/*.pdf")))
    tabs = sorted(set(os.path.splitext(os.path.basename(p))[0] for p in glob.glob(f"{DIRS['tables']}/*.csv")))
    extra = {
        "D_fig_lengths": "Sentence-length distributions of the frozen splits (English source, Telugu target).",
        "D_fig_prepositions": "(a) Number of prepositions per source sentence by split; (b) most frequent prepositions.",
        "D_fig_en_te_length_hexbin": "English vs Telugu word counts (train): Telugu is agglutinative.",
        "DIAG_fig_env_drift": "Appendix: output length and BLEU grouped by transformers version (motivates the pinned environment).",
        "V_fig_paired_deltas": "Validation: PS-CAB minus every other condition, per seed, with bootstrap 95% CI.",
    }
    L = ["# Paper assets (auto-generated)", "", "Figures are in `figures/` (PDF = vector for LaTeX, PNG = 300 dpi).",
         "Tables are in `tables/` (CSV + LaTeX).", "", "## Main-paper figures"]
    for f in figs:
        if f.startswith("FIG"):
            L.append(f"- **{f}** — {FIG_CAPTIONS.get(f, '')}")
    L += ["", "## Data / validation / appendix figures"]
    for f in figs:
        if not f.startswith("FIG"):
            L.append(f"- {f} — {extra.get(f, '')}")
    L += ["", "## Tables"] + [f"- {t}" for t in tabs]
    with open(f"{R}/PAPER_ASSETS.md", "w", encoding="utf-8") as fh:
        fh.write("\n".join(L))
    print(f"[assets] {len(figs)} figures, {len(tabs)} tables indexed -> {R}/PAPER_ASSETS.md")


def paper_figures(ctx):
    print(f"\n{'#'*72}\nPAPER FIGURES (test set; built from saved predictions)\n{'#'*72}")
    apply_paper_style()
    for fn in (fig01_architecture, fig02_main_results, fig03_forest, fig04_seed_deltas, fig05_h2, fig06_prep_type_heatmap,
               fig07_density_length, fig08_sentence_level, fig09_output_length, fig10_metric_agreement, fig11_gates,
               fig12_module_magnitude, fig13_attention_robustness, fig14_cv, fig15_errors_cost, fig16_learning_curves,
               fig17_external, qualitative_examples):
        fn(ctx)
    write_assets_index()


# ==============================================================================
# H. DRIVER
# ==============================================================================
build_cv_folds()
if DO_CV:
    run_next_cv_runs(CV_RUNS_THIS_SESSION)
cv_report()
CV_COMPLETE = all(run_done(r) for r in cv_queue())
if CV_COMPLETE:
    print("[cv] CROSS-VALIDATION COMPLETE.")
    final_test_evaluation()
    human_eval_agreement()
else:
    print(f"[cv] {sum(run_done(r) for r in cv_queue())}/{len(cv_queue())} CV runs complete; "
          f"re-run Part 1 + Part 3 next session. (Final evaluation waits for CV.)")
print(f"\nAll outputs in {R}/ (tables: CSV + LaTeX, figures: PDF + PNG). Session time left {session_hours_left():.1f}h.")


########################################################################
CROSS-VALIDATION REPORT (five-fold grouped CV; held-out folds; test split untouched)
########################################################################
no CV run completed yet.
[cv] 0/10 CV runs complete; re-run Part 1 + Part 3 next session. (Final evaluation waits for CV.)

All outputs in /kaggle/working/pscab_q1/ (tables: CSV + LaTeX, figures: PDF + PNG). Session time left 10.8h.


In [9]:
"""
================================================================================
PART 1 / 3 (v3) -- CORE LIBRARY + DATA + MAIN EXPERIMENT (baseline + PS-CAB, seed 42)
English -> Telugu NMT, facebook/mbart-large-50-many-to-many-mmt
================================================================================
v3 CHANGES (environment-drift fix + paper figures):

  E1  ENVIRONMENT PIN: transformers / sacrebleu are pinned at the very top of this
      cell (before transformers is imported). The run aborts if the pin cannot be met,
      so no two runs are ever decoded with different library versions again.
  E2  DRIFT DIAGNOSTIC (runs once, cached): output-length ratio / BP / BLEU / TER of
      every run grouped by library version, and (if the pscab_s42 .h5 exists) the
      old-environment pscab_s42 model re-decoded in the current environment.
      Figures + table go to the appendix (DIAG_*).
  E3  SUPERSEDE: completed runs produced under a different transformers version are
      moved to runs_superseded/ (kept for transparency) and re-queued automatically.
      CV records without an env field are back-filled with the session's logged version
      (they were produced in a 5.16.1 session) -- set CFG['env']['backfill_cv_env']=False
      to re-run them instead.
  E4  RESTORE picks the NEWEST previous output (marker timestamp), not the one with
      the most run records (superseding lowers the count).
  E5  Every run record stores the env AND the effective generation configuration.
  E6  New figures: dataset statistics (D_fig_*), validation figures (V_fig_*).

RUN THIS CELL FIRST IN *EVERY* KAGGLE SESSION. It defines every function used by
Parts 2 and 3. It is fully resumable: finished work is detected on disk and skipped.

What this part does (first session):
  1. Restores results from a previous session (if attached as an input).
  2. Builds a NEW frozen split (new seed -> a test set nobody has looked at),
     with group-aware splitting over exact AND near-duplicates (MinHash-LSH on
     BOTH English and Telugu, no skipped buckets), then a cross-split leakage
     audit that re-splits automatically until it passes.
  3. Freezes optional external test sets (FLORES+ devtest; your own CSVs).
  4. Zero-shot mBART-50 reference (no fine-tuning).
  5. Trains baseline_s42 and pscab_s42 with an IDENTICAL step budget
     (no wall-clock truncation: a run either completes or is not started).
  6. Validation-only analysis report (the test set is NEVER scored before Part 3;
     test predictions are generated with frozen models and stored blind).

Protocol fixes vs. the previous notebook (see review):
  - Step-matched training for every condition (was: time-truncated, queue-biased).
  - Deterministic, per-sentence random-mask control on content tokens only
    (was: re-sampled every forward pass, included special tokens).
  - New control: pscab_all_tokens (same module on every content token) to test
    whether *preposition targeting* matters or only the extra capacity.
  - Corpus-level paired bootstrap CIs + approximate-randomization p-values on
    BLEU/chrF++ (was: mean sentence-BLEU).
  - Gate analysis compared against the UNTRAINED gate (init control) + gamma and
    bias/hidden norm ratio logged; attention-mass analysis; figures saved as PDF+PNG.

KAGGLE WORKFLOW (because of 12 h / GPU limits):
  Session 1 : Part 1 cell               ("Save Version" -> "Save & Run All")
  Session 2+: attach the previous version's OUTPUT as an input to the notebook,
              run Part 1 cell (quick: everything is resumed) then Part 2 cell.
              Repeat Part 2 until it says the main queue is complete.
  Then      : Part 1 cell + Part 3 cell (cross-validation, 2 runs per session);
              when CV and the main queue are complete, Part 3 does the final
              test evaluation, statistics, tables, figures and report.
================================================================================
"""

# ==============================================================================
# 00. ENVIRONMENT PIN  (must execute before transformers is imported)
# ==============================================================================
ENV_PIN = {"transformers": "5.16.1", "sacrebleu": "2.6.0"}
import importlib.metadata as _md
import subprocess as _sp0
import sys as _sys0
for _pkg, _ver in ENV_PIN.items():
    try:
        _cur = _md.version(_pkg)
    except _md.PackageNotFoundError:
        _cur = None
    if _cur != _ver:
        if _pkg in _sys0.modules:
            raise RuntimeError(f"{_pkg} {_cur} is already imported in this kernel. Restart the kernel and run "
                               f"this cell FIRST so {_pkg}=={_ver} can be installed before import.")
        print(f"[pin] installing {_pkg}=={_ver} (found {_cur})")
        _sp0.run([_sys0.executable, "-m", "pip", "install", "-q", f"{_pkg}=={_ver}"], check=True)

# ==============================================================================
# 0. BOOTSTRAP (small packages only; never touches torch/numpy versions)
# ==============================================================================
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "0"
os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")
os.environ["TOKENIZERS_PARALLELISM"] = "false"

import importlib, importlib.util, subprocess, sys as _sys
_REQ = {"sacrebleu": "sacrebleu", "sentencepiece": "sentencepiece", "h5py": "h5py"}
_miss = [p for m, p in _REQ.items() if importlib.util.find_spec(m) is None]
if _miss:
    print(f"[bootstrap] installing {_miss}")
    _r = subprocess.run([_sys.executable, "-m", "pip", "install", "-q", *_miss])
    if _r.returncode != 0:
        print(f"[bootstrap] pip install failed for {_miss}; continuing (install them manually if needed).")
    importlib.invalidate_caches()
if importlib.util.find_spec("spacy") is not None:
    try:
        import spacy as _sp
        _sp.load("en_core_web_sm")
    except Exception:
        try:
            subprocess.run([_sys.executable, "-m", "spacy", "download", "en_core_web_sm"], check=True)
            importlib.invalidate_caches()
        except Exception as _e:
            print(f"[bootstrap] spaCy model download failed ({_e}); fallback list will be used.")

# ==============================================================================
# 1. IMPORTS
# ==============================================================================
import re, gc, json, math, time, glob, shutil, random, zlib, hashlib, unicodedata, platform
from collections import defaultdict

import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset

import sacrebleu
from sacrebleu.metrics import BLEU, CHRF, TER
from transformers import MBartForConditionalGeneration, MBart50TokenizerFast

try:
    get_ipython().run_line_magic("matplotlib", "inline")
except Exception:
    pass
try:
    import spacy
    _SPACY_OK = True
except Exception:
    _SPACY_OK = False
try:
    import h5py
    _H5_OK = True
except Exception:
    _H5_OK = False

import transformers as _tf
from transformers.utils import logging as _hf_logging
_hf_logging.set_verbosity_error()          # silences the repeated max_length/max_new_tokens notice
if _tf.__version__ != ENV_PIN["transformers"]:
    raise RuntimeError(f"transformers {_tf.__version__} != pinned {ENV_PIN['transformers']}. Restart the kernel; "
                       f"this cell must be the first thing executed in the session.")

SESSION_START = time.time()

# ==============================================================================
# 2. CONFIGURATION  (edit here only)
# ==============================================================================
CFG = {
    "paths": {
        "raw_csv": "/kaggle/input/datasets/panjanagalaxmi/14-16run-pscab/bpcc_telugu_fixed100k samples.csv",
        "results_dir": "/kaggle/working/pscab_q1",
        "previous_results_dir": None,   # optional explicit path; else auto-searched in /kaggle/input
        "en_col": None, "te_col": None,
    },
    "data": {
        "max_selected_pairs": 40000,
        "select_seed": 2026,             # NEW seed -> new, never-inspected test set
        "split_seed": 2026,
        "split": {"train": 0.8, "val": 0.1, "test": 0.1},
        "near_dup_threshold": 0.8,       # Jaccard on char-5-grams, EN and TE separately
        "minhash_perms": 128, "minhash_bands": 16,
    },
    "external": {
        "try_flores_plus": True,         # needs Internet + HF_TOKEN (Kaggle secret) with access granted
        "csvs": {},                      # e.g. {"in22_gen": "/kaggle/input/.../in22_gen_en_te.csv"} (cols en,te)
    },
    "model": {"backbone": "facebook/mbart-large-50-many-to-many-mmt", "src_lang": "en_XX",
              "tgt_lang": "te_IN", "max_source_length": 128, "max_target_length": 128},
    "optim": {
        "lr": 5e-5, "pscab_lr": 1e-3, "weight_decay": 0.01, "warmup_frac": 0.10,
        "label_smoothing": 0.1, "clip": 1.0, "fp16": True,
        "train_batch_size": 4, "grad_accum": 16,        # effective batch 64
        "max_steps": 3000,                              # IDENTICAL for every condition (= 6 epochs of 32k)
        "eval_every": 250, "patience_evals": 4, "min_delta": 0.002,
        "eval_batch_size": 16,
    },
    "decoding": {"num_beams": 4, "max_new_tokens": 128},
    "experiment": {
        "seeds": [42, 123, 2024],
        "mask_seed": 13,
        "robustness_subset": 2000, "diag_subset": 1000, "attention_subset": 300,
        "save_models_for": ["pscab_s42", "baseline_s42", "pscab_random_mask_s123", "pscab_all_tokens_s123"],
        "export_h5_for": ["pscab_s42"],
    },
    "session": {"limit_hours": 11.6, "default_run_hours": 3.6},
    "env": {
        "supersede_other_env": True,     # move runs from another transformers version aside and re-run them
        "backfill_cv_env": True,         # CV records written before v3 lack 'env'; they ran under 5.16.1
        "drift_regenerate": True,        # re-decode the old pscab_s42 (.h5) in this env for the appendix (once)
    },
    "cv": {"k": 5, "models": ["baseline", "pscab"], "seed": 42, "inner_val_size": 2000},
}

R = CFG["paths"]["results_dir"]
DIRS = {k: f"{R}/{k}" for k in ["data", "runs", "preds", "logs", "figures", "tables",
                                   "analysis", "models", "cv"]}
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

# Conditions. All PS-CAB variants are the identity function at step 0.
CONDITIONS = {
    "baseline":          {"use_pscab": False, "gate": "learned", "gamma": "learned", "mask": "true"},
    "pscab":             {"use_pscab": True,  "gate": "learned", "gamma": "learned", "mask": "true"},
    "pscab_no_gate":     {"use_pscab": True,  "gate": "none",    "gamma": "learned", "mask": "true"},
    "pscab_no_gamma":    {"use_pscab": True,  "gate": "learned", "gamma": "fixed",   "mask": "true"},
    "pscab_random_mask": {"use_pscab": True,  "gate": "learned", "gamma": "learned", "mask": "random"},
    "pscab_all_tokens":  {"use_pscab": True,  "gate": "learned", "gamma": "learned", "mask": "all"},
}
ABLATIONS = ["pscab_random_mask", "pscab_all_tokens", "pscab_no_gate", "pscab_no_gamma"]


def build_main_queue():
    """Prioritise completing random-mask and all-token controls across all seeds."""
    s = CFG["experiment"]["seeds"]

    q = [("baseline", s[0]), ("pscab", s[0])]
    q += [(a, s[0]) for a in ABLATIONS]

    for sd in s[1:]:
        q += [("baseline", sd), ("pscab", sd)]

    q += [("pscab_random_mask", s[1]),
          ("pscab_all_tokens", s[1])]
    q += [("pscab_random_mask", s[2]),
          ("pscab_all_tokens", s[2])]

    q += [("pscab_no_gate", s[1]),
          ("pscab_no_gamma", s[1])]
    q += [("pscab_no_gate", s[2]),
          ("pscab_no_gamma", s[2])]

    return [f"{c}_s{sd}" for c, sd in q]


def parse_run_id(run_id):
    """'pscab_no_gate_s42' -> ('pscab_no_gate', 42, None); 'cv3_pscab_s42' -> ('pscab', 42, 3)."""
    fold = None
    m = re.match(r"^cv(\d+)_(.+)$", run_id)
    if m:
        fold, run_id = int(m.group(1)), m.group(2)
    cond, seed = run_id.rsplit("_s", 1)
    return cond, int(seed), fold


# ==============================================================================
# 3. PERSISTENCE ACROSS KAGGLE SESSIONS
# ==============================================================================
def _find_previous_results():
    if CFG["paths"]["previous_results_dir"]:
        return CFG["paths"]["previous_results_dir"]
    cands = []
    for depth in range(1, 6):
        pat = "/kaggle/input/" + "/".join(["*"] * depth) + "/pscab_q1/RESULTS_MARKER.json"
        cands += glob.glob(pat)
    if not cands:
        return None

    def key(marker):
        try:
            with open(marker) as f:
                t = json.load(f).get("updated", "")
        except Exception:
            t = ""
        return (t, len(glob.glob(os.path.join(os.path.dirname(marker), "runs", "*.json"))))

    best = max(cands, key=key)
    print(f"[restore] {len(cands)} previous output(s) found; newest = {os.path.dirname(best)} ({key(best)[0]})")
    return os.path.dirname(best)


def restore_previous_results():
    os.makedirs(R, exist_ok=True)
    fresh = not glob.glob(f"{R}/runs/*.json") and not glob.glob(f"{R}/runs_superseded/*.json")
    prev = _find_previous_results()
    if prev and fresh and os.path.abspath(prev) != os.path.abspath(R):
        print(f"[restore] copying previous results from {prev} "
              f"({len(glob.glob(f'{prev}/runs/*.json'))} run records) ...")
        shutil.copytree(prev, R, dirs_exist_ok=True)
    for d in DIRS.values():
        os.makedirs(d, exist_ok=True)
    os.makedirs(f"{R}/runs_superseded", exist_ok=True)
    with open(f"{R}/RESULTS_MARKER.json", "w") as f:
        json.dump({"updated": time.strftime("%Y-%m-%d %H:%M:%S")}, f)


def jdump(obj, path):
    with open(path, "w", encoding="utf-8") as f:
        json.dump(obj, f, indent=2, ensure_ascii=False, default=_json_default)


def _json_default(o):
    if isinstance(o, (np.integer,)):
        return int(o)
    if isinstance(o, (np.floating,)):
        return float(o)
    if isinstance(o, np.ndarray):
        return o.tolist()
    return str(o)


def jload(path):
    if not os.path.exists(path):
        return None
    with open(path, encoding="utf-8") as f:
        return json.load(f)


def read_csv_str(path):
    return pd.read_csv(path, dtype=str, keep_default_na=False)


def run_record_path(run_id):
    return f"{DIRS['runs']}/{run_id}.json"


def run_done(run_id):
    r = jload(run_record_path(run_id))
    return bool(r) and r.get("status") == "complete"


def session_hours_left():
    return CFG["session"]["limit_hours"] - (time.time() - SESSION_START) / 3600


def estimated_run_hours():
    hrs = [r["wall_hours_total"] for r in (jload(p) for p in glob.glob(f"{DIRS['runs']}/*.json"))
           if r and r.get("status") == "complete" and r.get("wall_hours_total") and r["kind"] != "zeroshot"]
    return (max(hrs) * 1.1) if hrs else CFG["session"]["default_run_hours"]


# ==============================================================================
# 4. SMALL UTILITIES
# ==============================================================================
def set_all_seeds(seed):
    random.seed(seed); np.random.seed(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)
    torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


def free_gpu():
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()


def amp_ctx():
    return torch.autocast(device_type="cuda", dtype=torch.float16,
                          enabled=torch.cuda.is_available() and CFG["optim"]["fp16"])


def make_scaler():
    en = torch.cuda.is_available() and CFG["optim"]["fp16"]
    try:
        return torch.amp.GradScaler("cuda", enabled=en)
    except Exception:
        return torch.cuda.amp.GradScaler(enabled=en)


def environment_info():
    info = {"python": platform.python_version(), "torch": torch.__version__,
            "cuda": torch.version.cuda, "gpu": torch.cuda.get_device_name(0) if torch.cuda.is_available() else None,
            "sacrebleu": sacrebleu.__version__, "spacy": _SPACY_OK, "env_pin": ENV_PIN}
    try:
        import transformers
        info["transformers"] = transformers.__version__
    except Exception:
        pass
    return info


def save_fig(name):
    plt.tight_layout()
    for ext in ("pdf", "png"):
        plt.savefig(f"{DIRS['figures']}/{name}.{ext}", dpi=300 if ext == "png" else None, bbox_inches="tight")
    plt.show()
    plt.close()


def save_table(df, name, caption=""):
    df.to_csv(f"{DIRS['tables']}/{name}.csv", index=False)
    try:
        with open(f"{DIRS['tables']}/{name}.tex", "w", encoding="utf-8") as f:
            f.write(df.to_latex(index=False, float_format="%.2f", caption=caption, label=f"tab:{name}"))
    except Exception:
        pass


# ==============================================================================
# 5. DATA: load, clean, exact dedup
# ==============================================================================
TELUGU_RE = re.compile(r"[\u0C00-\u0C7F]")
LATIN_RE = re.compile(r"[A-Za-z]")
URL_RE = re.compile(r"https?://|www\.")
HTML_RE = re.compile(r"<[^>]+>")
LATIN_WORD_RE = re.compile(r"\b[A-Za-z]{2,}\b")


def normalize_text(s):
    return re.sub(r"\s+", " ", unicodedata.normalize("NFC", str(s))).strip()


def normkey(s):
    return re.sub(r"\s+", " ", unicodedata.normalize("NFC", str(s)).lower()).strip()


def locate_raw_csv():
    p = CFG["paths"]["raw_csv"]
    if os.path.exists(p):
        return p
    hits = [h for h in glob.glob("/kaggle/input/**/*.csv", recursive=True)
            if "bpcc" in h.lower() and "tel" in h.lower()]
    if hits:
        print(f"[data] configured path missing; using {hits[0]}")
        return hits[0]
    raise FileNotFoundError(f"Dataset not found at {p!r}; set CFG['paths']['raw_csv'].")


def load_and_clean():
    path = locate_raw_csv()
    df = pd.read_csv(path, encoding="utf-8-sig", keep_default_na=False, na_values=[""], low_memory=False)
    print(f"[data] loaded {path}: {df.shape}, columns={list(df.columns)}")
    te_col = CFG["paths"]["te_col"] or max(
        df.columns, key=lambda c: df[c].astype(str).head(1000).apply(
            lambda x: len(TELUGU_RE.findall(x)) / max(len(x), 1)).mean())
    rest = [c for c in df.columns if c != te_col]
    en_col = CFG["paths"]["en_col"] or max(
        rest, key=lambda c: df[c].astype(str).head(1000).apply(
            lambda x: len(LATIN_RE.findall(x)) / max(len(x), 1)).mean())
    print(f"[data] English column={en_col!r}, Telugu column={te_col!r}")
    df = df.rename(columns={en_col: "en", te_col: "te"})[["en", "te"]].copy()
    n0 = len(df)
    df["en"] = df["en"].fillna("").astype(str).apply(normalize_text)
    df["te"] = df["te"].fillna("").astype(str).apply(normalize_text)
    reasons = {}
    checks = [
        ("empty", (df["en"] == "") | (df["te"] == "")),
        ("invalid_unicode", df["en"].str.contains("\ufffd") | df["te"].str.contains("\ufffd")),
        ("no_telugu_in_target", ~df["te"].apply(lambda s: bool(TELUGU_RE.search(s)))),
        ("no_latin_in_source", ~df["en"].apply(lambda s: bool(LATIN_RE.search(s)))),
        ("html", df["en"].str.contains(HTML_RE) | df["te"].str.contains(HTML_RE)),
        ("url_only", df["en"].apply(lambda s: bool(URL_RE.search(s)) and len(s.split()) <= 3)),
    ]
    keep = pd.Series(True, index=df.index)
    for name, m in checks:
        reasons[name] = int((m & keep).sum())
        keep &= ~m
    df = df[keep].reset_index(drop=True)
    n_clean = len(df)
    key = df["en"].apply(normkey) + " ||| " + df["te"].apply(normkey)
    dup = key.duplicated(keep="first")
    df = df[~dup].reset_index(drop=True)
    report = {"original_rows": n0, "removed_by_rule": reasons, "rows_after_cleaning": n_clean,
              "exact_duplicate_pairs_removed": int(dup.sum()), "rows_after_exact_dedup": len(df),
              "same_english_multiple_targets": int(df.groupby(df["en"].apply(normkey))["te"].nunique().gt(1).sum())}
    print(json.dumps(report, indent=2))
    return df, report


# ==============================================================================
# 6. NEAR-DUPLICATES: deterministic MinHash-LSH (char 5-grams), exact-Jaccard verified
# ==============================================================================
_MH_P = np.uint64((1 << 31) - 1)


def _shingle_hashes(s, n=5):
    s = normkey(s)
    grams = {s[i:i + n] for i in range(max(len(s) - n + 1, 1))}
    arr = np.fromiter((zlib.crc32(g.encode("utf-8")) % int(_MH_P) for g in grams), dtype=np.uint64,
                      count=len(grams))
    return np.unique(arr)


def minhash_near_duplicate_pairs(texts, threshold, n_perm=128, n_bands=16, seed=1):
    """Returns verified pairs (i, j, jaccard) with Jaccard >= threshold. No bucket is skipped:
    buckets above 3000 members are verified in a star pattern against their first member."""
    rng = np.random.RandomState(seed)
    A = rng.randint(1, int(_MH_P), size=n_perm).astype(np.uint64)
    B = rng.randint(0, int(_MH_P), size=n_perm).astype(np.uint64)
    sh = [_shingle_hashes(t) for t in texts]
    sigs = np.empty((len(texts), n_perm), dtype=np.uint64)
    for i, h in enumerate(sh):
        sigs[i] = ((h[:, None] * A[None, :] + B[None, :]) % _MH_P).min(axis=0)
    rows = n_perm // n_bands
    cand = set()
    for b in range(n_bands):
        buckets = defaultdict(list)
        block = sigs[:, b * rows:(b + 1) * rows]
        for i in range(len(texts)):
            buckets[block[i].tobytes()].append(i)
        for members in buckets.values():
            if len(members) < 2:
                continue
            if len(members) <= 3000:
                for x in range(len(members)):
                    for y in range(x + 1, len(members)):
                        cand.add((members[x], members[y]))
            else:
                for y in members[1:]:
                    cand.add((members[0], y))
    out = []
    for i, j in cand:
        a, b_ = sh[i], sh[j]
        inter = np.intersect1d(a, b_, assume_unique=True).size
        jac = inter / (a.size + b_.size - inter)
        if jac >= threshold:
            out.append((i, j, float(jac)))
    return out


class UnionFind:
    def __init__(self, n):
        self.p = list(range(n))

    def find(self, x):
        while self.p[x] != x:
            self.p[x] = self.p[self.p[x]]
            x = self.p[x]
        return x

    def union(self, a, b):
        ra, rb = self.find(a), self.find(b)
        if ra != rb:
            self.p[ra] = rb


def group_ids_for(df, extra_pairs=()):
    """Groups rows that share a normalized EN, a normalized TE, or are near-duplicates (EN or TE)."""
    n = len(df)
    uf = UnionFind(n)
    for col in ("en", "te"):
        first = {}
        for i, k in enumerate(df[col].apply(normkey)):
            if k in first:
                uf.union(first[k], i)
            else:
                first[k] = i
    thr = CFG["data"]["near_dup_threshold"]
    stats = {}
    for col in ("en", "te"):
        pairs = minhash_near_duplicate_pairs(df[col].tolist(), thr, CFG["data"]["minhash_perms"],
                                             CFG["data"]["minhash_bands"], seed=1)
        stats[f"near_dup_pairs_{col}"] = len(pairs)
        for i, j, _ in pairs:
            uf.union(i, j)
    for i, j in extra_pairs:
        uf.union(i, j)
    gid = np.array([uf.find(i) for i in range(n)])
    _, gid = np.unique(gid, return_inverse=True)
    return gid, stats


def split_by_groups(gid, seed):
    fr = CFG["data"]["split"]
    n = len(gid)
    groups = defaultdict(list)
    for i, g in enumerate(gid):
        groups[g].append(i)
    order = list(groups.keys())
    np.random.default_rng(seed).shuffle(order)
    t_test, t_val = int(n * fr["test"]), int(n * fr["val"])
    split = np.array(["train"] * n, dtype=object)
    cnt_test = cnt_val = 0
    for g in order:
        idx = groups[g]
        if cnt_test < t_test:
            split[idx] = "test"; cnt_test += len(idx)
        elif cnt_val < t_val:
            split[idx] = "val"; cnt_val += len(idx)
    return split


def leakage_audit(df, split):
    """Exact (normalized EN / TE / pair) + near-duplicate (Jaccard >= threshold, EN and TE,
    independent MinHash seed) overlaps between every pair of partitions."""
    thr = CFG["data"]["near_dup_threshold"]
    rep, cross_pairs = {}, []
    parts = ["train", "val", "test"]
    en_k, te_k = df["en"].apply(normkey).values, df["te"].apply(normkey).values
    for a_i in range(3):
        for b_i in range(a_i + 1, 3):
            a, b = parts[a_i], parts[b_i]
            ma, mb = split == a, split == b
            rep[f"{a}-{b}"] = {
                "exact_en": len(set(en_k[ma]) & set(en_k[mb])),
                "exact_te": len(set(te_k[ma]) & set(te_k[mb])),
                "exact_pair": len(set(zip(en_k[ma], te_k[ma])) & set(zip(en_k[mb], te_k[mb]))),
                "near_dup_en": 0, "near_dup_te": 0}
    for col in ("en", "te"):
        pairs = minhash_near_duplicate_pairs(df[col].tolist(), thr, CFG["data"]["minhash_perms"],
                                             CFG["data"]["minhash_bands"], seed=777)
        for i, j, _ in pairs:
            if split[i] != split[j]:
                a, b = sorted([split[i], split[j]], key=parts.index)
                rep[f"{a}-{b}"][f"near_dup_{col}"] += 1
                cross_pairs.append((i, j))
    total = sum(v for d in rep.values() for v in d.values())
    rep["status"] = "PASS" if total == 0 else "FAIL"
    rep["method"] = (f"exact normalized match (EN, TE, pair) + MinHash-LSH char-5-gram Jaccard >= {thr} "
                     f"on EN and TE separately ({CFG['data']['minhash_perms']} perms, independent seed), "
                     f"every candidate verified by exact Jaccard, no buckets skipped")
    return rep, cross_pairs


# ==============================================================================
# 7. PREPOSITIONS + SUBWORD MASKS
# ==============================================================================
FALLBACK_PREPS = {"in", "on", "at", "to", "from", "for", "with", "by", "into", "through", "between",
                  "among", "under", "over", "before", "after", "during", "without", "within", "about",
                  "against", "along", "around", "behind", "below", "beneath", "beside", "beyond", "near",
                  "of", "off", "onto", "since", "toward", "towards", "underneath", "until", "upon"}
_NLP = None
_SPAN_CACHE = {}


def get_nlp():
    global _NLP
    if _NLP is None:
        _NLP = False
        if _SPACY_OK:
            try:
                _NLP = spacy.load("en_core_web_sm", disable=["parser", "ner", "lemmatizer"])
            except Exception as e:
                print(f"[prep] spaCy unavailable ({e}); using fallback list.")
    return _NLP


def ensure_spans(sents):
    todo = [s for s in dict.fromkeys(sents) if s not in _SPAN_CACHE]
    if not todo:
        return
    nlp = get_nlp()
    if nlp:
        for s, doc in zip(todo, nlp.pipe(todo, batch_size=512)):
            _SPAN_CACHE[s] = [(t.idx, t.idx + len(t.text), t.text.lower()) for t in doc if t.pos_ == "ADP"]
    else:
        for s in todo:
            _SPAN_CACHE[s] = [(m.start(), m.end(), m.group(0).lower())
                              for m in re.finditer(r"\b[a-zA-Z]+\b", s) if m.group(0).lower() in FALLBACK_PREPS]


def prep_count(s):
    ensure_spans([s])
    return len(_SPAN_CACHE[s])


_MASK_CACHE = {}


def compute_masks(sents, tokenizer, mode):
    """mode: 'true' (preposition subwords), 'random' (same count, fixed per sentence, content tokens only),
    'all' (every content token). Special/pad tokens (offset s==e) are never masked."""
    L = CFG["model"]["max_source_length"]
    key = (hashlib.md5("\n".join(sents).encode("utf-8")).hexdigest(), mode)
    if key in _MASK_CACHE:
        return _MASK_CACHE[key]
    ensure_spans(sents)
    out = []
    for i in range(0, len(sents), 1000):
        part = sents[i:i + 1000]
        enc = tokenizer(part, return_offsets_mapping=True, truncation=True, max_length=L, padding="max_length")
        for sent, offs in zip(part, enc["offset_mapping"]):
            content = [k for k, (s, e) in enumerate(offs) if s != e]
            true = np.zeros(L, dtype=np.uint8)
            for k in content:
                s, e = offs[k]
                for ps, pe, _ in _SPAN_CACHE[sent]:
                    if s < pe and e > ps:
                        true[k] = 1
                        break
            if mode == "true":
                m = true
            elif mode == "all":
                m = np.zeros(L, dtype=np.uint8); m[content] = 1
            elif mode == "random":
                m = np.zeros(L, dtype=np.uint8)
                kk = min(int(true.sum()), len(content))
                if kk:
                    h = int(hashlib.md5(f"{CFG['experiment']['mask_seed']}|{sent}".encode()).hexdigest()[:8], 16)
                    m[np.random.default_rng(h).choice(content, size=kk, replace=False)] = 1
            else:
                raise ValueError(mode)
            out.append(m)
    _MASK_CACHE[key] = out
    return out


# ==============================================================================
# 8. PS-CAB MODULE + MODEL WRAPPER
# ==============================================================================
class PSCAB(nn.Module):
    """H' = H + gamma * g(H) * (W_b H) * mask   (applied to encoder outputs, i.e. the keys/values
    of every decoder cross-attention layer). Identity at step 0 in every variant:
      gamma learned : gamma=0, W_b ~ N(0, 0.02)
      gamma fixed   : gamma=1 (buffer), W_b = 0
    NOTE FOR THE PAPER: this is a gated feature modulation of the cross-attention memory,
    not an additive bias on attention logits -- describe it that way."""

    def __init__(self, d, gate="learned", gamma="learned"):
        super().__init__()
        self.gate_mode, self.gamma_mode = gate, gamma
        self.W_b = nn.Linear(d, d)
        nn.init.zeros_(self.W_b.bias)
        if gamma == "learned":
            nn.init.normal_(self.W_b.weight, 0.0, 0.02)
            self.gamma = nn.Parameter(torch.zeros(1))
        else:
            nn.init.zeros_(self.W_b.weight)
            self.register_buffer("gamma", torch.ones(1))
        if gate == "learned":
            self.W_g = nn.Linear(d, d)

    def gate(self, H):
        return torch.sigmoid(self.W_g(H)) if self.gate_mode == "learned" else torch.ones_like(H)

    def forward(self, H, mask):
        m = mask.unsqueeze(-1).to(H.dtype)
        return H + self.gamma * self.gate(H) * self.W_b(H) * m


def shift_right_mbart(labels, pad_id):
    x = labels.clone()
    x[x == -100] = pad_id
    idx = (x.ne(pad_id).sum(dim=1) - 1).unsqueeze(-1)
    start = x.gather(1, idx).squeeze(-1)
    out = x.clone()
    out[:, 1:] = x[:, :-1]
    out[:, 0] = start
    return out


class PSCABMBart(nn.Module):
    def __init__(self, cond, attn_impl=None):
        super().__init__()
        spec = CONDITIONS[cond]
        kw = {"attn_implementation": attn_impl} if attn_impl else {}
        self.backbone = MBartForConditionalGeneration.from_pretrained(CFG["model"]["backbone"], **kw)
        self.pad_id = self.backbone.config.pad_token_id
        self.use_pscab = spec["use_pscab"]
        if self.use_pscab:
            self.pscab = PSCAB(self.backbone.config.d_model, spec["gate"], spec["gamma"])

    def encode(self, input_ids, attention_mask, mask):
        enc = self.backbone.model.encoder(input_ids=input_ids, attention_mask=attention_mask, return_dict=True)
        if self.use_pscab:
            enc.last_hidden_state = self.pscab(enc.last_hidden_state, mask)
        return enc

    def forward(self, input_ids, attention_mask, mask, labels):
        enc = self.encode(input_ids, attention_mask, mask)
        out = self.backbone(attention_mask=attention_mask, encoder_outputs=enc,
                            decoder_input_ids=shift_right_mbart(labels, self.pad_id),
                            use_cache=False, return_dict=True)
        return out.logits

    @torch.no_grad()
    def generate(self, input_ids, attention_mask, mask, **kw):
        enc = self.encode(input_ids, attention_mask, mask)
        return self.backbone.generate(encoder_outputs=enc, attention_mask=attention_mask, **kw)

    def param_report(self):
        return {"total_parameters": sum(p.numel() for p in self.parameters()),
                "pscab_added_parameters": sum(p.numel() for n, p in self.named_parameters() if n.startswith("pscab."))}


# ==============================================================================
# 9. TOKENIZER / DATASET / COLLATE
# ==============================================================================
_TOK = None


def get_tokenizer():
    global _TOK
    if _TOK is None:
        _TOK = MBart50TokenizerFast.from_pretrained(CFG["model"]["backbone"])
        _TOK.src_lang, _TOK.tgt_lang = CFG["model"]["src_lang"], CFG["model"]["tgt_lang"]
    return _TOK


class PairDataset(Dataset):
    def __init__(self, df, tok, mode):
        self.en, self.te = df["en"].tolist(), df["te"].tolist()
        self.mask = compute_masks(self.en, tok, mode)
        self.true = compute_masks(self.en, tok, "true")

    def __len__(self):
        return len(self.en)

    def __getitem__(self, i):
        return self.en[i], self.te[i], self.mask[i], self.true[i]


def collate(batch, tok):
    en, te, m, t = zip(*batch)
    src = tok(list(en), max_length=CFG["model"]["max_source_length"], truncation=True, padding=True,
              return_tensors="pt")
    tgt = tok(text_target=list(te), max_length=CFG["model"]["max_target_length"], truncation=True,
              padding=True, return_tensors="pt")
    labels = tgt["input_ids"].clone()
    labels[labels == tok.pad_token_id] = -100
    L = src["input_ids"].shape[1]
    return {"input_ids": src["input_ids"], "attention_mask": src["attention_mask"],
            "mask": torch.from_numpy(np.stack(m)[:, :L].astype(np.int64)),
            "true_mask": torch.from_numpy(np.stack(t)[:, :L].astype(np.int64)),
            "labels": labels}


# ==============================================================================
# 10. METRICS + CORPUS-LEVEL PAIRED STATISTICS
# ==============================================================================
_BLEU = None


def bleu_metric():
    """spBLEU with the FLORES-200 SentencePiece tokenizer (standard for Indic MT); falls back to 13a."""
    global _BLEU
    if _BLEU is None:
        for tok in ("flores200", "flores101", "13a"):
            try:
                b = BLEU(tokenize=tok)
                b.corpus_score(["తెలుగు వాక్యం"], [["తెలుగు వాక్యం"]])
                _BLEU = b
                print(f"[metrics] BLEU tokenizer = {tok}")
                break
            except Exception as e:
                print(f"[metrics] BLEU tokenizer {tok} unavailable: {e}")
    return _BLEU


def chrf_metric():
    return CHRF(word_order=2)


def compute_metrics(hyps, refs):
    hyps = [str(h) for h in hyps]
    b, c = bleu_metric(), chrf_metric()
    res = {"bleu": round(b.corpus_score(hyps, [refs]).score, 4),
           "chrf++": round(c.corpus_score(hyps, [refs]).score, 4),
           "n": len(refs), "bleu_signature": str(b.get_signature()), "chrf_signature": str(c.get_signature())}
    try:
        t = TER()
        res["ter"] = round(t.corpus_score(hyps, [refs]).score, 4)
    except Exception as e:
        print(f"[metrics] TER failed ({e}); NaN")
        res["ter"] = float("nan")
    return res


class StatMetric:
    """Per-sentence sufficient statistics so that resampling = summing rows (exact corpus-level score)."""

    def __init__(self, name):
        self.name = name
        self.m = {"bleu": bleu_metric, "chrf++": chrf_metric}.get(name, lambda: None)()

    def stats(self, hyps, refs, seg_scores=None):
        if self.name == "comet":
            return np.asarray(seg_scores, dtype=np.float64).reshape(-1, 1)
        return np.asarray(self.m._extract_corpus_statistics([str(h) for h in hyps], [refs]), dtype=np.int64)

    def score(self, stat_sum, n):
        if self.name == "comet":
            return float(stat_sum[0] / n) * 100.0
        return float(self.m._compute_score_from_stats(list(stat_sum)).score)


def paired_test(sa, sb, metric, n_boot=2000, n_ar=10000, seed=12345):
    """A minus B. Returns observed delta, bootstrap 95% CI (Koehn 2004) and two-sided
    approximate-randomization p-value (Riezler & Maxwell 2005)."""
    rng = np.random.default_rng(seed)
    N = len(sa)
    obs = metric.score(sa.sum(0), N) - metric.score(sb.sum(0), N)
    boots = np.empty(n_boot)
    for k in range(n_boot):
        idx = rng.integers(0, N, N)
        boots[k] = metric.score(sa[idx].sum(0), N) - metric.score(sb[idx].sum(0), N)
    hits = 0
    for _ in range(n_ar):
        sw = rng.random(N) < 0.5
        a = np.where(sw[:, None], sb, sa).sum(0)
        b = np.where(sw[:, None], sa, sb).sum(0)
        if abs(metric.score(a, N) - metric.score(b, N)) >= abs(obs) - 1e-12:
            hits += 1
    lo, hi = np.percentile(boots, [2.5, 97.5])
    return {"delta": round(obs, 4), "ci95_low": round(float(lo), 4), "ci95_high": round(float(hi), 4),
            "p_ar": round((hits + 1) / (n_ar + 1), 5), "n": N}


def holm(pvals: dict, alpha=0.05):
    items = sorted(pvals.items(), key=lambda kv: kv[1])
    m, out, ok = len(items), {}, True
    for r, (k, p) in enumerate(items, 1):
        thr = alpha / (m - r + 1)
        ok = ok and p < thr
        out[k] = {"holm_threshold": round(thr, 5), "significant_holm": bool(ok)}
    return out


# ==============================================================================
# 11. GENERATION
# ==============================================================================
def gen_kwargs(tok):
    return {"num_beams": CFG["decoding"]["num_beams"], "max_new_tokens": CFG["decoding"]["max_new_tokens"],
            "forced_bos_token_id": tok.convert_tokens_to_ids(CFG["model"]["tgt_lang"])}


def generation_record(model, tok):
    """Effective decoding configuration (explicit kwargs + the model's stored generation_config)."""
    out = {"explicit_kwargs": gen_kwargs(tok)}
    try:
        gc_ = model.backbone.generation_config.to_dict()
        out["model_generation_config"] = {k: gc_.get(k) for k in (
            "max_length", "num_beams", "early_stopping", "length_penalty", "no_repeat_ngram_size",
            "forced_eos_token_id", "decoder_start_token_id", "repetition_penalty", "do_sample")}
    except Exception as e:
        out["model_generation_config"] = f"unavailable ({e})"
    return out


@torch.no_grad()
def generate(model, tok, df, mode, bs=None):
    bs = bs or CFG["optim"]["eval_batch_size"]
    order = np.argsort(-df["en"].str.len().values, kind="stable")
    sdf = df.iloc[order].reset_index(drop=True)
    loader = DataLoader(PairDataset(sdf, tok, mode), batch_size=bs, shuffle=False,
                        collate_fn=lambda b: collate(b, tok))
    tgt_id = tok.convert_tokens_to_ids(CFG["model"]["tgt_lang"])
    model.eval()
    out, t0 = [], time.time()
    for batch in loader:
        batch = {k: v.to(DEVICE) for k, v in batch.items()}
        with amp_ctx():
            g = model.generate(batch["input_ids"], batch["attention_mask"], batch["mask"], **gen_kwargs(tok))
        out.extend(tok.batch_decode(g, skip_special_tokens=True))
    el = time.time() - t0
    preds = [None] * len(df)
    for k, o in enumerate(order):
        preds[o] = out[k]
    return preds, {"sentences_per_sec": len(df) / max(el, 1e-6), "gen_seconds": el}


def save_preds(run_id, split_name, df, preds):
    d = df[["en", "te"]].copy()
    d["prediction"] = preds
    d.to_csv(f"{DIRS['preds']}/{run_id}__{split_name}.csv", index=False)


def load_preds(run_id, split_name):
    p = f"{DIRS['preds']}/{run_id}__{split_name}.csv"
    return read_csv_str(p)["prediction"].tolist() if os.path.exists(p) else None


# ==============================================================================
# 12. TRAINING (step-matched, eval every N steps, early stopping on validation NLL)
# ==============================================================================
@torch.no_grad()
def val_nll(model, loader):
    model.eval()
    tot, ntok = 0.0, 0
    for batch in loader:
        batch = {k: v.to(DEVICE) for k, v in batch.items()}
        with amp_ctx():
            logits = model(batch["input_ids"], batch["attention_mask"], batch["mask"], batch["labels"])
        l = F.cross_entropy(logits.float().view(-1, logits.size(-1)), batch["labels"].view(-1),
                            ignore_index=-100, reduction="sum")
        tot += l.item()
        ntok += int((batch["labels"] != -100).sum())
    model.train()
    return tot / max(ntok, 1)


def train_model(cond, seed, train_df, val_df, run_id):
    o = CFG["optim"]
    spec = CONDITIONS[cond]
    print(f"\n{'='*72}\nTRAIN {run_id}: {spec}\n{'='*72}")
    set_all_seeds(seed)
    tok = get_tokenizer()
    g = torch.Generator(); g.manual_seed(seed)
    tl = DataLoader(PairDataset(train_df, tok, spec["mask"]), batch_size=o["train_batch_size"], shuffle=True,
                    generator=g, collate_fn=lambda b: collate(b, tok), drop_last=False)
    vl = DataLoader(PairDataset(val_df, tok, spec["mask"]), batch_size=o["eval_batch_size"], shuffle=False,
                    collate_fn=lambda b: collate(b, tok))
    model = PSCABMBart(cond).to(DEVICE)
    init_gate = None
    if model.use_pscab and spec["gate"] == "learned":
        init_gate = {k: v.detach().cpu().clone() for k, v in model.pscab.W_g.state_dict().items()}
    base = [p for n, p in model.named_parameters() if not n.startswith("pscab.")]
    extra = [p for n, p in model.named_parameters() if n.startswith("pscab.")]
    groups = [{"params": base, "lr": o["lr"]}]
    if extra:
        groups.append({"params": extra, "lr": o["pscab_lr"], "weight_decay": 0.0})
    opt = torch.optim.AdamW(groups, weight_decay=o["weight_decay"])
    total, warm = o["max_steps"], max(1, int(o["warmup_frac"] * o["max_steps"]))
    sched = torch.optim.lr_scheduler.LambdaLR(
        opt, lambda s: s / warm if s < warm else 0.5 * (1 + math.cos(math.pi * min(1.0, (s - warm) / max(1, total - warm)))))
    scaler = make_scaler()
    params = base + extra

    log, best, best_step, best_state, bad = [], float("inf"), 0, None, 0
    step, micro, run_loss, run_n, nonfinite, stop = 0, 0, 0.0, 0, 0, "max_steps_reached"
    t0 = time.time()
    model.train()
    opt.zero_grad(set_to_none=True)
    epoch = 0
    while step < total and stop == "max_steps_reached":
        for batch in tl:
            batch = {k: v.to(DEVICE) for k, v in batch.items()}
            with amp_ctx():
                logits = model(batch["input_ids"], batch["attention_mask"], batch["mask"], batch["labels"])
                loss = F.cross_entropy(logits.view(-1, logits.size(-1)), batch["labels"].view(-1),
                                       ignore_index=-100, label_smoothing=o["label_smoothing"])
            if torch.isfinite(loss):
                scaler.scale(loss / o["grad_accum"]).backward()
                run_loss += loss.item(); run_n += 1
            else:
                nonfinite += 1
            micro += 1
            if micro % o["grad_accum"] != 0:
                continue
            scaler.unscale_(opt)
            torch.nn.utils.clip_grad_norm_(params, o["clip"])
            scaler.step(opt); scaler.update(); sched.step()
            opt.zero_grad(set_to_none=True)
            step += 1
            if step % o["eval_every"] == 0 or step == total:
                v = val_nll(model, vl)
                row = {"step": step, "epoch": round(step * o["grad_accum"] * o["train_batch_size"] / len(train_df), 3),
                       "train_loss_ls": run_loss / max(run_n, 1), "val_nll": v,
                       "lr": sched.get_last_lr()[0], "elapsed_h": (time.time() - t0) / 3600}
                if model.use_pscab and spec["gamma"] == "learned":
                    row["gamma"] = float(model.pscab.gamma.item())
                log.append(row)
                run_loss, run_n = 0.0, 0
                flag = ""
                if v < best - o["min_delta"]:
                    best, best_step, bad = v, step, 0
                    best_state = {k: t.detach().cpu().clone() for k, t in model.state_dict().items()}
                    flag = "  <- best"
                else:
                    bad += 1
                print(f"[{run_id}] step {step}/{total} ep {row['epoch']:.2f} train(ls) {row['train_loss_ls']:.4f} "
                      f"val_nll {v:.4f} lr {row['lr']:.2e} {row['elapsed_h']:.2f}h{flag}")
                if bad >= o["patience_evals"]:
                    stop = "early_stopping"
                    break
            if step >= total:
                break
        epoch += 1
    train_h = (time.time() - t0) / 3600
    model.load_state_dict(best_state)
    del opt, sched, scaler, best_state
    free_gpu()
    pd.DataFrame(log).to_csv(f"{DIRS['logs']}/{run_id}_curve.csv", index=False)
    meta = {"steps_trained": step, "best_step": best_step, "best_val_nll": best, "stop_reason": stop,
            "train_hours": train_h, "sec_per_step": train_h * 3600 / max(step, 1),
            "nonfinite_loss_batches": nonfinite, **model.param_report()}
    if model.use_pscab and spec["gamma"] == "learned":
        meta["gamma_at_best"] = float(model.pscab.gamma.item())
    print(f"[{run_id}] done: {step} steps, best step {best_step} (val_nll {best:.4f}), stop={stop}, {train_h:.2f}h")
    return model, tok, meta, init_gate


# ==============================================================================
# 13. IN-SESSION DIAGNOSTICS (need the model in memory; results saved to disk)
# ==============================================================================
@torch.no_grad()
def pscab_diagnostics(model, tok, df, cond, init_gate, run_id):
    """Gate values at preposition vs other content positions: trained gate vs the SAME gate at
    initialisation (control), gamma, and ||bias|| / ||H|| on masked positions."""
    spec = CONDITIONS[cond]
    sub = df.sample(n=min(CFG["experiment"]["diag_subset"], len(df)), random_state=0).reset_index(drop=True)
    loader = DataLoader(PairDataset(sub, tok, spec["mask"]), batch_size=CFG["optim"]["eval_batch_size"],
                        shuffle=False, collate_fn=lambda b: collate(b, tok))
    Wg0 = None
    if init_gate is not None:
        Wg0 = (init_gate["weight"].to(DEVICE).float(), init_gate["bias"].to(DEVICE).float())
    gt_p, gt_n, g0_p, g0_n, ratios = [], [], [], [], []
    pm = model.pscab
    learned_gate = spec["gate"] == "learned"
    for batch in loader:
        batch = {k: v.to(DEVICE) for k, v in batch.items()}
        with amp_ctx():
            H = model.backbone.model.encoder(input_ids=batch["input_ids"], attention_mask=batch["attention_mask"],
                                             return_dict=True).last_hidden_state
        H = H.float()
        valid = batch["attention_mask"].bool()
        tm = batch["true_mask"].bool() & valid
        content = valid.clone()
        content[:, 0] = False                                   # language-code token
        eos = batch["attention_mask"].sum(1) - 1
        content[torch.arange(len(eos), device=DEVICE), eos] = False   # </s>
        nm = content & ~tm
        G = pm.gate(H)
        if learned_gate:
            g = G.mean(-1)
            gt_p += g[tm].cpu().tolist(); gt_n += g[nm].cpu().tolist()
            if Wg0 is not None:
                g0 = torch.sigmoid(H @ Wg0[0].T + Wg0[1]).mean(-1)
                g0_p += g0[tm].cpu().tolist(); g0_n += g0[nm].cpu().tolist()
        am = batch["mask"].bool() & valid
        bias = pm.gamma * G * pm.W_b(H)
        r = bias.norm(dim=-1) / H.norm(dim=-1).clamp_min(1e-6)
        ratios += r[am].cpu().tolist()

    def auc(pos, neg):
        if not pos or not neg:
            return None
        try:
            from sklearn.metrics import roc_auc_score
            return float(roc_auc_score([1] * len(pos) + [0] * len(neg), pos + neg))
        except Exception:
            return None

    rep = {"gamma": float(model.pscab.gamma.item()),
           "bias_to_hidden_norm_ratio_mean": float(np.mean(ratios)) if ratios else None,
           "bias_to_hidden_norm_ratio_median": float(np.median(ratios)) if ratios else None,
           "n_prep_positions": len(gt_p), "n_other_positions": len(gt_n)}
    if gt_p:
        rep.update({"gate_trained_prep_mean": float(np.mean(gt_p)), "gate_trained_other_mean": float(np.mean(gt_n)),
                    "gate_trained_auc_prep_vs_other": auc(gt_p, gt_n)})
    if g0_p:
        rep.update({"gate_init_prep_mean": float(np.mean(g0_p)), "gate_init_other_mean": float(np.mean(g0_n)),
                    "gate_init_auc_prep_vs_other": auc(g0_p, g0_n)})
    rng = np.random.default_rng(0)
    samp = lambda x: rng.choice(np.array(x), size=min(20000, len(x)), replace=False) if x else np.array([])
    np.savez_compressed(f"{DIRS['analysis']}/{run_id}_gate_samples.npz", trained_prep=samp(gt_p),
                        trained_other=samp(gt_n), init_prep=samp(g0_p), init_other=samp(g0_n))
    return rep


@torch.no_grad()
def attention_mass(model, tok, df, cond, run_id):
    """Teacher-forced cross-attention mass on preposition source tokens (mean over heads and target
    positions), last layer and mean over layers, normalised by the uniform expectation n_prep/n_src."""
    try:
        eager = PSCABMBart(cond, attn_impl="eager")
        eager.load_state_dict(model.state_dict())
        eager.to(DEVICE).eval()
    except Exception as e:
        print(f"[attention] eager model unavailable ({e}); skipped.")
        return None
    sents = df["en"].tolist()
    ensure_spans(sents)
    cand = [i for i, s in enumerate(sents) if _SPAN_CACHE[s]]
    if not cand:
        del eager
        free_gpu()
        return None
    rng = np.random.default_rng(0)
    idx = sorted(rng.choice(cand, size=min(CFG["experiment"]["attention_subset"], len(cand)), replace=False))
    sub = df.iloc[idx].reset_index(drop=True)
    true = compute_masks(sub["en"].tolist(), tok, "true")
    applied = compute_masks(sub["en"].tolist(), tok, CONDITIONS[cond]["mask"])
    last, allL, ratio_last = [], [], []
    for k in range(len(sub)):
        src = tok(sub["en"][k], return_tensors="pt", truncation=True, max_length=CFG["model"]["max_source_length"]).to(DEVICE)
        tgt = tok(text_target=sub["te"][k], return_tensors="pt", truncation=True,
                  max_length=CFG["model"]["max_target_length"]).to(DEVICE)
        S = src["input_ids"].shape[1]
        pm = torch.from_numpy(applied[k][:S].astype(np.int64)).unsqueeze(0).to(DEVICE)
        tm = torch.from_numpy(true[k][:S].astype(bool)).to(DEVICE)
        if tm.sum() == 0:
            continue
        enc = eager.encode(src["input_ids"], src["attention_mask"], pm)
        out = eager.backbone(attention_mask=src["attention_mask"], encoder_outputs=enc,
                             decoder_input_ids=shift_right_mbart(tgt["input_ids"], eager.pad_id),
                             output_attentions=True, use_cache=False, return_dict=True)
        A = torch.stack([a[0].float().mean(0) for a in out.cross_attentions])  # (layers, T, S)
        mass = A[:, :, tm].sum(-1).mean(-1)                                     # (layers,)
        last.append(float(mass[-1])); allL.append(float(mass.mean()))
        ratio_last.append(float(mass[-1]) / (float(tm.sum()) / S))
    del eager
    free_gpu()
    return {"n_sentences": len(last), "prep_mass_last_layer": float(np.mean(last)),
            "prep_mass_mean_layers": float(np.mean(allL)),
            "prep_mass_last_layer_over_uniform": float(np.mean(ratio_last))}


PERTURBATIONS = {
    "punct_spacing": lambda s: re.sub(r"\s*([,.;:!?])", r" \1 ", s).strip(),
    "lowercase": lambda s: s.lower(),
    "drop_final_punct": lambda s: s[:-1].rstrip() if s and s[-1] in ".!?" else s,
}


def robustness(model, tok, df, cond):
    sub = df.sample(n=min(CFG["experiment"]["robustness_subset"], len(df)), random_state=1).reset_index(drop=True)
    refs = sub["te"].tolist()
    base, _ = generate(model, tok, sub, CONDITIONS[cond]["mask"])
    b = compute_metrics(base, refs)
    rows = [{"perturbation": "none", "bleu": b["bleu"], "chrf++": b["chrf++"], "delta_bleu": 0.0, "delta_chrf++": 0.0}]
    for name, fn in PERTURBATIONS.items():
        p = sub.copy(); p["en"] = p["en"].apply(fn)
        preds, _ = generate(model, tok, p, CONDITIONS[cond]["mask"])
        m = compute_metrics(preds, refs)
        rows.append({"perturbation": name, "bleu": m["bleu"], "chrf++": m["chrf++"],
                     "delta_bleu": round(m["bleu"] - b["bleu"], 4), "delta_chrf++": round(m["chrf++"] - b["chrf++"], 4)})
    return rows


def save_model_bundle(model, tok, run_id, record):
    d = f"{DIRS['models']}/{run_id}"
    os.makedirs(d, exist_ok=True)
    model.backbone.half().save_pretrained(d, safe_serialization=True)
    model.backbone.float()
    tok.save_pretrained(d)
    if model.use_pscab:
        torch.save({k: v.half() for k, v in model.pscab.state_dict().items()}, f"{d}/pscab_module_fp16.pt")
    jdump({"condition": record["condition"], "seed": record["seed"], "config": CFG,
           "reload": "PSCABMBart(cond); backbone=MBartForConditionalGeneration.from_pretrained(dir); "
                     "pscab.load_state_dict(torch.load('pscab_module_fp16.pt')) (cast to float)"},
          f"{d}/bundle_info.json")
    if run_id in CFG["experiment"]["export_h5_for"] and _H5_OK:
        with h5py.File(f"{d}/{run_id}_fp16_state_dict.h5", "w") as f:
            f.attrs["note"] = "PyTorch state_dict tensors (fp16) of PSCABMBart; NOT a Keras model."
            for k, v in model.state_dict().items():
                f.create_dataset(k, data=v.detach().half().cpu().numpy(), compression="gzip")
    print(f"[save] model bundle -> {d}")


# ==============================================================================
# 14. RUN EXECUTORS
# ==============================================================================
def load_splits():
    return tuple(read_csv_str(f"{DIRS['data']}/{n}.csv") for n in ("train", "val", "test"))


def external_sets():
    man = jload(f"{DIRS['data']}/external_manifest.json") or {}
    return {k: read_csv_str(v) for k, v in man.items() if os.path.exists(v)}


def execute_main_run(run_id):
    """Train one (condition, seed) on the frozen split. Generates validation predictions (scored now),
    and TEST / external predictions (stored blind, scored only in Part 3)."""
    if run_done(run_id):
        print(f"[skip] {run_id} already complete.")
        return jload(run_record_path(run_id))
    need = estimated_run_hours()
    if session_hours_left() < need:
        print(f"[time] {run_id} needs ~{need:.1f}h but only {session_hours_left():.1f}h left in this "
              f"session -> NOT started (never truncated). Run it next session.")
        return None
    cond, seed, _ = parse_run_id(run_id)
    t_all = time.time()
    train, val, test = load_splits()
    model, tok, meta, init_gate = train_model(cond, seed, train, val, run_id)
    mode = CONDITIONS[cond]["mask"]
    rec = {"run_id": run_id, "kind": "main", "condition": cond, "seed": seed, "spec": CONDITIONS[cond],
           "status": "complete", "env": environment_info(), **meta}

    rec["generation"] = generation_record(model, tok)
    vp, vinfo = generate(model, tok, val, mode)
    save_preds(run_id, "val", val, vp)
    rec["val_metrics"] = compute_metrics(vp, val["te"].tolist())
    rec["inference_sentences_per_sec"] = vinfo["sentences_per_sec"]
    print(f"[{run_id}] VAL {rec['val_metrics']['bleu']} BLEU / {rec['val_metrics']['chrf++']} chrF++")

    tp, _ = generate(model, tok, test, mode)
    save_preds(run_id, "test", test, tp)                    # blind: not scored here
    for name, edf in external_sets().items():
        ep, _ = generate(model, tok, edf, mode)
        save_preds(run_id, f"ext_{name}", edf, ep)

    if model.use_pscab:
        rec["diagnostics"] = pscab_diagnostics(model, tok, val, cond, init_gate, run_id)
        print(f"[{run_id}] diagnostics: {rec['diagnostics']}")
    if seed == CFG["experiment"]["seeds"][0]:
        rec["attention"] = attention_mass(model, tok, val, cond, run_id)
        rec["robustness_val"] = robustness(model, tok, val, cond)
    if run_id in CFG["experiment"]["save_models_for"]:
        save_model_bundle(model, tok, run_id, rec)
    del model
    free_gpu()
    rec["wall_hours_total"] = (time.time() - t_all) / 3600
    jdump(rec, run_record_path(run_id))
    print(f"[{run_id}] COMPLETE in {rec['wall_hours_total']:.2f}h")
    return rec


def execute_zeroshot():
    rid = "zeroshot_mbart50"
    if run_done(rid):
        print(f"[skip] {rid} already complete.")
        return
    t0 = time.time()
    train, val, test = load_splits()
    tok = get_tokenizer()
    model = PSCABMBart("baseline").to(DEVICE).eval()
    vp, info = generate(model, tok, val, "true")
    save_preds(rid, "val", val, vp)
    tp, _ = generate(model, tok, test, "true")
    save_preds(rid, "test", test, tp)
    for name, edf in external_sets().items():
        ep, _ = generate(model, tok, edf, "true")
        save_preds(rid, f"ext_{name}", edf, ep)
    rec = {"run_id": rid, "kind": "zeroshot", "condition": "zeroshot_mbart50", "seed": None, "status": "complete",
           "env": environment_info(), "generation": generation_record(model, tok),
           "val_metrics": compute_metrics(vp, val["te"].tolist()), "inference_sentences_per_sec": info["sentences_per_sec"],
           "wall_hours_total": (time.time() - t0) / 3600, **model.param_report()}
    jdump(rec, run_record_path(rid))
    print(f"[zeroshot] VAL {rec['val_metrics']['bleu']} BLEU / {rec['val_metrics']['chrf++']} chrF++")
    del model
    free_gpu()


def run_next_main_runs(max_runs=2):
    q = build_main_queue()
    pending = [r for r in q if not run_done(r)]
    print(f"[queue] main: {len(q) - len(pending)}/{len(q)} complete; pending next: {pending[:max_runs]}")
    started = 0
    for rid in pending:
        if started >= max_runs:
            break
        rec = execute_main_run(rid)
        if rec is None:
            break
        started += 1
    left = [r for r in q if not run_done(r)]
    print(f"[queue] main queue after this session: {len(q) - len(left)}/{len(q)} complete. "
          + ("MAIN QUEUE COMPLETE." if not left else f"Next session will run: {left[:2]}"))


# ==============================================================================
# 15. DATA PREPARATION (frozen once)
# ==============================================================================
def prepare_flores_plus(train_all):
    try:
        try:
            from kaggle_secrets import UserSecretsClient
            os.environ.setdefault("HF_TOKEN", UserSecretsClient().get_secret("HF_TOKEN"))
        except Exception:
            pass
        from datasets import load_dataset
        en = load_dataset("openlanguagedata/flores_plus", "eng_Latn", split="devtest").to_pandas()
        te = load_dataset("openlanguagedata/flores_plus", "tel_Telu", split="devtest").to_pandas()
        df = en[["id", "text"]].rename(columns={"text": "en"}).merge(
            te[["id", "text"]].rename(columns={"text": "te"}), on="id")[["en", "te"]]
        return df
    except Exception as e:
        print(f"[external] FLORES+ not available ({e}). Accept the dataset terms on HF and add HF_TOKEN "
              f"as a Kaggle secret, or supply CSVs in CFG['external']['csvs'].")
        return None


def freeze_external_sets(all_df):
    man_p = f"{DIRS['data']}/external_manifest.json"
    if os.path.exists(man_p):
        return jload(man_p)
    os.makedirs(f"{DIRS['data']}/external", exist_ok=True)
    known_en, known_te = set(all_df["en"].apply(normkey)), set(all_df["te"].apply(normkey))
    sets = {}
    if CFG["external"]["try_flores_plus"]:
        f = prepare_flores_plus(all_df)
        if f is not None:
            sets["flores_plus_devtest"] = f
    for name, p in CFG["external"]["csvs"].items():
        if os.path.exists(p):
            d = read_csv_str(p)
            sets[name] = d[["en", "te"]]
    man, rep = {}, {}
    for name, d in sets.items():
        d = d.copy()
        d["en"], d["te"] = d["en"].apply(normalize_text), d["te"].apply(normalize_text)
        ov = d["en"].apply(normkey).isin(known_en) | d["te"].apply(normkey).isin(known_te)
        rep[name] = {"n_raw": len(d), "overlap_with_our_data_removed": int(ov.sum())}
        d = d[~ov].reset_index(drop=True)
        p = f"{DIRS['data']}/external/{name}.csv"
        d.to_csv(p, index=False)
        man[name] = p
        rep[name]["n_used"] = len(d)
    jdump(man, man_p)
    jdump(rep, f"{DIRS['data']}/external_report.json")
    print(f"[external] frozen external test sets: {rep if rep else 'none'} "
          f"(external sets are frozen now; adding one later would leave earlier runs without predictions)")
    return man


def stage_data_prep():
    names = ["train", "val", "test"]
    summ_p = f"{DIRS['data']}/data_summary.json"
    if all(os.path.exists(f"{DIRS['data']}/{n}.csv") for n in names) and os.path.exists(summ_p):
        print("[data] frozen split found -- reusing it.")
        summ = jload(summ_p)
        print(json.dumps(summ["leakage_audit"], indent=2))
        return summ
    df, clean_rep = load_and_clean()
    rng = np.random.default_rng(CFG["data"]["select_seed"])
    n = min(CFG["data"]["max_selected_pairs"], len(df))
    sel = df.iloc[np.sort(rng.choice(len(df), size=n, replace=False))].reset_index(drop=True)
    sel["row_id"] = np.arange(len(sel))
    print(f"[data] selected {len(sel)} pairs (seed={CFG['data']['select_seed']}); building near-duplicate groups ...")
    t0 = time.time()
    extra = []
    for it in range(5):
        gid, nd = group_ids_for(sel, extra)
        split = split_by_groups(gid, CFG["data"]["split_seed"])
        audit, cross = leakage_audit(sel, split)
        print(f"[data] split iteration {it}: audit {audit['status']} "
              f"({len(cross)} cross-split near-dup pairs) [{time.time() - t0:.0f}s]")
        if audit["status"] == "PASS":
            break
        extra += cross
    if audit["status"] != "PASS":
        raise RuntimeError("Leakage audit failed after 5 re-splits: " + json.dumps(audit))
    sel["group_id"], sel["split"] = gid, split
    for nm in names:
        sel[sel["split"] == nm][["row_id", "group_id", "en", "te"]].to_csv(f"{DIRS['data']}/{nm}.csv", index=False)
    sizes = {nm: int((split == nm).sum()) for nm in names}
    ensure_spans(sel["en"].tolist())
    prep_share = {nm: float(np.mean([len(_SPAN_CACHE[s]) > 0 for s in sel.loc[sel["split"] == nm, "en"]]))
                  for nm in names}
    summ = {"cleaning": clean_rep, "selected_pairs": len(sel), "split_sizes": sizes, "near_dup_grouping": nd,
            "n_groups": int(gid.max() + 1), "largest_group": int(np.bincount(gid).max()),
            "share_sentences_with_preposition": prep_share, "leakage_audit": audit,
            "preposition_detector": "spaCy en_core_web_sm ADP" if get_nlp() else "fallback list"}
    jdump(summ, summ_p)
    print(json.dumps(summ, indent=2))
    freeze_external_sets(sel)
    return summ


# ==============================================================================
# 16. PROGRESS REPORT (VALIDATION ONLY -- run after every part)
# ==============================================================================
def all_records(kind=None):
    recs = [jload(p) for p in sorted(glob.glob(f"{DIRS['runs']}/*.json"))]
    recs = [r for r in recs if r and r.get("status") == "complete"]
    return [r for r in recs if kind is None or r.get("kind") == kind]


def progress_report():
    print(f"\n{'#'*72}\nPROGRESS REPORT -- VALIDATION SET ONLY (test set not scored)\n{'#'*72}")
    recs = [r for r in all_records() if r.get("kind") in ("main", "zeroshot")]
    if not recs:
        print("no completed runs yet.")
        return
    train, val, test = load_splits()
    refs = val["te"].tolist()

    # --- Table P1: per-run validation results + training facts
    rows = []
    for r in recs:
        vm = r["val_metrics"]
        rows.append({"run": r["run_id"], "condition": r["condition"], "seed": r.get("seed"),
                     "steps": r.get("steps_trained"), "best_step": r.get("best_step"),
                     "stop": r.get("stop_reason"), "val_nll": r.get("best_val_nll"),
                     "BLEU": vm["bleu"], "chrF++": vm["chrf++"], "TER": vm["ter"],
                     "train_h": r.get("train_hours"), "transformers": rec_tf_version(r)})
    t1 = pd.DataFrame(rows)
    print("\n[P1] Per-run validation results\n" + t1.to_string(index=False))
    save_table(t1, "P1_validation_per_run", "Validation results per run")
    steps = t1.dropna(subset=["steps"])
    if len(steps) and steps["steps"].nunique() > 1:
        print("[P1 note] runs differ in steps_trained ONLY because of early stopping on validation NLL "
              "(identical max_steps budget); report best_step in the paper.")

    # --- Table P2: condition mean +- SD over seeds (validation)
    main = t1[t1["condition"].isin(CONDITIONS)]
    if len(main):
        agg = main.groupby("condition").agg(n_seeds=("seed", "nunique"), BLEU_mean=("BLEU", "mean"),
                                            BLEU_sd=("BLEU", "std"), chrF_mean=("chrF++", "mean"),
                                            chrF_sd=("chrF++", "std")).reset_index()
        print("\n[P2] Validation, mean +- SD over seeds\n" + agg.to_string(index=False))
        save_table(agg, "P2_validation_by_condition")

    # --- Table P3: paired corpus-level tests on validation (development statistics)
    preds = {r["run_id"]: load_preds(r["run_id"], "val") for r in recs}
    comps = []
    for r in recs:
        c, s = r["condition"], r.get("seed")
        if c == "pscab":
            comps.append((f"pscab_s{s}", f"baseline_s{s}"))
        elif c in ABLATIONS:
            comps.append((f"pscab_s{s}", f"{c}_s{s}"))
    comps = [(a, b) for a, b in comps if preds.get(a) and preds.get(b)]
    if comps:
        rows = []
        for mname in ("bleu", "chrf++"):
            M = StatMetric(mname)
            cache = {k: M.stats(v, refs) for k, v in preds.items() if v}
            pv = {}
            for a, b in comps:
                res = paired_test(cache[a], cache[b], M, n_boot=1000, n_ar=2000)
                rows.append({"metric": mname, "A": a, "B": b, **res})
                pv[f"{mname}:{a}-{b}"] = res["p_ar"]
            h = holm(pv)
            for row in rows:
                k = f"{row['metric']}:{row['A']}-{row['B']}"
                if k in h:
                    row.update(h[k])
        t3 = pd.DataFrame(rows)
        print("\n[P3] Paired tests on VALIDATION (A - B; AR p-value, bootstrap 95% CI, Holm per metric)\n"
              + t3.to_string(index=False))
        save_table(t3, "P3_validation_paired_tests")

    # --- Table P4: preposition subsets on validation (+ noise floor)
    ensure_spans(val["en"].tolist())
    has = np.array([len(_SPAN_CACHE[s]) > 0 for s in val["en"]])
    rows = []
    for rid, p in preds.items():
        if not p:
            continue
        P = np.array(p, dtype=object); Rf = np.array(refs, dtype=object)
        for name, m in [("with_prep", has), ("no_prep", ~has)]:
            if m.sum():
                mm = compute_metrics(list(P[m]), list(Rf[m]))
                rows.append({"run": rid, "subset": name, "n": int(m.sum()), "BLEU": mm["bleu"], "chrF++": mm["chrf++"]})
    t4 = pd.DataFrame(rows)
    print("\n[P4] Validation by preposition subset\n" + t4.to_string(index=False))
    print("[P4 note] On no_prep sentences the true-mask PS-CAB module is an exact no-op; any baseline-vs-pscab "
          "gap there measures training variance only (use it as a noise floor for H2).")
    save_table(t4, "P4_validation_preposition_subsets")

    # --- Table P5: module diagnostics
    rows = [{"run": r["run_id"], **r["diagnostics"]} for r in recs if r.get("diagnostics")]
    if rows:
        t5 = pd.DataFrame(rows)
        print("\n[P5] PS-CAB diagnostics (gate trained vs init control; gamma; bias/hidden ratio)\n" + t5.to_string(index=False))
        save_table(t5, "P5_pscab_diagnostics")
    rows = [{"run": r["run_id"], **r["attention"]} for r in recs if r.get("attention")]
    if rows:
        t6 = pd.DataFrame(rows)
        print("\n[P6] Cross-attention mass on preposition tokens (validation, teacher-forced)\n" + t6.to_string(index=False))
        save_table(t6, "P6_attention_mass")
    rows = [{"run": r["run_id"], **x} for r in recs if r.get("robustness_val") for x in r["robustness_val"]]
    if rows:
        t7 = pd.DataFrame(rows)
        print("\n[P7] Robustness to source perturbations (validation subset)\n" + t7.to_string(index=False))
        save_table(t7, "P7_robustness_validation")

    # --- Figures (validation only)
    try:
        validation_figures()
    except Exception as e:
        print(f"[figures] validation_figures failed: {e}")
    for r in recs:
        f = f"{DIRS['analysis']}/{r['run_id']}_gate_samples.npz"
        if os.path.exists(f) and r["run_id"].endswith(f"_s{CFG['experiment']['seeds'][0]}"):
            z = np.load(f)
            if z["trained_prep"].size:
                plt.figure(figsize=(7, 3.8))
                bins = np.linspace(0, 1, 60)
                plt.hist(z["trained_other"], bins, alpha=.5, density=True, label="trained: other")
                plt.hist(z["trained_prep"], bins, alpha=.5, density=True, label="trained: preposition")
                if z["init_prep"].size:
                    plt.hist(z["init_prep"], bins, histtype="step", density=True, label="init: preposition")
                    plt.hist(z["init_other"], bins, histtype="step", density=True, label="init: other")
                plt.xlabel("mean gate value"); plt.legend(fontsize=7); plt.title(f"Gate distribution ({r['run_id']}, validation)")
                save_fig(f"P_fig_gate_{r['run_id']}")
    q = build_main_queue()
    done = [x for x in q if run_done(x)]
    print(f"\n[status] main queue {len(done)}/{len(q)} complete; zero-shot "
          f"{'done' if run_done('zeroshot_mbart50') else 'pending'}; session time left {session_hours_left():.1f}h")



# ==============================================================================
# 16b. PAPER STYLE, ENVIRONMENT DRIFT DIAGNOSTIC, SUPERSEDING, DATASET / VALIDATION FIGURES
# ==============================================================================
COND_LABEL = {"zeroshot_mbart50": "Zero-shot mBART-50", "baseline": "Baseline", "pscab": "PS-CAB",
              "pscab_no_gate": "PS-CAB −gate", "pscab_no_gamma": "PS-CAB −γ (fixed)",
              "pscab_random_mask": "Random mask", "pscab_all_tokens": "All tokens"}
COND_COLOR = {"zeroshot_mbart50": "#7f7f7f", "baseline": "#1f77b4", "pscab": "#d62728",
              "pscab_no_gate": "#ff7f0e", "pscab_no_gamma": "#9467bd",
              "pscab_random_mask": "#2ca02c", "pscab_all_tokens": "#8c564b"}
COND_ORDER = ["baseline", "pscab", "pscab_no_gate", "pscab_no_gamma", "pscab_random_mask", "pscab_all_tokens"]


def apply_paper_style():
    plt.rcParams.update({"font.size": 9, "axes.titlesize": 10, "axes.labelsize": 9, "legend.fontsize": 7.5,
                         "xtick.labelsize": 8, "ytick.labelsize": 8, "figure.dpi": 110,
                         "axes.spines.top": False, "axes.spines.right": False, "axes.grid": True,
                         "grid.alpha": 0.25, "savefig.bbox": "tight"})


def rec_tf_version(r):
    return (r.get("env") or {}).get("transformers")


def _records_incl_superseded(kinds=("main", "zeroshot")):
    out = []
    for folder, sup in ((DIRS["runs"], False), (f"{R}/runs_superseded", True)):
        for p in sorted(glob.glob(f"{folder}/*.json")):
            r = jload(p)
            if r and r.get("status") == "complete" and r.get("kind") in kinds:
                r["_superseded"] = sup
                out.append(r)
    return out


def env_drift_diagnostic():
    """E2: one-time appendix diagnostic. Groups every run by transformers version and compares
    output length ratio / BP / BLEU / chrF++ / TER / val NLL; optionally re-decodes the old-environment
    pscab_s42 model (.h5) in the current environment."""
    cache = f"{DIRS['analysis']}/env_drift_diagnostic.json"
    if os.path.exists(cache):
        c = jload(cache)
        print(f"\n[drift] cached diagnostic (versions seen: {c.get('versions')}); re-decode: {c.get('redecode')}")
        return c
    recs = [r for r in _records_incl_superseded() if not r["_superseded"]]
    versions = sorted({str(rec_tf_version(r) or "unknown") for r in recs})
    if not recs or versions == [ENV_PIN["transformers"]]:
        return None
    print(f"\n{'#'*72}\nENVIRONMENT DRIFT DIAGNOSTIC (appendix) -- versions present: {versions}\n{'#'*72}")
    _, val, _ = load_splits()
    refs = val["te"].tolist()
    B = bleu_metric()
    rows = []
    for r in recs:
        pr = load_preds(r["run_id"], "val")
        if pr is None:
            continue
        sc = B.corpus_score(pr, [refs])
        rows.append({"run": r["run_id"], "condition": r["condition"], "transformers": str(rec_tf_version(r) or "unknown"),
                     "val_nll": r.get("best_val_nll"), "BLEU": round(sc.score, 2), "BP": round(sc.bp, 4),
                     "len_ratio": round(sc.sys_len / sc.ref_len, 4), "chrF++": r["val_metrics"]["chrf++"],
                     "TER": r["val_metrics"]["ter"], "mean_hyp_words": round(float(np.mean([len(x.split()) for x in pr])), 2)})
    d = pd.DataFrame(rows)
    print(d.to_string(index=False))
    save_table(d, "DIAG_env_drift_per_run", "Validation outputs grouped by transformers version (appendix)")

    redecode = None
    ps = jload(run_record_path("pscab_s42"))
    h5s = glob.glob(f"{DIRS['models']}/pscab_s42/*.h5")
    if (CFG["env"]["drift_regenerate"] and torch.cuda.is_available() and ps and h5s
            and rec_tf_version(ps) != ENV_PIN["transformers"]):
        try:
            print(f"[drift] re-decoding pscab_s42 (trained under transformers {rec_tf_version(ps)}) "
                  f"with transformers {ENV_PIN['transformers']} ...")
            m = PSCABMBart("pscab")
            with h5py.File(h5s[0], "r") as f:
                sd = {k: torch.from_numpy(f[k][()]).float() for k in f.keys()}
            miss, unexp = m.load_state_dict(sd, strict=False)
            m.to(DEVICE).eval()
            new, _ = generate(m, get_tokenizer(), val, "true")
            old = load_preds("pscab_s42", "val")
            so, sn = B.corpus_score(old, [refs]), B.corpus_score(new, [refs])
            mn = compute_metrics(new, refs)
            redecode = {"old_env": rec_tf_version(ps), "new_env": ENV_PIN["transformers"],
                        "missing_keys": len(miss), "unexpected_keys": len(unexp),
                        "old_bleu": round(so.score, 2), "new_bleu": round(sn.score, 2),
                        "old_len_ratio": round(so.sys_len / so.ref_len, 4), "new_len_ratio": round(sn.sys_len / sn.ref_len, 4),
                        "old_chrf": ps["val_metrics"]["chrf++"], "new_chrf": mn["chrf++"],
                        "old_ter": ps["val_metrics"]["ter"], "new_ter": mn["ter"],
                        "identical_outputs_share": round(float(np.mean([a == b for a, b in zip(old, new)])), 4),
                        "note": "same weights (fp16 .h5 snapshot), only the decoding environment differs"}
            redecode["verdict"] = ("DRIFT CONFIRMED" if abs(redecode["new_bleu"] - redecode["old_bleu"]) > 0.5
                                   else "no material drift (<0.5 BLEU)")
            print("[drift] re-decode:", json.dumps(redecode, indent=2))
            pd.DataFrame({"en": val["en"], "te": val["te"], "prediction": new}).to_csv(
                f"{DIRS['analysis']}/pscab_s42_REDECODED_val.csv", index=False)
            del m
            free_gpu()
        except Exception as e:
            redecode = {"error": str(e)}
            print(f"[drift] re-decode failed: {e}")
    else:
        print("[drift] re-decode skipped (needs GPU + models/pscab_s42/*.h5 + an old-env pscab_s42 record).")

    if len(d):
        fig, ax = plt.subplots(1, 2, figsize=(10, 3.6))
        vs = sorted(d["transformers"].unique())
        cmap = {v: c for v, c in zip(vs, ["#1f77b4", "#d62728", "#2ca02c", "#9467bd"])}
        ax[0].bar(range(len(d)), d["len_ratio"], color=[cmap[v] for v in d["transformers"]])
        ax[0].set_xticks(range(len(d))); ax[0].set_xticklabels(d["run"], rotation=40, ha="right", fontsize=7)
        ax[0].axhline(1.0, ls="--", c="gray", lw=.8); ax[0].set_ylabel("hyp/ref length (spBLEU tokens)")
        ax[0].set_title("(a) Output length by run")
        ax[0].set_ylim(min(d["len_ratio"].min(), 1) - 0.03, max(d["len_ratio"].max(), 1) + 0.03)
        for v in vs:
            sub = d[d["transformers"] == v]
            ax[1].scatter(sub["val_nll"], sub["BLEU"], c=cmap[v], label=f"transformers {v}", s=40)
        if redecode and "new_bleu" in redecode:
            ax[1].annotate("pscab_s42 re-decoded", xy=(ps["best_val_nll"], redecode["new_bleu"]), fontsize=7)
            ax[1].scatter([ps["best_val_nll"]], [redecode["new_bleu"]], marker="*", s=140, c="k")
        ax[1].set_xlabel("best validation NLL (teacher-forced)"); ax[1].set_ylabel("validation spBLEU")
        ax[1].set_title("(b) Decoding vs training quality"); ax[1].legend()
        save_fig("DIAG_fig_env_drift")
    out = {"versions": versions, "per_run": rows, "redecode": redecode, "pinned": ENV_PIN}
    jdump(out, cache)
    return out


def supersede_other_env_runs():
    """E3: move completed runs from a different transformers version out of the active set."""
    if not CFG["env"]["supersede_other_env"]:
        return []
    for d in ("preds_superseded", "logs_superseded", "runs_superseded"):
        os.makedirs(f"{R}/{d}", exist_ok=True)
    moved = []
    for p in sorted(glob.glob(f"{DIRS['runs']}/*.json")):
        r = jload(p)
        if not r or r.get("status") != "complete":
            continue
        v = rec_tf_version(r)
        if v is None and r.get("kind") == "cv" and CFG["env"]["backfill_cv_env"]:
            r.setdefault("env", {})
            r["env"].update({"transformers": ENV_PIN["transformers"], "backfilled_from_session_log": True})
            jdump(r, p)
            print(f"[supersede] {r['run_id']}: env back-filled ({ENV_PIN['transformers']}, from the session log)")
            continue
        if v == ENV_PIN["transformers"]:
            continue
        rid = r["run_id"]
        os.replace(p, f"{R}/runs_superseded/{rid}.json")
        for f in glob.glob(f"{DIRS['preds']}/{rid}__*.csv"):
            os.replace(f, f"{R}/preds_superseded/{os.path.basename(f)}")
        for f in glob.glob(f"{DIRS['logs']}/{rid}_curve.csv"):
            os.replace(f, f"{R}/logs_superseded/{os.path.basename(f)}")
        moved.append({"run_id": rid, "transformers": v, "moved": time.strftime("%Y-%m-%d %H:%M:%S")})
    if moved:
        log_p = f"{DIRS['analysis']}/superseded_runs.json"
        jdump((jload(log_p) or []) + moved, log_p)
        print(f"[supersede] {len(moved)} run(s) from another environment re-queued: {[m['run_id'] for m in moved]}")
    return moved


def dataset_figures():
    """Descriptive statistics of the frozen splits (no model output involved)."""
    if os.path.exists(f"{DIRS['figures']}/D_fig_lengths.pdf"):
        return
    print(f"\n{'#'*72}\nDATASET FIGURES\n{'#'*72}")
    splits = dict(zip(["train", "val", "test"], load_splits()))
    rows, cnts = [], {}
    for name, d in splits.items():
        en_l, te_l = d["en"].str.split().str.len(), d["te"].str.split().str.len()
        ensure_spans(d["en"].tolist())
        c = np.array([len(_SPAN_CACHE[x]) for x in d["en"]])
        cnts[name] = c
        rows.append({"split": name, "pairs": len(d), "EN_words_mean": round(en_l.mean(), 2), "EN_words_median": int(en_l.median()),
                     "TE_words_mean": round(te_l.mean(), 2), "TE_words_median": int(te_l.median()),
                     "TE/EN_word_ratio": round(te_l.sum() / en_l.sum(), 3),
                     "%_with_preposition": round(100 * (c > 0).mean(), 2), "prepositions_per_sentence": round(c.mean(), 3)})
    t = pd.DataFrame(rows)
    print(t.to_string(index=False))
    save_table(t, "D1_split_statistics", "Statistics of the frozen splits")

    fig, ax = plt.subplots(1, 2, figsize=(9, 3.2))
    bins = np.arange(0, 61, 2)
    for name, d in splits.items():
        ax[0].hist(d["en"].str.split().str.len().clip(upper=60), bins, histtype="step", density=True, lw=1.4, label=name)
        ax[1].hist(d["te"].str.split().str.len().clip(upper=60), bins, histtype="step", density=True, lw=1.4, label=name)
    ax[0].set_title("(a) English source length"); ax[1].set_title("(b) Telugu target length")
    for a in ax:
        a.set_xlabel("words (clipped at 60)"); a.set_ylabel("density"); a.legend()
    save_fig("D_fig_lengths")

    fig, ax = plt.subplots(1, 2, figsize=(9.5, 3.4))
    labels = ["0", "1", "2", "3", "4+"]
    w = 0.27
    for i, (name, c) in enumerate(cnts.items()):
        share = [100 * np.mean(np.minimum(c, 4) == k) for k in range(5)]
        ax[0].bar(np.arange(5) + (i - 1) * w, share, w, label=name)
    ax[0].set_xticks(range(5)); ax[0].set_xticklabels(labels)
    ax[0].set_xlabel("prepositions per source sentence"); ax[0].set_ylabel("% of sentences")
    ax[0].set_title("(a) Preposition density"); ax[0].legend()
    from collections import Counter
    words = Counter(w_ for s_ in splits["train"]["en"] for _, _, w_ in _SPAN_CACHE[s_])
    top = words.most_common(20)[::-1]
    ax[1].barh([k for k, _ in top], [v for _, v in top], color="#d62728")
    ax[1].set_xlabel("occurrences (train)"); ax[1].set_title("(b) 20 most frequent prepositions")
    save_fig("D_fig_prepositions")

    d = splits["train"]
    plt.figure(figsize=(4.6, 4))
    plt.hexbin(d["en"].str.split().str.len().clip(upper=60), d["te"].str.split().str.len().clip(upper=60),
               gridsize=35, mincnt=1, bins="log", cmap="viridis")
    plt.plot([0, 60], [0, 60], ls="--", c="w", lw=.8)
    plt.colorbar(label="log10(count)")
    plt.xlabel("English words"); plt.ylabel("Telugu words")
    plt.title("Telugu is agglutinative: fewer words per sentence")
    save_fig("D_fig_en_te_length_hexbin")


def _curve(rid, folder=None):
    p = f"{folder or DIRS['logs']}/{rid}_curve.csv"
    return pd.read_csv(p) if os.path.exists(p) else None


def validation_figures():
    """V_fig_*: validation-only figures over all completed main runs (current environment)."""
    recs = all_records("main")
    if not recs:
        return
    by_c = defaultdict(list)
    for r in recs:
        by_c[r["condition"]].append(r)
    conds = [c for c in COND_ORDER if c in by_c]

    # V1 learning curves: mean over seeds with min-max band
    plt.figure(figsize=(7.5, 4))
    for c in conds:
        cs = [x for x in (_curve(r["run_id"]) for r in by_c[c]) if x is not None]
        if not cs:
            continue
        L = min(len(x) for x in cs)
        M = np.stack([x["val_nll"].values[:L] for x in cs])
        st = cs[0]["step"].values[:L]
        plt.plot(st, M.mean(0), marker="o", ms=3, c=COND_COLOR[c], label=f"{COND_LABEL[c]} (n={len(cs)})")
        if len(cs) > 1:
            plt.fill_between(st, M.min(0), M.max(0), color=COND_COLOR[c], alpha=.15)
    plt.xlabel("optimizer step"); plt.ylabel("validation NLL / token")
    plt.title("Validation NLL (identical step budget; band = seed range)"); plt.legend(ncol=2)
    save_fig("V_fig_learning_curves")

    # V2 validation metrics with seed dots
    fig, ax = plt.subplots(1, 2, figsize=(10, 3.6))
    for a, key, lab in ((ax[0], "chrf++", "chrF++"), (ax[1], "bleu", "spBLEU")):
        for i, c in enumerate(conds):
            v = [r["val_metrics"][key] for r in by_c[c]]
            a.bar(i, np.mean(v), color=COND_COLOR[c], alpha=.55)
            a.scatter([i] * len(v), v, c="k", s=14, zorder=3)
        a.set_xticks(range(len(conds))); a.set_xticklabels([COND_LABEL[c] for c in conds], rotation=25, ha="right")
        allv = [r["val_metrics"][key] for r in recs]
        a.set_ylim(min(allv) - 1.5, max(allv) + 1); a.set_ylabel(f"validation {lab}")
    ax[0].set_title("(a) chrF++ (bars = mean, dots = seeds)"); ax[1].set_title("(b) spBLEU")
    save_fig("V_fig_validation_metrics")

    # V3 module diagnostics
    dg = [(r["run_id"], r["condition"], r["diagnostics"]) for r in recs if r.get("diagnostics")]
    if dg:
        fig, ax = plt.subplots(1, 3, figsize=(12, 3.6))
        x = np.arange(len(dg))
        names = [d[0] for d in dg]
        ax[0].bar(x, [d[2].get("gamma", np.nan) for d in dg], color=[COND_COLOR[d[1]] for d in dg])
        ax[0].axhline(0, c="gray", lw=.8); ax[0].set_title("(a) learned scale γ")
        ax[1].bar(x, [d[2].get("bias_to_hidden_norm_ratio_mean", np.nan) for d in dg], color=[COND_COLOR[d[1]] for d in dg])
        ax[1].set_title("(b) ‖γ·g·W_bH‖ / ‖H‖ on masked tokens")
        tr = [d[2].get("gate_trained_auc_prep_vs_other", np.nan) or np.nan for d in dg]
        it = [d[2].get("gate_init_auc_prep_vs_other", np.nan) or np.nan for d in dg]
        ax[2].bar(x - .2, it, .4, label="gate at init", color="#bbbbbb")
        ax[2].bar(x + .2, tr, .4, label="gate trained", color="#d62728")
        ax[2].axhline(.5, ls="--", c="k", lw=.8); ax[2].set_ylim(0, 1)
        ax[2].set_title("(c) gate AUC: preposition vs other tokens"); ax[2].legend()
        for a in ax:
            a.set_xticks(x); a.set_xticklabels(names, rotation=40, ha="right", fontsize=7)
        save_fig("V_fig_module_diagnostics")

    # V4 attention mass, V5 robustness (seed-0 runs)
    at = [(r["run_id"], r["condition"], r["attention"]) for r in recs if r.get("attention")]
    if at:
        fig, ax = plt.subplots(1, 2, figsize=(10, 3.4))
        x = np.arange(len(at))
        ax[0].bar(x, [a[2]["prep_mass_last_layer_over_uniform"] for a in at], color=[COND_COLOR[a[1]] for a in at])
        ax[0].axhline(1, ls="--", c="k", lw=.8)
        ax[0].set_title("(a) last-layer attention on prepositions ÷ uniform share")
        ax[1].bar(x, [a[2]["prep_mass_mean_layers"] for a in at], color=[COND_COLOR[a[1]] for a in at])
        ax[1].set_title("(b) mean-over-layers attention mass on prepositions")
        for a in ax:
            a.set_xticks(x); a.set_xticklabels([q[0] for q in at], rotation=35, ha="right", fontsize=7)
        save_fig("V_fig_attention_mass")
    rb = [(r["run_id"], r["condition"], r["robustness_val"]) for r in recs if r.get("robustness_val")]
    if rb:
        perts = [p_ for p_ in PERTURBATIONS]
        plt.figure(figsize=(8, 3.6))
        w = 0.8 / len(rb)
        for i, (rid, c, rows) in enumerate(rb):
            dmap = {q["perturbation"]: q["delta_chrf++"] for q in rows}
            plt.bar(np.arange(len(perts)) + i * w, [dmap.get(p_, np.nan) for p_ in perts], w,
                    color=COND_COLOR[c], label=rid)
        plt.axhline(0, c="gray", lw=.8)
        plt.xticks(np.arange(len(perts)) + w * (len(rb) - 1) / 2, perts)
        plt.ylabel("Δ chrF++ vs unperturbed"); plt.title("Robustness to source perturbations (validation)")
        plt.legend(fontsize=6.5, ncol=2)
        save_fig("V_fig_robustness")

    # V6 output length ratio per run (validation)
    _, val, _ = load_splits()
    refs = val["te"].tolist()
    B = bleu_metric()
    rows = []
    for r in all_records():
        if r.get("kind") not in ("main", "zeroshot"):
            continue
        pr = load_preds(r["run_id"], "val")
        if pr is None:
            continue
        sc = B.corpus_score(pr, [refs])
        rows.append((r["run_id"], r["condition"], sc.sys_len / sc.ref_len))
    if rows:
        plt.figure(figsize=(8, 3.4))
        plt.bar(range(len(rows)), [q[2] for q in rows], color=[COND_COLOR.get(q[1], "#777") for q in rows])
        plt.axhline(1, ls="--", c="k", lw=.8)
        plt.xticks(range(len(rows)), [q[0] for q in rows], rotation=40, ha="right", fontsize=7)
        vals = [q[2] for q in rows]
        plt.ylim(min(vals + [1]) - .03, max(vals + [1]) + .03)
        plt.ylabel("hyp/ref length (spBLEU tokens)"); plt.title("Output length ratio on validation")
        save_fig("V_fig_output_length_ratio")


# ==============================================================================
# 17. PART 1 DRIVER
# ==============================================================================
apply_paper_style()
restore_previous_results()
jdump(CFG, f"{R}/config_used.json")
print("[env]", environment_info())
bleu_metric()
DATA_SUMMARY = stage_data_prep()
DRIFT = env_drift_diagnostic()            # E2 (once, cached; must run before superseding)
SUPERSEDED = supersede_other_env_runs()   # E3
try:
    dataset_figures()
except Exception as _e:
    print(f"[figures] dataset_figures failed: {_e}")

# "auto": run Part 1 training only while its 3 items (zero-shot, baseline_s42, pscab_s42) are incomplete.
# In later sessions this cell then just loads the library + restored results (fast) before Part 2/3.
RUN_PART1 = "auto"
_part1_items = ["zeroshot_mbart50"] + build_main_queue()[:2]
if RUN_PART1 is True or (RUN_PART1 == "auto" and not all(run_done(x) for x in _part1_items)):
    execute_zeroshot()
    for _rid in build_main_queue()[:2]:          # baseline_s42, pscab_s42
        if execute_main_run(_rid) is None:
            break
    progress_report()
    print("\nPART 1 finished. Save the version; next session attach its output as input, run Part 1 then Part 2.")
else:
    print("[part1] core runs already complete -- library loaded, results restored. Now run Part 2 or Part 3.")

[restore] 2 previous output(s) found; newest = /kaggle/input/notebooks/panjanagalaxmi/notebook80ed48e07f/pscab_q1 (2026-10-12 06:18:51)
[env] {'python': '3.13.15', 'torch': '2.11.0+cpu', 'cuda': None, 'gpu': None, 'sacrebleu': '2.6.0', 'spacy': True, 'env_pin': {'transformers': '5.16.1', 'sacrebleu': '2.6.0'}, 'transformers': '5.16.1'}
[metrics] BLEU tokenizer = flores200
[data] frozen split found -- reusing it.
{
  "train-val": {
    "exact_en": 0,
    "exact_te": 0,
    "exact_pair": 0,
    "near_dup_en": 0,
    "near_dup_te": 0
  },
  "train-test": {
    "exact_en": 0,
    "exact_te": 0,
    "exact_pair": 0,
    "near_dup_en": 0,
    "near_dup_te": 0
  },
  "val-test": {
    "exact_en": 0,
    "exact_te": 0,
    "exact_pair": 0,
    "near_dup_en": 0,
    "near_dup_te": 0
  },
  "status": "PASS",
  "method": "exact normalized match (EN, TE, pair) + MinHash-LSH char-5-gram Jaccard >= 0.8 on EN and TE separately (128 perms, independent seed), every candidate verified by exact Jaccard, 

In [10]:
# ---- SKIP the runs whose outputs are missing / not worth retraining ----
SKIP_RUNS = {
    "pscab_no_gate_s123",  "pscab_no_gamma_s123",
    "pscab_no_gate_s2024", "pscab_no_gamma_s2024",
}

_orig_build_main_queue = build_main_queue
def build_main_queue():
    return [r for r in _orig_build_main_queue() if r not in SKIP_RUNS]

q = build_main_queue()
print(f"[skip] queue now {len(q)} runs; complete: {sum(run_done(r) for r in q)}/{len(q)}")
print("[skip] still missing:", [r for r in q if not run_done(r)])

[skip] queue now 14 runs; complete: 14/14
[skip] still missing: []


In [ ]:
"""
================================================================================
PART 3 / 3 (v3) -- GROUPED K-FOLD CROSS-VALIDATION + FINAL TEST EVALUATION & REPORT
================================================================================
v3 CHANGES:
  V3-1  CV records store env + generation config; the final evaluation refuses to mix
        transformers versions (all runs must come from the pinned environment of Part 1).
  V3-2  PAPER FIGURES FIG01-FIG17 (architecture, main results, effect forest, per-seed deltas,
        H2, per-preposition heatmap, density/length, sentence-level win/tie/loss, output
        length, metric agreement, gates, module magnitude, attention + robustness, CV,
        errors + cost, learning curves, external sets), a qualitative-examples table
        (LaTeX/Markdown/CSV) and PAPER_ASSETS.md with a draft caption for every figure.
  V3-3  PAPER_FIGURES_ONLY: regenerate all figures from saved results without re-scoring.

Run the Part 1 cell first in the session (with completed runs restored it does not retrain).
Each execution of this cell:
  1. trains up to 2 pending cross-validation runs (baseline / PS-CAB per fold),
  2. prints the CV analysis for the folds completed so far,
  3. if the main queue (Parts 1-2) is complete, performs the FINAL held-out test evaluation.

Cross-validation design: pool = train + validation (test split never used), K group-folds
(near-duplicate groups never straddle folds). Fold f: held-out = fold f, early-stopping set =
2,000 sentences from the other folds, training = the rest. Same step budget and seed.
The folds are five partitions of ONE dataset -- report them as "five-fold grouped
cross-validation", not as five independent experiments.

What changed vs the previous Part 3 (reporting / statistics only; no training change):
  P3-1  SD / CI shown as "N/A" (never 0) when a condition has a single seed.
  P3-2  FORCE_FINAL output is stamped PRELIMINARY and written to a different file; the final
        summary is only written when the whole main queue is complete.
  P3-3  External benchmarks reported twice: contamination-FILTERED (what Part 1 froze, with the
        number removed) and OFFICIAL FULL sets (scored with the saved baseline_s42 / pscab_s42).
  P3-4  Hierarchical bootstrap relabelled "seed-aware uncertainty characterisation"; new per-seed
        consistency table (deltas per seed, #seeds better, paired t over seeds) is the primary
        seed-level evidence; Holm-ADJUSTED p added to the per-seed paired tests.
  P3-5  CV wording/notes corrected (folds are not independent replications).
  P3-6  Human-evaluation agreement (Cohen's kappa) computed once two annotated sheets exist.
  P3-7  COMET can be switched off (USE_COMET) and is never installed during CV-only sessions;
        the evidence report states dataset-selection rationale and the pretraining-contamination limit.
================================================================================
"""
assert "run_next_main_runs" in globals(), "Run the Part 1 cell first in this session."

CV_RUNS_THIS_SESSION = 2
DO_CV = True             # set False to skip CV training this session
FORCE_FINAL = False      # keep False for the paper. True = preliminary look only (stamped PRELIMINARY)
USE_COMET = True         # False = skip COMET (score saved prediction CSVs separately later)
RUN_FULL_EXTERNAL = True                       # score OFFICIAL full external sets with the saved models
PAPER_FIGURES = True     # build FIG01-FIG17 + qualitative table + PAPER_ASSETS.md after the final evaluation
FULL_EXTERNAL_FOR = ["baseline_s42", "pscab_s42"]   # model bundles Part 1 saved to disk

from scipy import stats as sps


# ==============================================================================
# 0. SMALL HELPERS
# ==============================================================================
def holm_adjust(pvals: dict):
    items = sorted(pvals.items(), key=lambda kv: kv[1])
    m, out, running = len(items), {}, 0.0
    for r, (k, p) in enumerate(items):
        running = max(running, min(1.0, (m - r) * p))
        out[k] = running
    return out


def na_df(df):
    """Display/save copy: NaN -> 'N/A' (SD with a single observation is undefined, not zero)."""
    return df.astype(object).where(df.notna(), "N/A")


# ==============================================================================
# A. CROSS-VALIDATION
# ==============================================================================
def build_cv_folds():
    p = f"{DIRS['cv']}/cv_folds.csv"
    if os.path.exists(p):
        return read_csv_str(p)
    train, val, _ = load_splits()
    pool = pd.concat([train, val], ignore_index=True)
    k = CFG["cv"]["k"]
    sizes = pool.groupby("group_id").size()
    order = list(sizes.index)
    np.random.default_rng(CFG["data"]["split_seed"] + 1).shuffle(order)
    fold_size, assign = [0] * k, {}
    for gname in order:                                   # greedy: next group -> smallest fold
        f = int(np.argmin(fold_size))
        assign[gname] = f
        fold_size[f] += int(sizes[gname])
    pool["fold"] = pool["group_id"].map(assign).astype(int)
    pool.to_csv(p, index=False)
    print(f"[cv] built {k} group-folds over {len(pool)} pairs: sizes={fold_size}")
    return read_csv_str(p)


def cv_queue():
    k, seed = CFG["cv"]["k"], CFG["cv"]["seed"]
    return [f"cv{f}_{m}_s{seed}" for f in range(k) for m in CFG["cv"]["models"]]


def cv_fold_data(fold):
    pool = build_cv_folds()
    pool["fold"] = pool["fold"].astype(int)
    held = pool[pool["fold"] == fold].reset_index(drop=True)
    rest = pool[pool["fold"] != fold].reset_index(drop=True)
    groups = rest["group_id"].unique().tolist()
    np.random.default_rng(1000 + fold).shuffle(groups)
    chosen, n = set(), 0
    gsize = rest.groupby("group_id").size()
    for gname in groups:
        if n >= CFG["cv"]["inner_val_size"]:
            break
        chosen.add(gname); n += int(gsize[gname])
    inner_val = rest[rest["group_id"].isin(chosen)].reset_index(drop=True)
    tr = rest[~rest["group_id"].isin(chosen)].reset_index(drop=True)
    assert not (set(tr["group_id"]) & set(held["group_id"])) and not (set(inner_val["group_id"]) & set(held["group_id"]))
    assert len(tr) > 0, "CV training set is empty -- reduce CFG['cv']['inner_val_size']."
    return tr, inner_val, held


def execute_cv_run(run_id):
    if run_done(run_id):
        print(f"[skip] {run_id} already complete.")
        return jload(run_record_path(run_id))
    need = estimated_run_hours()
    if session_hours_left() < need:
        print(f"[time] {run_id} needs ~{need:.1f}h, {session_hours_left():.1f}h left -> not started.")
        return None
    cond, seed, fold = parse_run_id(run_id)
    t0 = time.time()
    tr, iv, held = cv_fold_data(fold)
    print(f"[cv] {run_id}: train={len(tr)} early-stop={len(iv)} held-out={len(held)}")
    model, tok, meta, _ = train_model(cond, seed, tr, iv, run_id)
    gen_rec = generation_record(model, tok)
    hp, info = generate(model, tok, held, CONDITIONS[cond]["mask"])
    save_preds(run_id, "heldout", held, hp)
    rec = {"run_id": run_id, "kind": "cv", "condition": cond, "seed": seed, "fold": fold, "status": "complete",
           "n_train": len(tr), "n_early_stop": len(iv), "n_heldout": len(held),
           "heldout_metrics": compute_metrics(hp, held["te"].tolist()), "env": environment_info(),
           "generation": gen_rec, **meta}
    rec["val_metrics"] = rec["heldout_metrics"]          # for the generic progress table
    del model
    free_gpu()
    rec["wall_hours_total"] = (time.time() - t0) / 3600
    jdump(rec, run_record_path(run_id))
    print(f"[cv] {run_id} held-out {rec['heldout_metrics']['bleu']} BLEU / {rec['heldout_metrics']['chrf++']} chrF++")
    return rec


def run_next_cv_runs(max_runs=2):
    q = cv_queue()
    pending = [r for r in q if not run_done(r)]
    print(f"[cv-queue] {len(q) - len(pending)}/{len(q)} complete; next: {pending[:max_runs]}")
    started = 0
    for rid in pending:
        if started >= max_runs:
            break
        if execute_cv_run(rid) is None:
            break
        started += 1


def cv_report():
    print(f"\n{'#'*72}\nCROSS-VALIDATION REPORT (five-fold grouped CV; held-out folds; test split untouched)\n{'#'*72}")
    k, seed = CFG["cv"]["k"], CFG["cv"]["seed"]
    rows, pairs = [], []
    for f in range(k):
        b, p = f"cv{f}_baseline_s{seed}", f"cv{f}_pscab_s{seed}"
        for rid in (b, p):
            if run_done(rid):
                r = jload(run_record_path(rid))
                rows.append({"fold": f, "model": r["condition"], "n_heldout": r["n_heldout"], "n_train": r["n_train"],
                             "best_step": r["best_step"], "BLEU": r["heldout_metrics"]["bleu"],
                             "chrF++": r["heldout_metrics"]["chrf++"], "TER": r["heldout_metrics"]["ter"]})
        if run_done(b) and run_done(p):
            pairs.append(f)
    if not rows:
        print("no CV run completed yet.")
        return None
    df = pd.DataFrame(rows)
    print(df.to_string(index=False))
    save_table(df, "F14a_cv_per_fold")
    if len(pairs) < 2:
        print(f"[cv] {len(pairs)} complete fold pair(s); statistics need >= 2.")
        return df
    out = []
    for m in ("BLEU", "chrF++"):
        b = df[df["model"] == "baseline"].set_index("fold").loc[pairs, m].values
        p = df[df["model"] == "pscab"].set_index("fold").loc[pairs, m].values
        d = p - b
        n_te = df[df["fold"].isin(pairs)]["n_heldout"].mean()
        n_tr = df[df["fold"].isin(pairs)]["n_train"].mean()
        kk = len(d)
        var = np.var(d, ddof=1) if kk > 1 else np.nan
        t_nb = (d.mean() / math.sqrt((1 / kk + n_te / n_tr) * var)
                if (n_tr and n_tr > 0 and np.isfinite(var) and var > 0) else np.nan)
        p_nb = 2 * sps.t.sf(abs(t_nb), kk - 1) if np.isfinite(t_nb) else np.nan
        try:
            p_w = sps.wilcoxon(d).pvalue if kk >= 2 and np.any(d != 0) else np.nan
        except Exception:
            p_w = np.nan
        out.append({"metric": m, "folds": kk, "baseline_mean": b.mean(), "baseline_sd": b.std(ddof=1),
                    "pscab_mean": p.mean(), "pscab_sd": p.std(ddof=1), "mean_delta": d.mean(), "sd_delta": d.std(ddof=1),
                    "folds_pscab_better": int((d > 0).sum()), "p_corrected_resampled_t": p_nb, "p_wilcoxon": p_w})
    # pooled out-of-fold approximate randomization
    pa, pb, rr = [], [], []
    for f in pairs:
        A = read_csv_str(f"{DIRS['preds']}/cv{f}_pscab_s{seed}__heldout.csv")
        B = read_csv_str(f"{DIRS['preds']}/cv{f}_baseline_s{seed}__heldout.csv")
        pa += A["prediction"].tolist(); pb += B["prediction"].tolist(); rr += A["te"].tolist()
    for mn in ("bleu", "chrf++"):
        M = StatMetric(mn)
        res = paired_test(M.stats(pa, rr), M.stats(pb, rr), M, n_boot=1000, n_ar=5000)
        for o in out:
            if o["metric"].lower().startswith(mn[:4]):
                o.update({"pooled_oof_delta": res["delta"], "pooled_oof_ci": f"[{res['ci95_low']}, {res['ci95_high']}]",
                          "pooled_oof_p_ar": res["p_ar"], "pooled_n": res["n"]})
    s = pd.DataFrame(out)
    print("\n" + s.to_string(index=False))
    print("[cv note] These are the folds of ONE dataset with one training seed, not independent experiments: "
          "write 'five-fold grouped cross-validation'. Wilcoxon with k=5 cannot go below p=0.0625 (two-sided); "
          "rely on the corrected resampled t-test and the pooled out-of-fold AR, and report per-fold values + mean ± SD.")
    save_table(s, "F14b_cv_summary", "Five-fold grouped cross-validation (PS-CAB minus baseline)")
    plt.figure(figsize=(6, 4))
    for f in pairs:
        b = df[(df.fold == f) & (df.model == "baseline")]["chrF++"].values[0]
        p = df[(df.fold == f) & (df.model == "pscab")]["chrF++"].values[0]
        plt.plot([0, 1], [b, p], marker="o", color="tab:green" if p > b else "tab:red")
    plt.xticks([0, 1], ["baseline", "PS-CAB"]); plt.ylabel("held-out chrF++"); plt.title(f"{len(pairs)}-fold CV (paired by fold)")
    save_fig("F_fig_cv_paired")
    return s


# ==============================================================================
# B. COMET (installed without touching torch / numpy; only called from the final evaluation)
# ==============================================================================
_COMET = None


def get_comet():
    global _COMET
    if _COMET is not None:
        return _COMET
    _COMET = False
    if not USE_COMET:
        print("[comet] disabled (USE_COMET=False).")
        return _COMET
    tv = torch.__version__
    if importlib.util.find_spec("comet") is None:
        print("[comet] installing unbabel-comet (--no-deps, keeps torch/numpy unchanged) ...")
        subprocess.run([_sys.executable, "-m", "pip", "install", "-q", "--no-deps", "unbabel-comet", "entmax", "jsonargparse"])
        for pkg, mod in (("pytorch-lightning", "pytorch_lightning"), ("torchmetrics", "torchmetrics")):
            if importlib.util.find_spec(mod) is None:
                subprocess.run([_sys.executable, "-m", "pip", "install", "-q", pkg])
        importlib.invalidate_caches()
    try:
        from comet import download_model, load_from_checkpoint
        _COMET = load_from_checkpoint(download_model("Unbabel/wmt22-comet-da"))
        print(f"[comet] Unbabel/wmt22-comet-da loaded (torch {tv})")
    except Exception as e:
        print(f"[comet] unavailable ({e}). COMET reported as null. Score the saved prediction CSVs "
              f"in a separate notebook with `comet-score`; no training run needs to be repeated.")
    return _COMET


def comet_segments(run_id, split_name, df_src):
    path = f"{DIRS['analysis']}/comet__{run_id}__{split_name}.npy"
    if os.path.exists(path):
        return np.load(path)
    cm = get_comet()
    if not cm:
        return None
    preds = load_preds(run_id, split_name)
    data = [{"src": s, "mt": h, "ref": r} for s, h, r in zip(df_src["en"], preds, df_src["te"])]
    out = cm.predict(data, batch_size=64, gpus=1 if torch.cuda.is_available() else 0, progress_bar=False)
    arr = np.asarray(out.scores, dtype=np.float64)
    np.save(path, arr)
    return arr


# ==============================================================================
# C. SEED-LEVEL AND SUBSET STATISTICS
# ==============================================================================
def hier_bootstrap(A_list, B_list, metric, N, n=2000, seed=7):
    """Paired hierarchical bootstrap over seeds AND sentences (mean over seeds of A_s - B_s).
    With only K=3 seeds this is a seed-aware uncertainty characterisation, NOT 3 independent
    replications multiplied by N sentences."""
    rng = np.random.default_rng(seed)
    K = len(A_list)
    obs = np.mean([metric.score(a.sum(0), N) - metric.score(b.sum(0), N) for a, b in zip(A_list, B_list)])
    d = np.empty(n)
    for i in range(n):
        ks = rng.integers(0, K, K)
        idx = rng.integers(0, N, N)
        d[i] = np.mean([metric.score(A_list[k][idx].sum(0), N) - metric.score(B_list[k][idx].sum(0), N) for k in ks])
    lo, hi = np.percentile(d, [2.5, 97.5])
    p = min(1.0, 2 * min((d <= 0).mean(), (d >= 0).mean()))
    return {"delta_mean_over_seeds": round(float(obs), 4), "ci95_low": round(float(lo), 4),
            "ci95_high": round(float(hi), 4), "bootstrap_sign_p": round(max(p, 1 / n), 5), "n_seeds": K}


def seed_consistency(A_list, B_list, metric, N):
    """Primary seed-level evidence: per-seed deltas (corpus-level, same seed paired), mean, SD,
    number of seeds where A is better, and a paired t-test over seeds (low power with 3 seeds)."""
    d = np.array([metric.score(a.sum(0), N) - metric.score(b.sum(0), N) for a, b in zip(A_list, B_list)])
    k = len(d)
    sd = float(d.std(ddof=1)) if k > 1 else np.nan
    try:
        p_t = float(sps.ttest_1samp(d, 0.0).pvalue) if k >= 2 and sd > 0 else np.nan
    except Exception:
        p_t = np.nan
    return {"n_seeds": k, "seed_deltas": [round(float(x), 3) for x in d], "mean_delta": round(float(d.mean()), 4),
            "sd_delta": round(sd, 4) if np.isfinite(sd) else np.nan, "n_seeds_pscab_better": int((d > 0).sum()),
            "p_paired_t_over_seeds": round(p_t, 4) if np.isfinite(p_t) else np.nan}


def did_bootstrap(A_list, B_list, metric, prep_idx, nop_idx, n=2000, seed=11):
    """
    H2 difference-in-differences with hierarchical bootstrap over
    BOTH seeds and sentences.

    DiD =
        [PS-CAB - baseline on preposition sentences]
        -
        [PS-CAB - baseline on no-preposition sentences]

    Bootstrap resamples:
      1. seeds with replacement
      2. sentences within each selected seed
    """
    rng = np.random.default_rng(seed)

    K = len(A_list)

    def did_for_indices(seed_ids, prep_samples, noprep_samples):
        prep_deltas = []
        noprep_deltas = []

        for k, ip, iq in zip(seed_ids, prep_samples, noprep_samples):
            dp = (
                metric.score(A_list[k][ip].sum(0), len(ip))
                - metric.score(B_list[k][ip].sum(0), len(ip))
            )

            dn = (
                metric.score(A_list[k][iq].sum(0), len(iq))
                - metric.score(B_list[k][iq].sum(0), len(iq))
            )

            prep_deltas.append(dp)
            noprep_deltas.append(dn)

        return float(np.mean(prep_deltas)), float(np.mean(noprep_deltas))

    # Observed effect
    observed_seed_ids = list(range(K))
    observed_prep = [prep_idx] * K
    observed_noprep = [nop_idx] * K

    dp, dn = did_for_indices(
        observed_seed_ids,
        observed_prep,
        observed_noprep
    )

    observed_did = dp - dn

    # Hierarchical bootstrap over seeds AND sentences
    boots = np.empty(n, dtype=np.float64)

    for i in range(n):

        # Resample seeds with replacement
        seed_ids = rng.integers(0, K, size=K)

        prep_samples = []
        noprep_samples = []

        for k in seed_ids:
            prep_samples.append(
                rng.choice(prep_idx, size=len(prep_idx), replace=True)
            )

            noprep_samples.append(
                rng.choice(nop_idx, size=len(nop_idx), replace=True)
            )

        bp, bn = did_for_indices(
            seed_ids,
            prep_samples,
            noprep_samples
        )

        boots[i] = bp - bn

    lo, hi = np.percentile(boots, [2.5, 97.5])

    p = min(
        1.0,
        2 * min(
            float((boots <= 0).mean()),
            float((boots >= 0).mean())
        )
    )

    return {
        "delta_prep": round(float(dp), 4),
        "delta_noprep": round(float(dn), 4),
        "DiD": round(float(observed_did), 4),
        "DiD_ci95_low": round(float(lo), 4),
        "DiD_ci95_high": round(float(hi), 4),
        "bootstrap_sign_p": round(max(p, 1 / n), 5),
        "n_seeds": K
    }

# ==============================================================================
# D. EXTERNAL BENCHMARKS: contamination-filtered (frozen in Part 1) AND official full sets
# ==============================================================================
def raw_external_sets():
    """Official, UNFILTERED external sets (normalised only). Needs HF access for FLORES+ / your CSVs."""
    sets = {}
    if CFG["external"]["try_flores_plus"]:
        f = prepare_flores_plus(None)
        if f is not None:
            sets["flores_plus_devtest"] = f
    for name, p in CFG["external"]["csvs"].items():
        if os.path.exists(p):
            sets[name] = read_csv_str(p)[["en", "te"]]
    out = {}
    for name, d in sets.items():
        d = d.copy()
        d["en"], d["te"] = d["en"].apply(normalize_text), d["te"].apply(normalize_text)
        out[name] = d.reset_index(drop=True)
    return out


def _load_saved_model(run_id):
    cond, _, _ = parse_run_id(run_id)
    d = f"{DIRS['models']}/{run_id}"
    model = PSCABMBart(cond)
    model.backbone = MBartForConditionalGeneration.from_pretrained(d).float()
    if model.use_pscab:
        sd = torch.load(f"{d}/pscab_module_fp16.pt", map_location="cpu")
        model.pscab.load_state_dict({k: v.float() for k, v in sd.items()})
    tok = MBart50TokenizerFast.from_pretrained(d)
    tok.src_lang, tok.tgt_lang = CFG["model"]["src_lang"], CFG["model"]["tgt_lang"]
    return model.to(DEVICE).eval(), tok


def external_full_report(comet_ok):
    """Scores the OFFICIAL full external sets with the saved models (full sets were not generated in
    Part 1, which only decoded the contamination-filtered copies)."""
    print(f"\n{'#'*72}\nEXTERNAL BENCHMARKS -- OFFICIAL FULL SETS (no filtering)\n{'#'*72}")
    if not RUN_FULL_EXTERNAL:
        print("[external-full] disabled.")
        return None
    raw = raw_external_sets()
    if not raw:
        print("[external-full] no official external set available (FLORES+ needs Internet + HF_TOKEN with access; "
              "or add CSVs to CFG['external']['csvs']). Nothing scored.")
        return None
    ids = [r for r in FULL_EXTERNAL_FOR if os.path.isdir(f"{DIRS['models']}/{r}")]
    if not ids:
        print("[external-full] saved model bundles not found; nothing scored.")
        return None
    for rid in ids:
        cond, _, _ = parse_run_id(rid)
        todo = [n for n in raw if load_preds(rid, f"extfull_{n}") is None]
        if not todo:
            continue
        model, tok = _load_saved_model(rid)
        for n in todo:
            p, _ = generate(model, tok, raw[n], CONDITIONS[cond]["mask"])
            save_preds(rid, f"extfull_{n}", raw[n], p)
            print(f"[external-full] {rid} on {n}: {len(raw[n])} sentences decoded")
        del model
        free_gpu()
    rows = []
    for n, d in raw.items():
        er = d["te"].tolist()
        for rid in ids:
            p = load_preds(rid, f"extfull_{n}")
            if p is None:
                continue
            m = compute_metrics(p, er)
            row = {"set": n, "version": "official-full", "n": len(er), "run": rid,
                   "BLEU": m["bleu"], "chrF++": m["chrf++"], "TER": m["ter"]}
            if comet_ok:
                cs = comet_segments(rid, f"extfull_{n}", d)
                row["COMET"] = round(float(np.mean(cs)) * 100, 2) if cs is not None else None
            rows.append(row)
        if all(load_preds(r, f"extfull_{n}") for r in ("baseline_s42", "pscab_s42")):
            M = StatMetric("chrf++")
            res = paired_test(M.stats(load_preds("pscab_s42", f"extfull_{n}"), er),
                              M.stats(load_preds("baseline_s42", f"extfull_{n}"), er), M)
            print(f"[external-full] {n}: chrF++ pscab_s42 - baseline_s42 = {res}")
    if rows:
        t = pd.DataFrame(rows)
        print(t.to_string(index=False))
        save_table(t, "F13b_external_official_full", "External benchmarks, official full sets (no filtering)")
        return t
    return None


# ==============================================================================
# E. FINAL TEST EVALUATION
# ==============================================================================
def final_test_evaluation():
    print(f"\n{'#'*72}\nFINAL HELD-OUT TEST EVALUATION\n{'#'*72}")
    q = build_main_queue()
    missing = [r for r in q if not run_done(r)]
    if missing and not FORCE_FINAL:
        print(f"[final] main queue incomplete ({len(missing)} missing: {missing[:4]}...). "
              f"Finish Part 2 first. Do NOT set FORCE_FINAL=True for the paper: looking at test results "
              f"while experimental decisions are still open invalidates the held-out test.")
        return
    prelim = bool(missing)
    if prelim:
        print("!" * 72 + f"\n[final] PRELIMINARY MODE: {len(missing)} main runs missing. These numbers are NOT for the paper.\n" + "!" * 72)
    train, val, test = load_splits()
    refs = test["te"].tolist()
    N = len(refs)
    recs = {r["run_id"]: r for r in all_records() if r["kind"] in ("main", "zeroshot")}
    runs = [rid for rid in recs if load_preds(rid, "test") is not None]
    if not runs:
        print("[final] no test predictions found.")
        return
    vers = {str(rec_tf_version(recs[r])) for r in runs}
    if len(vers) > 1 or vers != {ENV_PIN["transformers"]}:
        print(f"[final] transformers versions among the runs: {vers} (pinned {ENV_PIN['transformers']}).")
        if not FORCE_FINAL:
            print("[final] refusing to compare runs decoded under different environments. Run Part 1 (it supersedes "
                  "and re-queues them) and finish the re-runs first.")
            return
    preds = {rid: load_preds(rid, "test") for rid in runs}
    seeds = CFG["experiment"]["seeds"]
    f7 = f8 = f11 = pd.DataFrame()

    # ---- F1 dataset table
    ds = jload(f"{DIRS['data']}/data_summary.json")
    f1 = pd.DataFrame([{"original": ds["cleaning"]["original_rows"], "after_cleaning": ds["cleaning"]["rows_after_cleaning"],
                        "after_exact_dedup": ds["cleaning"]["rows_after_exact_dedup"], "selected": ds["selected_pairs"],
                        **{f"n_{k}": v for k, v in ds["split_sizes"].items()}, "near_dup_groups": ds["n_groups"],
                        "leakage_audit": ds["leakage_audit"]["status"]}])
    print("\n[F1] Data\n" + f1.to_string(index=False))
    save_table(f1, "F1_dataset")

    # ---- F2 per-run test metrics (+COMET)
    comet_ok = USE_COMET and (get_comet() is not False)
    seg = {}
    rows = []
    for rid in runs:
        m = compute_metrics(preds[rid], refs)
        row = {"run": rid, "condition": recs[rid]["condition"], "seed": recs[rid].get("seed"),
               "BLEU": m["bleu"], "chrF++": m["chrf++"], "TER": m["ter"]}
        if comet_ok:
            seg[rid] = comet_segments(rid, "test", test)
            row["COMET"] = round(float(np.mean(seg[rid])) * 100, 2) if seg[rid] is not None else None
        rows.append(row)
    f2 = pd.DataFrame(rows)
    f2["seed"] = f2["seed"].astype("Int64")
    print("\n[F2] Test results per run\n" + f2.to_string(index=False))
    sig = compute_metrics(preds[runs[0]], refs)
    print(f"BLEU signature: {sig['bleu_signature']}\nchrF signature: {sig['chrf_signature']}")
    save_table(f2, "F2_test_per_run")

    # ---- F3 condition mean +- SD, 95% t-CI over seeds (SD / CI = N/A for a single seed)
    metrics_cols = [c for c in ("BLEU", "chrF++", "TER", "COMET") if c in f2.columns]
    rows = []
    for cond in ["zeroshot_mbart50"] + list(CONDITIONS):
        sub = f2[f2["condition"] == cond]
        if not len(sub):
            continue
        row = {"condition": cond, "n_seeds": len(sub)}
        for c in metrics_cols:
            v = sub[c].dropna().astype(float).values
            if not len(v):
                continue
            sd = v.std(ddof=1) if len(v) > 1 else np.nan
            half = sps.t.ppf(0.975, len(v) - 1) * sd / math.sqrt(len(v)) if len(v) > 1 else np.nan
            row[f"{c}_mean"] = round(v.mean(), 2)
            row[f"{c}_sd"] = round(sd, 2) if np.isfinite(sd) else np.nan
            row[f"{c}_ci95"] = round(half, 2) if np.isfinite(half) else np.nan
        rows.append(row)
    f3 = pd.DataFrame(rows)
    print("\n[F3] Test, mean ± SD (95% t-CI half-width) over seeds; N/A = single seed\n" + na_df(f3).to_string(index=False))
    save_table(na_df(f3), "F3_test_by_condition", "Test results, mean ± SD over seeds (N/A = single seed)")

    # ---- F4 per-seed paired tests (AR p + bootstrap CI), Holm thresholds + Holm-ADJUSTED p per metric
    mets = ["bleu", "chrf++"] + (["comet"] if comet_ok and all(seg.get(r) is not None for r in runs) else [])
    comps = []
    for s in seeds:
        if f"pscab_s{s}" in preds and f"baseline_s{s}" in preds:
            comps.append((f"pscab_s{s}", f"baseline_s{s}"))
        for a in ABLATIONS:
            if f"pscab_s{s}" in preds and f"{a}_s{s}" in preds:
                comps.append((f"pscab_s{s}", f"{a}_s{s}"))
    rows, stat_cache = [], {}
    for mn in mets:
        M = StatMetric(mn)
        stat_cache[mn] = {rid: (M.stats(preds[rid], refs, seg.get(rid)) if mn != "comet" else M.stats(None, None, seg[rid]))
                          for rid in runs}
        pv = {}
        for a, b in comps:
            res = paired_test(stat_cache[mn][a], stat_cache[mn][b], M)
            rows.append({"metric": mn, "A": a, "B": b, **res})
            pv[f"{mn}|{a}|{b}"] = res["p_ar"]
        h = holm(pv)
        adj = holm_adjust(pv) if pv else {}
        for r in rows:
            k = f"{r['metric']}|{r['A']}|{r['B']}"
            if k in h:
                r.update(h[k])
                r["p_holm_adjusted"] = round(adj[k], 5)
    f4 = pd.DataFrame(rows)
    print("\n[F4] Paired tests on TEST (A - B, same seed): AR p (10k), bootstrap 95% CI (2k), Holm per metric\n" + f4.to_string(index=False))
    save_table(f4, "F4_test_paired_tests")

    # ---- F5 seed-level evidence: (a) per-seed consistency [primary], (b) hierarchical bootstrap [characterisation]
    rows5b, rows5 = [], []
    for mn in mets:
        M = StatMetric(mn)
        for other in ["baseline"] + ABLATIONS:
            ss_ = [s for s in seeds if f"pscab_s{s}" in preds and f"{other}_s{s}" in preds]
            if not ss_:
                continue
            A = [stat_cache[mn][f"pscab_s{s}"] for s in ss_]
            B = [stat_cache[mn][f"{other}_s{s}"] for s in ss_]
            rows5b.append({"metric": mn, "comparison": f"pscab - {other}", **seed_consistency(A, B, M, N)})
            rows5.append({"metric": mn, "comparison": f"pscab - {other}", **hier_bootstrap(A, B, M, N)})
    f5b, f5 = pd.DataFrame(rows5b), pd.DataFrame(rows5)
    print("\n[F5a] PRIMARY seed-level evidence: per-seed deltas (same-seed pairing), mean, SD, #seeds better, "
          "paired t over seeds (3 seeds = very low power)\n" + na_df(f5b).to_string(index=False))
    save_table(na_df(f5b.assign(seed_deltas=f5b["seed_deltas"].astype(str))) if len(f5b) else f5b,
               "F5a_test_seed_consistency", "PS-CAB effect per seed")
    print("\n[F5b] Seed-aware uncertainty characterisation (hierarchical bootstrap over seeds and sentences). "
          "With only K seeds this is NOT K independent replications x N sentences; do not present it as "
          "population-level proof.\n" + f5.to_string(index=False))
    save_table(f5, "F5b_test_hierarchical_bootstrap", "Seed-aware uncertainty characterisation (hierarchical bootstrap)")

    # ---- F6 H2: preposition subsets + difference-in-differences
    ensure_spans(test["en"].tolist())
    counts = np.array([len(_SPAN_CACHE[s]) for s in test["en"]])
    prep_idx, nop_idx = np.where(counts > 0)[0], np.where(counts == 0)[0]
    rows = []
    ss = [s for s in seeds if f"pscab_s{s}" in preds and f"baseline_s{s}" in preds]
    if ss and len(nop_idx) >= 30:
        for mn in mets:
            M = StatMetric(mn)
            rows.append({"metric": mn, "n_prep": len(prep_idx), "n_noprep": len(nop_idx),
                         **did_bootstrap([stat_cache[mn][f"pscab_s{s}"] for s in ss],
                                         [stat_cache[mn][f"baseline_s{s}"] for s in ss], M, prep_idx, nop_idx)})
    f6 = pd.DataFrame(rows)
    if len(f6):
        print("\n[F6] H2 difference-in-differences (PS-CAB - baseline on prep sentences minus on no-prep sentences)\n"
              + f6.to_string(index=False))
        print("[F6 note] On no-prep sentences the true-mask module is an exact no-op, so delta_noprep estimates "
              "pure training variance; H2 is supported only if the DiD CI excludes 0 and is positive. "
              f"Only {len(nop_idx)} no-prep sentences exist, so H2 has limited power.")
        save_table(f6, "F6_H2_difference_in_differences")

    # ---- F7 preposition density / type, F8 length buckets (mean over seeds of PS-CAB - baseline)
    def subset_delta(mask, mn="chrf++"):
        M = StatMetric(mn)
        v = [M.score(stat_cache[mn][f"pscab_s{s}"][mask].sum(0), mask.sum())
             - M.score(stat_cache[mn][f"baseline_s{s}"][mask].sum(0), mask.sum()) for s in ss]
        return round(float(np.mean(v)), 3), (round(float(np.std(v, ddof=1)), 3) if len(v) > 1 else np.nan)

    rows = []
    if ss:
        bucket = np.array(["0" if c == 0 else "1" if c == 1 else "2" if c == 2 else "3+" for c in counts])
        for b in ["0", "1", "2", "3+"]:
            m = bucket == b
            if m.sum() >= 30:
                d, sd = subset_delta(m)
                rows.append({"group": f"#prep={b}", "n": int(m.sum()), "chrF++_delta_mean": d, "sd_over_seeds": sd})
        words = [[w for _, _, w in _SPAN_CACHE[s]] for s in test["en"]]
        for pw in ["of", "in", "to", "for", "on", "with", "from", "by", "at", "into", "about", "after", "over", "during"]:
            m = np.array([pw in w for w in words])
            if m.sum() >= 30:
                d, sd = subset_delta(m)
                rows.append({"group": f"prep='{pw}'", "n": int(m.sum()), "chrF++_delta_mean": d, "sd_over_seeds": sd})
        f7 = pd.DataFrame(rows)
        print("\n[F7] PS-CAB - baseline by preposition density / type (test, chrF++, mean over seeds; N/A = single seed)\n"
              + na_df(f7).to_string(index=False))
        save_table(na_df(f7), "F7_preposition_density_type")
        lens = test["en"].apply(lambda s: len(s.split())).values
        rows = []
        for lo, hi, lab in [(1, 10, "1-10"), (11, 20, "11-20"), (21, 30, "21-30"), (31, 10**6, ">30")]:
            m = (lens >= lo) & (lens <= hi)
            if m.sum() >= 30:
                d, sd = subset_delta(m)
                row = {"length": lab, "n": int(m.sum()), "chrF++_delta_mean": d, "sd_over_seeds": sd}
                for cond in ("baseline", "pscab"):
                    M = StatMetric("chrf++")
                    row[f"{cond}_chrF++"] = round(np.mean([M.score(stat_cache["chrf++"][f"{cond}_s{s}"][m].sum(0), m.sum()) for s in ss]), 2)
                rows.append(row)
        f8 = pd.DataFrame(rows)
        print("\n[F8] By source length (test; N/A = single seed)\n" + na_df(f8).to_string(index=False))
        save_table(na_df(f8), "F8_length_buckets")

    # ---- F9 error heuristics (automatic flags, NOT a linguistic taxonomy)
    rows = []
    for rid in runs:
        p = preds[rid]
        ratio = np.array([len(h) / max(len(r), 1) for h, r in zip(p, refs)])
        rows.append({"run": rid,
                     "untranslated_latin_words_%": 100 * np.mean([bool(LATIN_WORD_RE.search(h)) for h in p]),
                     "repeated_trigram_%": 100 * np.mean([_has_repeat(h) for h in p]),
                     "empty_%": 100 * np.mean([h.strip() == "" for h in p]),
                     "len_ratio_<0.5_%": 100 * np.mean(ratio < 0.5), "len_ratio_>2_%": 100 * np.mean(ratio > 2)})
    f9 = pd.DataFrame(rows).round(3)
    print("\n[F9] Automatic error flags (test)\n" + f9.to_string(index=False))
    save_table(f9, "F9_error_flags")

    # ---- F10 cost, F11 diagnostics
    rows = []
    for rid, r in recs.items():
        if r["kind"] != "main":
            continue
        rows.append({"run": rid, "total_params": r.get("total_parameters"), "added_params": r.get("pscab_added_parameters"),
                     "steps": r.get("steps_trained"), "best_step": r.get("best_step"), "stop": r.get("stop_reason"),
                     "sec_per_step": round(r.get("sec_per_step", float("nan")), 3), "train_h": round(r.get("train_hours", 0), 2),
                     "infer_sent_per_s": round(r.get("inference_sentences_per_sec", float("nan")), 2)})
    f10 = pd.DataFrame(rows)
    print("\n[F10] Cost (identical step budget; differences in steps come from early stopping only)\n" + f10.to_string(index=False))
    save_table(f10, "F10_cost")
    rows = [{"run": rid, **r["diagnostics"]} for rid, r in recs.items() if r.get("diagnostics")]
    if rows:
        f11 = pd.DataFrame(rows)
        print("\n[F11] Module diagnostics (validation). Attention mass / gate statistics are mechanistic "
              "DIAGNOSTICS of source-token utilisation, not proof of mechanism.\n" + f11.to_string(index=False))
        save_table(f11, "F11_module_diagnostics")

    # ---- F13 external test sets: (a) contamination-FILTERED (frozen in Part 1), (b) OFFICIAL FULL
    ext = external_sets()
    rep = jload(f"{DIRS['data']}/external_report.json") or {}
    if rep:
        print("\n[F13 note] contamination filtering applied in Part 1 (n_raw -> n_used):\n" + json.dumps(rep, indent=2))
    rows = []
    for name, edf in ext.items():
        er = edf["te"].tolist()
        for rid in runs:
            p = load_preds(rid, f"ext_{name}")
            if p is None:
                continue
            m = compute_metrics(p, er)
            row = {"set": name, "version": "contamination-filtered", "n": len(er), "run": rid,
                   "BLEU": m["bleu"], "chrF++": m["chrf++"], "TER": m["ter"]}
            if comet_ok:
                cs = comet_segments(rid, f"ext_{name}", edf)
                row["COMET"] = round(float(np.mean(cs)) * 100, 2) if cs is not None else None
            rows.append(row)
        for s in seeds:
            a, b = load_preds(f"pscab_s{s}", f"ext_{name}"), load_preds(f"baseline_s{s}", f"ext_{name}")
            if a and b:
                M = StatMetric("chrf++")
                res = paired_test(M.stats(a, er), M.stats(b, er), M)
                print(f"[F13] {name} (filtered) seed {s}: chrF++ pscab-baseline {res}")
    if rows:
        f13 = pd.DataFrame(rows)
        print("\n[F13] External test sets -- CONTAMINATION-FILTERED (do not call these the official benchmark score)\n"
              + f13.to_string(index=False))
        save_table(f13, "F13_external_contamination_filtered")
    else:
        print("\n[F13] no filtered external predictions (no external set was frozen in Part 1).")
    external_full_report(comet_ok)

    # ---- Figures
    if len(f3) and "chrF++_mean" in f3:
        plt.figure(figsize=(8, 4))
        x = range(len(f3))
        plt.bar(x, f3["chrF++_mean"], yerr=f3["chrF++_sd"].fillna(0), capsize=4)
        plt.xticks(x, f3["condition"], rotation=25, ha="right")
        plt.ylim(max(0, f3["chrF++_mean"].min() - 5), f3["chrF++_mean"].max() + 2)
        plt.ylabel("test chrF++"); plt.title("Test chrF++ (mean; error bar = SD, none for a single seed)")
        save_fig("F_fig_test_chrf_by_condition")
    if len(f5):
        sub = f5[f5["metric"] == "chrf++"]
        if len(sub):
            plt.figure(figsize=(7, 3.6))
            y = np.arange(len(sub))
            plt.errorbar(sub["delta_mean_over_seeds"], y, xerr=[sub["delta_mean_over_seeds"] - sub["ci95_low"],
                                                                 sub["ci95_high"] - sub["delta_mean_over_seeds"]], fmt="o", capsize=4)
            plt.axvline(0, ls="--", c="gray"); plt.yticks(y, sub["comparison"])
            plt.xlabel("chrF++ difference (seed-aware 95% interval)"); plt.title("Effect sizes on test")
            save_fig("F_fig_forest_chrf")
    if len(f6):
        r = f6[f6["metric"] == "chrf++"].iloc[0]
        plt.figure(figsize=(5, 3.6))
        plt.bar(["with prepositions", "no prepositions\n(module inactive)"], [r["delta_prep"], r["delta_noprep"]])
        plt.axhline(0, c="gray", lw=.8); plt.ylabel("chrF++: PS-CAB - baseline")
        plt.title(f"H2: DiD = {r['DiD']} [{r['DiD_ci95_low']}, {r['DiD_ci95_high']}]")
        save_fig("F_fig_H2_did")

    export_blinded_human_eval(test, counts)
    final_evidence_report(f3, f4, f5, f5b, f6, prelim)
    if PAPER_FIGURES:
        ctx = dict(test=test, refs=refs, N=N, preds=preds, recs=recs, runs=runs, seeds=seeds, stat_cache=stat_cache,
                   seg=seg, mets=mets, comet_ok=comet_ok, counts=counts, f2=f2, f3=f3, f4=f4, f5=f5, f5b=f5b, f6=f6,
                   f7=f7, f8=f8, f9=f9, f10=f10, f11=f11, prelim=prelim)
        paper_figures(ctx)


def _has_repeat(s, n=3):
    t = s.split()
    grams = [tuple(t[i:i + n]) for i in range(len(t) - n + 1)]
    return len(grams) != len(set(grams))


# ==============================================================================
# F. HUMAN EVALUATION: blinded sheet export + agreement once annotated
# ==============================================================================
def export_blinded_human_eval(test, counts, n_prep=150, n_noprep=50):
    a_id, b_id = "baseline_s42", "pscab_s42"
    pa, pb = load_preds(a_id, "test"), load_preds(b_id, "test")
    if not pa or not pb:
        return
    rng = np.random.default_rng(2026)
    pi = np.where(counts > 0)[0]; ni = np.where(counts == 0)[0]
    idx = np.concatenate([rng.choice(pi, min(n_prep, len(pi)), replace=False),
                          rng.choice(ni, min(n_noprep, len(ni)), replace=False)])
    rng.shuffle(idx)
    sheet, key = [], []
    for k, i in enumerate(idx):
        flip = bool(rng.integers(0, 2))
        A, B = (pb[i], pa[i]) if flip else (pa[i], pb[i])
        sheet.append({"item": k, "source_en": test["en"].iloc[i], "reference_te": test["te"].iloc[i],
                      "translation_A": A, "translation_B": B, "adequacy_A_1to5": "", "adequacy_B_1to5": "",
                      "fluency_A_1to5": "", "fluency_B_1to5": "", "preposition_relation_correct_A_yes_no": "",
                      "preposition_relation_correct_B_yes_no": "", "preference_A_B_tie": ""})
        key.append({"item": k, "test_row": int(i), "A": b_id if flip else a_id, "B": a_id if flip else b_id,
                    "has_preposition": bool(counts[i] > 0)})
    pd.DataFrame(sheet).to_csv(f"{DIRS['analysis']}/HUMAN_EVAL_SHEET_blinded.csv", index=False)
    pd.DataFrame(key).to_csv(f"{DIRS['analysis']}/HUMAN_EVAL_KEY_do_not_share.csv", index=False)
    print(f"\n[human-eval] blinded sheet ({len(idx)} items, A/B randomised) + separate key written to {DIRS['analysis']}. "
          f"A blank sheet is NOT evidence: have >= 2 native Telugu annotators fill copies saved as "
          f"HUMAN_EVAL_ANNOTATOR1.csv / HUMAN_EVAL_ANNOTATOR2.csv in that folder, then re-run this cell.")


def human_eval_agreement():
    """Cohen's kappa (quadratic-weighted for 1-5 scores) between two annotators + unblinded preference counts."""
    a_dir = DIRS["analysis"]
    p1, p2 = f"{a_dir}/HUMAN_EVAL_ANNOTATOR1.csv", f"{a_dir}/HUMAN_EVAL_ANNOTATOR2.csv"
    kp = f"{a_dir}/HUMAN_EVAL_KEY_do_not_share.csv"
    if not (os.path.exists(p1) and os.path.exists(p2) and os.path.exists(kp)):
        print("\n[human-eval] annotator sheets not found -> no human-evaluation claim can be made yet.")
        return
    from sklearn.metrics import cohen_kappa_score
    a1, a2, key = read_csv_str(p1), read_csv_str(p2), read_csv_str(kp)
    ordinal = ["adequacy_A_1to5", "adequacy_B_1to5", "fluency_A_1to5", "fluency_B_1to5"]
    nominal = ["preposition_relation_correct_A_yes_no", "preposition_relation_correct_B_yes_no", "preference_A_B_tie"]
    rows = []
    for c in ordinal + nominal:
        x, y = a1[c].str.strip().str.lower(), a2[c].str.strip().str.lower()
        ok = (x != "") & (y != "")
        if ok.sum() < 10:
            continue
        kappa = (cohen_kappa_score(x[ok].astype(int), y[ok].astype(int), weights="quadratic")
                 if c in ordinal else cohen_kappa_score(x[ok], y[ok]))
        rows.append({"item_type": c, "n_rated_by_both": int(ok.sum()), "cohens_kappa": round(float(kappa), 3),
                     "weighting": "quadratic" if c in ordinal else "none"})
    if rows:
        t = pd.DataFrame(rows)
        print("\n[human-eval] inter-annotator agreement\n" + t.to_string(index=False))
        save_table(t, "F15_human_eval_agreement")
    for nm, a in (("annotator1", a1), ("annotator2", a2)):
        pref = a["preference_A_B_tie"].str.strip().str.lower()
        cnt = {"pscab_s42": 0, "baseline_s42": 0, "tie": 0}
        for k_, v in zip(key["item"].astype(int), key["A"]):
            pv = pref.iloc[k_] if k_ < len(pref) else ""
            if pv == "a":
                cnt[v] += 1
            elif pv == "b":
                cnt[key.loc[key["item"].astype(int) == k_, "B"].iloc[0]] += 1
            elif pv == "tie":
                cnt["tie"] += 1
        print(f"[human-eval] {nm} unblinded preference counts: {cnt}")


# ==============================================================================
# G. FINAL EVIDENCE REPORT
# ==============================================================================
def final_evidence_report(f3, f4, f5, f5b, f6, prelim=False):
    L = ["=" * 72,
         ("PRELIMINARY EVIDENCE SUMMARY -- NOT FOR THE PAPER (main queue incomplete)" if prelim else
          "FINAL EVIDENCE SUMMARY (generated from saved results; no claim is auto-asserted)"), "=" * 72]
    ds = jload(f"{DIRS['data']}/data_summary.json")
    L.append(f"Data: {ds['selected_pairs']} pairs, split {ds['split_sizes']}, leakage audit {ds['leakage_audit']['status']} "
             f"({ds['leakage_audit']['method']}).")
    L.append(f"Dataset-selection rationale: the cleaned BPCC corpus had {ds['cleaning']['rows_after_exact_dedup']} unique valid "
             f"pairs; a fixed {ds['selected_pairs']}-pair subset was drawn at random with seed {CFG['data']['select_seed']} before any "
             f"training, to keep baseline, ablations and seeds under one identical, T4-feasible regime. Selection used no "
             f"validation/test score, sentence difficulty or preposition frequency.")
    L.append(f"Protocol: identical max_steps={CFG['optim']['max_steps']} for all conditions, early stopping on validation NLL "
             f"(patience {CFG['optim']['patience_evals']} evals), seeds {CFG['experiment']['seeds']}, beam {CFG['decoding']['num_beams']}.")
    checks = {}
    if len(f5b):
        for mn in f5b["metric"].unique():
            r = f5b[(f5b["metric"] == mn) & (f5b["comparison"] == "pscab - baseline")]
            h = f5[(f5["metric"] == mn) & (f5["comparison"] == "pscab - baseline")] if len(f5) else f5
            if len(r) and len(h):
                r, h = r.iloc[0], h.iloc[0]
                excl = h["ci95_low"] > 0 or h["ci95_high"] < 0
                consistent = int(r["n_seeds_pscab_better"]) == int(r["n_seeds"])
                checks[f"main_{mn}"] = bool(h["ci95_low"] > 0 and consistent)
                L.append(f"PS-CAB - baseline [{mn}]: per-seed deltas {r['seed_deltas']}, mean {r['mean_delta']}, "
                         f"better in {r['n_seeds_pscab_better']}/{r['n_seeds']} seeds; seed-aware 95% interval "
                         f"{h['ci95_low']}..{h['ci95_high']} -> "
                         f"{'interval excludes 0' if excl else 'interval includes 0: no reliable difference'}"
                         f"{'' if consistent else '; NOT consistent across seeds'}")
        for abl in ABLATIONS:
            r = f5b[(f5b["metric"] == "chrf++") & (f5b["comparison"] == f"pscab - {abl}")]
            h = f5[(f5["metric"] == "chrf++") & (f5["comparison"] == f"pscab - {abl}")] if len(f5) else f5
            if len(r) and len(h):
                r, h = r.iloc[0], h.iloc[0]
                checks[f"beats_{abl}"] = bool(h["ci95_low"] > 0 and int(r["n_seeds_pscab_better"]) == int(r["n_seeds"]))
                L.append(f"PS-CAB - {abl} [chrF++]: per-seed {r['seed_deltas']}, mean {r['mean_delta']}, "
                         f"seed-aware interval {h['ci95_low']}..{h['ci95_high']}")
    if len(f6):
        r = f6[f6["metric"] == "chrf++"].iloc[0]
        checks["H2"] = bool(r["DiD_ci95_low"] > 0)
        L.append(f"H2 (DiD, chrF++): {r['DiD']} (CI {r['DiD_ci95_low']}..{r['DiD_ci95_high']}) -> "
                 f"{'supported' if checks['H2'] else 'not supported'}")
    L.append("")
    need = ["main_chrf++", "H2", "beats_pscab_random_mask", "beats_pscab_all_tokens"]
    L.append("Criteria for a preposition-specific claim: " + ", ".join(
        f"{k}={'PASS' if checks.get(k) else ('FAIL' if k in checks else 'MISSING')}" for k in need))
    if all(checks.get(k) for k in need):
        L.append("EVIDENCE PATTERN: consistent with a preposition-specific improvement (main effect > 0 and consistent "
                 "across seeds, H2 > 0, beats random-mask and all-token controls). Still confirm with COMET, CV and human evaluation.")
    elif checks.get("main_chrf++"):
        L.append("EVIDENCE PATTERN: an overall improvement exists but is NOT shown to be preposition-specific "
                 "(see FAIL/MISSING above) -> do not claim the preposition hypothesis; frame accordingly.")
    else:
        L.append("EVIDENCE PATTERN: no reliable improvement over the baseline. Report as a controlled negative result "
                 "(the design, controls and noise-floor analysis are the contribution), do not claim gains.")
    L += ["", "Paper checklist:",
          "  - report sacreBLEU signatures (printed in F2), COMET model name, seeds, steps, GPU",
          "  - describe PS-CAB as gated modulation of encoder states (cross-attention memory), not a logit bias",
          "  - gate statistics and attention mass are mechanistic diagnostics of source-token utilisation, not proof of mechanism",
          "  - report the no-preposition noise floor next to every subset result",
          "  - seeds: lead with per-seed results, mean ± SD (N/A for one seed), same-seed paired tests; call the hierarchical "
          "bootstrap a seed-aware uncertainty characterisation (only 3 seeds)",
          "  - CV: 'five-fold grouped cross-validation' (folds of one dataset, not independent experiments)",
          "  - external sets: report BOTH official full scores and contamination-filtered scores, with the number removed",
          "  - limitation: zero overlap between our splits does not imply mBART-50 never saw these sentences during pretraining",
          "  - human evaluation: only claim it after >= 2 annotators filled the sheets and kappa is reported"]
    text = "\n".join(L)
    fname = "PRELIMINARY_EVIDENCE_SUMMARY.txt" if prelim else "FINAL_EVIDENCE_SUMMARY.txt"
    with open(f"{R}/{fname}", "w", encoding="utf-8") as f:
        f.write(text)
    print("\n" + text)


# ==============================================================================
# G2. PAPER FIGURES (FIG01-FIG21), QUALITATIVE EXAMPLES, ASSET INDEX
#     Every figure is built only from saved predictions / records (no new decoding),
#     saved as PDF (vector, for LaTeX) + PNG (300 dpi), and listed in PAPER_ASSETS.md.
# ==============================================================================
FIG_CAPTIONS = {}


def _safe(fn):
    def wrap(*a, **k):
        try:
            return fn(*a, **k)
        except Exception as e:
            print(f"[paper-fig] {fn.__name__} skipped: {type(e).__name__}: {e}")
    wrap.__name__ = fn.__name__
    return wrap


def _cap(name, text):
    FIG_CAPTIONS[name] = text


def _runs_of(ctx, cond):
    return [f"{cond}_s{s}" for s in ctx["seeds"] if f"{cond}_s{s}" in ctx["preds"]]


def _conds(ctx):
    return [c for c in COND_ORDER if _runs_of(ctx, c)]


def _sent_chrf(ctx, rid):
    cache = ctx.setdefault("_sent", {})
    if rid not in cache:
        M = StatMetric("chrf++")
        cache[rid] = np.array([M.score(row, 1) for row in ctx["stat_cache"]["chrf++"][rid]])
    return cache[rid]


def _subset_score(ctx, rid, mask, mn="chrf++"):
    M = StatMetric(mn)
    st = ctx["stat_cache"][mn][rid][mask]
    return M.score(st.sum(0), int(mask.sum()))


def _cond_subset(ctx, cond, mask, mn="chrf++"):
    v = [_subset_score(ctx, r, mask, mn) for r in _runs_of(ctx, cond)]
    return (float(np.mean(v)), float(np.std(v, ddof=1)) if len(v) > 1 else np.nan) if v else (np.nan, np.nan)


def _cond_delta(ctx, cond, mask, mn="chrf++", ref="baseline"):
    v = [_subset_score(ctx, f"{cond}_s{s}", mask, mn) - _subset_score(ctx, f"{ref}_s{s}", mask, mn)
         for s in ctx["seeds"] if f"{cond}_s{s}" in ctx["preds"] and f"{ref}_s{s}" in ctx["preds"]]
    return (float(np.mean(v)), float(np.std(v, ddof=1)) if len(v) > 1 else np.nan, len(v)) if v else (np.nan, np.nan, 0)


@_safe
def fig01_architecture(ctx):
    from matplotlib.patches import FancyBboxPatch
    fig, ax = plt.subplots(figsize=(10, 3.4))
    ax.set_xlim(0, 10); ax.set_ylim(0, 3.4); ax.axis("off")

    def box(x, y, w, h, txt, fc):
        ax.add_patch(FancyBboxPatch((x, y), w, h, boxstyle="round,pad=0.04", fc=fc, ec="k", lw=1))
        ax.text(x + w / 2, y + h / 2, txt, ha="center", va="center", fontsize=8)

    def arrow(x0, y0, x1, y1):
        ax.annotate("", xy=(x1, y1), xytext=(x0, y0), arrowprops=dict(arrowstyle="->", lw=1.2))

    box(0.1, 1.4, 1.4, 0.7, "English source\n(en_XX)", "#eeeeee")
    box(1.9, 1.4, 1.7, 0.7, "mBART-50 encoder\n(12 layers) → H", "#cfe2f3")
    box(4.0, 1.15, 2.5, 1.2, "PS-CAB\n g = σ(W_g H)\n H' = H + γ · g ⊙ (W_b H) ⊙ m", "#f4cccc")
    box(4.0, 2.6, 2.5, 0.6, "preposition mask m\n(spaCy ADP → subword offsets)", "#fff2cc")
    box(6.9, 1.4, 1.7, 0.7, "mBART-50 decoder\ncross-attn K,V = H'", "#cfe2f3")
    box(8.9, 1.4, 1.0, 0.7, "Telugu\n(te_IN)", "#eeeeee")
    arrow(1.5, 1.75, 1.9, 1.75); arrow(3.6, 1.75, 4.0, 1.75); arrow(5.25, 2.6, 5.25, 2.35)
    arrow(6.5, 1.75, 6.9, 1.75); arrow(8.6, 1.75, 8.9, 1.75)
    ax.text(5.25, 0.55, "Controls: −gate (g≡1) · −γ (γ≡1, W_b zero-init) · random mask (same #tokens) · all tokens (m≡1)\n"
                        "All variants are the identity map at step 0; ~2.1M added parameters (0.34% of 611M).",
            ha="center", va="center", fontsize=7.5, style="italic")
    save_fig("FIG01_architecture")
    _cap("FIG01_architecture", "PS-CAB inserts a gated, preposition-masked residual on the encoder states that serve "
         "as keys/values of every decoder cross-attention layer. Controls used in the ablation are listed below.")


@_safe
def fig02_main_results(ctx):
    f2 = ctx["f2"]
    mets = [m for m in ("chrF++", "BLEU", "COMET", "TER") if m in f2.columns and f2[m].notna().any()]
    conds = _conds(ctx)
    fig, ax = plt.subplots(1, len(mets), figsize=(3.4 * len(mets), 3.6))
    ax = np.atleast_1d(ax)
    zs = f2[f2["condition"] == "zeroshot_mbart50"]
    for a, m in zip(ax, mets):
        allv = []
        for i, c in enumerate(conds):
            v = f2[f2["condition"] == c][m].dropna().astype(float).values
            allv += list(v)
            a.bar(i, v.mean(), color=COND_COLOR[c], alpha=.6, yerr=v.std(ddof=1) if len(v) > 1 else 0, capsize=3)
            a.scatter([i] * len(v), v, c="k", s=12, zorder=3)
        a.set_xticks(range(len(conds))); a.set_xticklabels([COND_LABEL[c] for c in conds], rotation=35, ha="right")
        lo, hi = min(allv), max(allv)
        pad = max(0.5, (hi - lo) * 0.6)
        a.set_ylim(lo - pad, hi + pad)
        zt = f" (zero-shot: {float(zs[m].iloc[0]):.1f})" if len(zs) and pd.notna(zs[m].iloc[0]) else ""
        a.set_title(f"{m}{' ↓' if m == 'TER' else ' ↑'}{zt}", fontsize=9)
    save_fig("FIG02_main_results")
    _cap("FIG02_main_results", "Held-out test results. Bars: mean over seeds; error bars: SD; dots: individual seeds. "
         "The y-axis is zoomed to the fine-tuned systems; the zero-shot mBART-50 score is given in each title.")


@_safe
def fig03_forest(ctx):
    f5 = ctx["f5"]
    if not len(f5):
        return
    mets = [m for m in ("chrf++", "bleu", "comet") if m in set(f5["metric"])]
    fig, ax = plt.subplots(1, len(mets), figsize=(3.8 * len(mets), 3.4), sharey=True)
    ax = np.atleast_1d(ax)
    for a, m in zip(ax, mets):
        sub = f5[f5["metric"] == m].reset_index(drop=True)
        y = np.arange(len(sub))[::-1]
        for yi, (_, r) in zip(y, sub.iterrows()):
            other = r["comparison"].split(" - ")[1]
            a.errorbar(r["delta_mean_over_seeds"], yi, xerr=[[r["delta_mean_over_seeds"] - r["ci95_low"]],
                                                              [r["ci95_high"] - r["delta_mean_over_seeds"]]],
                       fmt="o", capsize=3, color=COND_COLOR.get(other, "k"))
        a.axvline(0, ls="--", c="gray")
        a.set_yticks(y); a.set_yticklabels([f"PS-CAB − {COND_LABEL.get(c.split(' - ')[1], c)}" for c in sub["comparison"]])
        a.set_title({"chrf++": "chrF++", "bleu": "spBLEU", "comet": "COMET×100"}[m]); a.set_xlabel("difference")
    save_fig("FIG03_effect_forest")
    _cap("FIG03_effect_forest", "Effect of PS-CAB relative to the baseline and each control on the test set: mean "
         "difference over seeds with a seed-and-sentence (hierarchical) bootstrap 95% interval. Intervals crossing 0 "
         "indicate no reliable difference.")


@_safe
def fig04_seed_deltas(ctx):
    f5b = ctx["f5b"]
    if not len(f5b):
        return
    mets = [m for m in ("chrf++", "bleu", "comet") if m in set(f5b["metric"])]
    fig, ax = plt.subplots(1, len(mets), figsize=(3.8 * len(mets), 3.4))
    ax = np.atleast_1d(ax)
    mk = ["o", "s", "^", "D"]
    for a, m in zip(ax, mets):
        sub = f5b[f5b["metric"] == m].reset_index(drop=True)
        for i, (_, r) in enumerate(sub.iterrows()):
            d = list(r["seed_deltas"]) if not isinstance(r["seed_deltas"], str) else json.loads(r["seed_deltas"])
            for j, v in enumerate(d):
                a.scatter(i + (j - 1) * 0.12, v, marker=mk[j % 4], color=COND_COLOR.get(r["comparison"].split(" - ")[1], "k"),
                          s=26, label=f"seed {ctx['seeds'][j]}" if i == 0 else None)
            a.hlines(np.mean(d), i - .3, i + .3, color="k")
        a.axhline(0, ls="--", c="gray")
        a.set_xticks(range(len(sub)))
        a.set_xticklabels([COND_LABEL.get(c.split(" - ")[1], c) for c in sub["comparison"]], rotation=30, ha="right")
        a.set_ylabel(f"PS-CAB minus condition"); a.set_title({"chrf++": "chrF++", "bleu": "spBLEU", "comet": "COMET×100"}[m])
    ax[0].legend(fontsize=7)
    save_fig("FIG04_per_seed_deltas")
    _cap("FIG04_per_seed_deltas", "Same-seed paired differences on the test set (markers = seeds, line = mean). "
         "Consistency of sign across seeds is the primary seed-level evidence.")


@_safe
def fig05_h2(ctx):
    counts = ctx["counts"]
    prep, nop = counts > 0, counts == 0
    if nop.sum() < 30:
        return
    fig, ax = plt.subplots(1, 2, figsize=(10, 3.6))
    rng = np.random.default_rng(3)
    M = StatMetric("chrf++")
    rows = []
    for j, s in enumerate(ctx["seeds"]):
        a_, b_ = f"pscab_s{s}", f"baseline_s{s}"
        if a_ not in ctx["preds"] or b_ not in ctx["preds"]:
            continue
        for k, (nm, mask) in enumerate((("with prepositions", prep), ("no prepositions", nop))):
            idx = np.where(mask)[0]
            A, B = ctx["stat_cache"]["chrf++"][a_], ctx["stat_cache"]["chrf++"][b_]
            obs = M.score(A[idx].sum(0), len(idx)) - M.score(B[idx].sum(0), len(idx))
            bt = []
            for _ in range(400):
                ii = rng.choice(idx, len(idx))
                bt.append(M.score(A[ii].sum(0), len(ii)) - M.score(B[ii].sum(0), len(ii)))
            lo, hi = np.percentile(bt, [2.5, 97.5])
            rows.append((s, nm, obs, lo, hi))
            ax[0].errorbar(k + (j - 1) * .15, obs, yerr=[[obs - lo], [hi - obs]], fmt="o", capsize=3,
                           color=["#1f77b4", "#ff7f0e", "#2ca02c"][j % 3], label=f"seed {s}" if k == 0 else None)
    ax[0].axhline(0, ls="--", c="gray")
    ax[0].set_xticks([0, 1]); ax[0].set_xticklabels([f"with prepositions\n(n={prep.sum()})",
                                                     f"no prepositions\n(n={nop.sum()}, module inactive)"])
    ax[0].set_ylabel("chrF++: PS-CAB − baseline"); ax[0].set_title("(a) Subset effects per seed (95% CI)"); ax[0].legend()
    f6 = ctx["f6"]
    if len(f6):
        sub = f6.reset_index(drop=True)
        y = np.arange(len(sub))
        ax[1].errorbar(sub["DiD"], y, xerr=[sub["DiD"] - sub["DiD_ci95_low"], sub["DiD_ci95_high"] - sub["DiD"]],
                       fmt="o", capsize=4, color="#d62728")
        ax[1].axvline(0, ls="--", c="gray"); ax[1].set_yticks(y); ax[1].set_yticklabels(sub["metric"])
        ax[1].set_xlabel("difference-in-differences"); ax[1].set_title("(b) H2: Δ(prep) − Δ(no-prep), seed-aware CI")
    save_fig("FIG05_H2_preposition_effect")
    _cap("FIG05_H2_preposition_effect", "Hypothesis H2. (a) PS-CAB − baseline on sentences with and without prepositions; "
         "on the latter the module is an exact no-op, so the gap measures training variance (noise floor). "
         "(b) Difference-in-differences with a seed-and-sentence bootstrap 95% interval.")


@_safe
def fig06_prep_type_heatmap(ctx):
    test = ctx["test"]
    words = [[w for _, _, w in _SPAN_CACHE[s]] for s in test["en"]]
    types = ["of", "in", "to", "for", "on", "with", "from", "by", "at", "as", "into", "about", "after", "over",
             "during", "between", "through", "under", "against", "without", "before", "within"]
    types = [t for t in types if sum(t in w for w in words) >= 30]
    conds = [c for c in _conds(ctx) if c != "baseline"]
    if not types or not conds:
        return
    Mx = np.full((len(types), len(conds)), np.nan)
    for i, t in enumerate(types):
        mask = np.array([t in w for w in words])
        for j, c in enumerate(conds):
            Mx[i, j] = _cond_delta(ctx, c, mask)[0]
    v = np.nanmax(np.abs(Mx)) or 1
    plt.figure(figsize=(1.3 * len(conds) + 2.5, 0.32 * len(types) + 1.4))
    plt.imshow(Mx, cmap="RdBu", vmin=-v, vmax=v, aspect="auto")
    plt.colorbar(label="Δ chrF++ vs baseline")
    for i in range(len(types)):
        for j in range(len(conds)):
            plt.text(j, i, f"{Mx[i, j]:+.2f}", ha="center", va="center", fontsize=6.5)
    ns = [sum(t in w for w in words) for t in types]
    plt.yticks(range(len(types)), [f"{t} (n={n})" for t, n in zip(types, ns)])
    plt.xticks(range(len(conds)), [COND_LABEL[c] for c in conds], rotation=30, ha="right")
    plt.grid(False); plt.title("Per-preposition effect (test, mean over seeds)")
    save_fig("FIG06_preposition_type_heatmap")
    _cap("FIG06_preposition_type_heatmap", "chrF++ difference to the baseline on test sentences containing each "
         "preposition (n ≥ 30), averaged over seeds. Sentences can contain several prepositions, so rows overlap.")


@_safe
def fig07_density_length(ctx):
    counts = ctx["counts"]
    test = ctx["test"]
    lens = test["en"].str.split().str.len().values
    groups = [("#prep", [("0", counts == 0), ("1", counts == 1), ("2", counts == 2), ("3", counts == 3), ("4+", counts >= 4)]),
              ("source words", [("1-10", lens <= 10), ("11-20", (lens > 10) & (lens <= 20)),
                                ("21-30", (lens > 20) & (lens <= 30)), (">30", lens > 30)])]
    fig, ax = plt.subplots(2, 2, figsize=(10, 6.2))
    for col, (gname, buckets) in enumerate(groups):
        buckets = [(b, m) for b, m in buckets if m.sum() >= 30]
        x = np.arange(len(buckets))
        for c in _conds(ctx):
            ys = [_cond_subset(ctx, c, m)[0] for _, m in buckets]
            ax[0, col].plot(x, ys, marker="o", color=COND_COLOR[c], label=COND_LABEL[c])
            if c != "baseline":
                ds = [_cond_delta(ctx, c, m) for _, m in buckets]
                ax[1, col].errorbar(x, [d[0] for d in ds], yerr=[0 if np.isnan(d[1]) else d[1] for d in ds],
                                    marker="o", capsize=3, color=COND_COLOR[c], label=COND_LABEL[c])
        for r in (0, 1):
            ax[r, col].set_xticks(x); ax[r, col].set_xticklabels([f"{b}\n(n={m.sum()})" for b, m in buckets], fontsize=7)
        ax[1, col].axhline(0, ls="--", c="gray")
        ax[0, col].set_ylabel("chrF++"); ax[1, col].set_ylabel("Δ chrF++ vs baseline (± SD seeds)")
        ax[1, col].set_xlabel(gname)
    ax[0, 0].set_title("(a) by preposition density"); ax[0, 1].set_title("(b) by source length")
    ax[0, 0].legend(fontsize=6.5)
    save_fig("FIG07_density_and_length")
    _cap("FIG07_density_and_length", "Test chrF++ by number of prepositions in the source (left) and by source "
         "length (right). Top: absolute scores (mean over seeds); bottom: difference to the baseline (± SD over seeds).")


@_safe
def fig08_sentence_level(ctx):
    b_runs, p_runs = _runs_of(ctx, "baseline"), _runs_of(ctx, "pscab")
    pairs = [(f"pscab_s{s}", f"baseline_s{s}") for s in ctx["seeds"]
             if f"pscab_s{s}" in ctx["preds"] and f"baseline_s{s}" in ctx["preds"]]
    if not pairs:
        return
    d = np.mean([_sent_chrf(ctx, a) - _sent_chrf(ctx, b) for a, b in pairs], axis=0)
    base = np.mean([_sent_chrf(ctx, b) for _, b in pairs], axis=0)
    ps = np.mean([_sent_chrf(ctx, a) for a, _ in pairs], axis=0)
    fig, ax = plt.subplots(1, 3, figsize=(13, 3.7))
    ax[0].hist(np.clip(d, -30, 30), bins=np.arange(-30, 31, 1), color="#d62728", alpha=.8)
    w, l, t = (d > .5).mean(), (d < -.5).mean(), (np.abs(d) <= .5).mean()
    ax[0].set_title(f"(a) per-sentence Δ chrF++ (PS-CAB − baseline)\nwin {w:.1%} · tie {t:.1%} · loss {l:.1%}")
    ax[0].set_xlabel("Δ chrF++ (mean over seeds, clipped ±30)"); ax[0].set_ylabel("sentences")
    hb = ax[1].hexbin(base, ps, gridsize=40, mincnt=1, bins="log", cmap="viridis")
    ax[1].plot([0, 100], [0, 100], "w--", lw=.8)
    ax[1].set_xlabel("baseline sentence chrF++"); ax[1].set_ylabel("PS-CAB sentence chrF++")
    ax[1].set_title("(b) sentence-level agreement"); fig.colorbar(hb, ax=ax[1], label="log10 count")
    conds = [c for c in _conds(ctx) if c != "baseline"]
    W, T, L = [], [], []
    for c in conds:
        pr = [(f"{c}_s{s}", f"baseline_s{s}") for s in ctx["seeds"] if f"{c}_s{s}" in ctx["preds"] and f"baseline_s{s}" in ctx["preds"]]
        dd = np.mean([_sent_chrf(ctx, a) - _sent_chrf(ctx, b) for a, b in pr], axis=0)
        W.append((dd > .5).mean() * 100); T.append((np.abs(dd) <= .5).mean() * 100); L.append((dd < -.5).mean() * 100)
    x = np.arange(len(conds))
    ax[2].bar(x, W, color="#2ca02c", label="win"); ax[2].bar(x, T, bottom=W, color="#cccccc", label="tie (|Δ|≤0.5)")
    ax[2].bar(x, L, bottom=np.array(W) + np.array(T), color="#d62728", label="loss")
    ax[2].set_xticks(x); ax[2].set_xticklabels([COND_LABEL[c] for c in conds], rotation=30, ha="right")
    ax[2].set_ylabel("% of test sentences"); ax[2].set_title("(c) vs baseline"); ax[2].legend(fontsize=6.5)
    save_fig("FIG08_sentence_level")
    _cap("FIG08_sentence_level", "Sentence-level analysis on the test set. (a) Distribution of per-sentence chrF++ "
         "differences, PS-CAB − baseline (averaged over seeds). (b) Sentence-level scores of the two systems. "
         "(c) Win/tie/loss rates of every condition against the baseline.")


@_safe
def fig09_output_length(ctx):
    refs = ctx["refs"]
    B = bleu_metric()
    conds = ["zeroshot_mbart50"] + _conds(ctx)
    fig, ax = plt.subplots(1, 2, figsize=(11, 3.6))
    data, ratios = [], []
    for c in conds:
        rids = [r for r in ctx["runs"] if ctx["recs"][r]["condition"] == c]
        rr = []
        for rid in rids:
            sc = B.corpus_score(ctx["preds"][rid], [refs])
            rr.append(sc.sys_len / sc.ref_len)
        ratios.append(rr)
        data.append(np.mean([[len(h) / max(len(r), 1) for h, r in zip(ctx["preds"][rid], refs)] for rid in rids], axis=0))
    ax[0].boxplot([np.clip(x, 0, 3) for x in data], showfliers=False)
    ax[0].axhline(1, ls="--", c="gray")
    ax[0].set_xticks(range(1, len(conds) + 1)); ax[0].set_xticklabels([COND_LABEL[c] for c in conds], rotation=30, ha="right")
    ax[0].set_ylabel("hyp/ref characters"); ax[0].set_title("(a) per-sentence length ratio")
    for i, (c, rr) in enumerate(zip(conds, ratios)):
        ax[1].bar(i, np.mean(rr), color=COND_COLOR[c], alpha=.6)
        ax[1].scatter([i] * len(rr), rr, c="k", s=12, zorder=3)
    ax[1].axhline(1, ls="--", c="gray")
    ax[1].set_xticks(range(len(conds))); ax[1].set_xticklabels([COND_LABEL[c] for c in conds], rotation=30, ha="right")
    allr = [v for rr in ratios for v in rr]
    ax[1].set_ylim(min(allr + [1]) - .03, max(allr + [1]) + .03)
    ax[1].set_ylabel("corpus hyp/ref (spBLEU tokens)"); ax[1].set_title("(b) corpus length ratio (dots = seeds)")
    save_fig("FIG09_output_length")
    _cap("FIG09_output_length", "Output-length behaviour on the test set. Systematic length differences inflate or deflate "
         "BLEU through the brevity penalty without a change in chrF++; this figure documents that all compared systems "
         "decode with the same length behaviour (single pinned environment).")


@_safe
def fig10_metric_agreement(ctx):
    f2 = ctx["f2"]
    f = f2[f2["condition"] != "zeroshot_mbart50"]
    pairs = [("chrF++", "BLEU")] + ([("chrF++", "COMET")] if "COMET" in f and f["COMET"].notna().all() else []) + [("chrF++", "TER")]
    fig, ax = plt.subplots(1, len(pairs), figsize=(4 * len(pairs), 3.5))
    ax = np.atleast_1d(ax)
    for a, (x, y) in zip(ax, pairs):
        for c in _conds(ctx):
            sub = f[f["condition"] == c]
            a.scatter(sub[x], sub[y], color=COND_COLOR[c], label=COND_LABEL[c], s=30)
        tau = sps.kendalltau(f[x].astype(float), f[y].astype(float))
        a.set_xlabel(x); a.set_ylabel(y); a.set_title(f"Kendall τ = {tau.statistic:.2f} (p={tau.pvalue:.2f}, n={len(f)})")
    ax[0].legend(fontsize=6.5)
    save_fig("FIG10_metric_agreement")
    _cap("FIG10_metric_agreement", "Agreement between automatic metrics across all fine-tuned runs (test set). "
         "Low rank correlation between metrics means differences between systems are within metric noise.")


@_safe
def fig11_gates(ctx):
    s0 = ctx["seeds"][0]
    items = []
    for c in COND_ORDER:
        f = f"{DIRS['analysis']}/{c}_s{s0}_gate_samples.npz"
        if os.path.exists(f):
            z = np.load(f)
            if z["trained_prep"].size:
                items.append((c, z))
    if not items:
        return
    fig, ax = plt.subplots(1, len(items), figsize=(3.4 * len(items), 3.2), sharey=True)
    ax = np.atleast_1d(ax)
    bins = np.linspace(0, 1, 50)
    recs = ctx["recs"]
    for a, (c, z) in zip(ax, items):
        a.hist(z["trained_other"], bins, density=True, alpha=.45, color="#1f77b4", label="trained · other")
        a.hist(z["trained_prep"], bins, density=True, alpha=.45, color="#d62728", label="trained · preposition")
        if z["init_prep"].size:
            a.hist(z["init_other"], bins, density=True, histtype="step", color="#1f77b4", ls="--", label="init · other")
            a.hist(z["init_prep"], bins, density=True, histtype="step", color="#d62728", ls="--", label="init · preposition")
        dg = (recs.get(f"{c}_s{s0}") or {}).get("diagnostics", {})
        a.set_title(f"{COND_LABEL[c]}\nAUC trained {dg.get('gate_trained_auc_prep_vs_other', float('nan')):.2f} "
                    f"/ init {dg.get('gate_init_auc_prep_vs_other', float('nan')):.2f}", fontsize=8)
        a.set_xlabel("mean gate value")
    ax[0].legend(fontsize=6)
    save_fig("FIG11_gate_distributions")
    _cap("FIG11_gate_distributions", "Gate values g = σ(W_g H) at preposition vs other content tokens (validation, "
         f"seed {s0}), trained vs the same gate at initialisation. AUC = 0.5 means the gate does not distinguish prepositions.")


@_safe
def fig12_module_magnitude(ctx):
    rows = [(r["condition"], r["seed"], r["diagnostics"]) for r in ctx["recs"].values() if r.get("diagnostics")]
    if not rows:
        return
    conds = [c for c in COND_ORDER if any(r[0] == c for r in rows)]
    fig, ax = plt.subplots(1, 2, figsize=(9.5, 3.4))
    for k, (key, title) in enumerate((("gamma", "(a) learned scale γ"),
                                      ("bias_to_hidden_norm_ratio_mean", "(b) ‖γ·g·W_bH‖ / ‖H‖ on masked tokens"))):
        for i, c in enumerate(conds):
            v = [r[2].get(key) for r in rows if r[0] == c and r[2].get(key) is not None]
            ax[k].bar(i, np.mean(v), color=COND_COLOR[c], alpha=.6)
            ax[k].scatter([i] * len(v), v, c="k", s=14, zorder=3)
        ax[k].axhline(0, c="gray", lw=.8)
        ax[k].set_xticks(range(len(conds))); ax[k].set_xticklabels([COND_LABEL[c] for c in conds], rotation=30, ha="right")
        ax[k].set_title(title)
    save_fig("FIG12_module_magnitude")
    _cap("FIG12_module_magnitude", "Magnitude of the PS-CAB intervention (validation; dots = seeds). A large relative "
         "perturbation of the encoder states combined with unchanged translation quality indicates that the decoder "
         "learns to discount the added signal.")


@_safe
def fig13_attention_robustness(ctx):
    recs = [r for r in ctx["recs"].values() if r.get("kind") == "main" and r.get("attention")]
    rob = [r for r in ctx["recs"].values() if r.get("kind") == "main" and r.get("robustness_val")]
    if not recs and not rob:
        return
    fig, ax = plt.subplots(1, 2, figsize=(11, 3.6))
    conds = [c for c in COND_ORDER if any(r["condition"] == c for r in recs)]
    for i, c in enumerate(conds):
        v = [r["attention"]["prep_mass_last_layer_over_uniform"] for r in recs if r["condition"] == c]
        ax[0].bar(i, np.mean(v), color=COND_COLOR[c], alpha=.7)
        ax[0].scatter([i] * len(v), v, c="k", s=12, zorder=3)
    ax[0].axhline(1, ls="--", c="k", lw=.8)
    ax[0].set_xticks(range(len(conds))); ax[0].set_xticklabels([COND_LABEL[c] for c in conds], rotation=30, ha="right")
    ax[0].set_ylabel("attention mass ÷ uniform share"); ax[0].set_title("(a) last-layer cross-attention on prepositions")
    perts = list(PERTURBATIONS)
    rc = [c for c in COND_ORDER if any(r["condition"] == c for r in rob)]
    w = 0.8 / max(len(rc), 1)
    for i, c in enumerate(rc):
        vals = []
        for p_ in perts:
            v = [q["delta_chrf++"] for r in rob if r["condition"] == c for q in r["robustness_val"] if q["perturbation"] == p_]
            vals.append(np.mean(v) if v else np.nan)
        ax[1].bar(np.arange(len(perts)) + i * w, vals, w, color=COND_COLOR[c], label=COND_LABEL[c])
    ax[1].axhline(0, c="gray", lw=.8)
    ax[1].set_xticks(np.arange(len(perts)) + w * (len(rc) - 1) / 2); ax[1].set_xticklabels(perts)
    ax[1].set_ylabel("Δ chrF++ vs unperturbed"); ax[1].set_title("(b) robustness to source perturbations"); ax[1].legend(fontsize=6.5)
    save_fig("FIG13_attention_and_robustness")
    _cap("FIG13_attention_and_robustness", "(a) Teacher-forced cross-attention mass on preposition tokens in the last "
         "decoder layer, divided by their uniform share (validation). (b) chrF++ change under source perturbations "
         "(validation subset).")


@_safe
def fig14_cv(ctx):
    k, seed = CFG["cv"]["k"], CFG["cv"]["seed"]
    rows = []
    for f in range(k):
        for m in CFG["cv"]["models"]:
            r = jload(run_record_path(f"cv{f}_{m}_s{seed}"))
            if r and r.get("status") == "complete":
                rows.append({"fold": f, "model": m, **{x: r["heldout_metrics"][y] for x, y in (("chrF++", "chrf++"), ("BLEU", "bleu"))}})
    df = pd.DataFrame(rows)
    if not len(df) or df["fold"].nunique() < 2:
        return
    folds = sorted(df.groupby("fold").filter(lambda g: len(g) == 2)["fold"].unique())
    fig, ax = plt.subplots(1, 3, figsize=(12, 3.5))
    for a, m in zip(ax[:2], ("chrF++", "BLEU")):
        for f in folds:
            b = df[(df.fold == f) & (df.model == "baseline")][m].values[0]
            p = df[(df.fold == f) & (df.model == "pscab")][m].values[0]
            a.plot([0, 1], [b, p], marker="o", color="#2ca02c" if p > b else "#d62728")
            a.annotate(f"f{f}", (1.03, p), fontsize=7)
        a.set_xticks([0, 1]); a.set_xticklabels(["Baseline", "PS-CAB"]); a.set_ylabel(f"held-out {m}")
    d = [df[(df.fold == f) & (df.model == "pscab")]["chrF++"].values[0] - df[(df.fold == f) & (df.model == "baseline")]["chrF++"].values[0]
         for f in folds]
    ax[2].bar([f"fold {f}" for f in folds], d, color=["#2ca02c" if x > 0 else "#d62728" for x in d])
    ax[2].axhline(0, c="gray"); ax[2].axhline(np.mean(d), ls="--", c="k", label=f"mean {np.mean(d):+.2f}")
    ax[2].set_ylabel("Δ chrF++ (PS-CAB − baseline)"); ax[2].legend()
    ax[0].set_title("(a) chrF++ per fold"); ax[1].set_title("(b) spBLEU per fold"); ax[2].set_title("(c) paired differences")
    save_fig("FIG14_cross_validation")
    _cap("FIG14_cross_validation", f"{len(folds)}-fold grouped cross-validation over the train+validation pool "
         "(folds of one dataset, one seed). Lines connect the two systems on the same held-out fold.")


@_safe
def fig15_errors_cost(ctx):
    f9, f10 = ctx["f9"], ctx["f10"]
    fig, ax = plt.subplots(1, 3, figsize=(13, 3.5))
    if len(f9):
        f9 = f9.copy()
        f9["condition"] = f9["run"].map(lambda r: ctx["recs"][r]["condition"])
        cols = [c for c in f9.columns if c.endswith("%")]
        g = f9.groupby("condition")[cols].mean()
        conds = [c for c in ["zeroshot_mbart50"] + COND_ORDER if c in g.index and c != "zeroshot_mbart50"]
        w = 0.8 / max(len(conds), 1)
        for i, c in enumerate(conds):
            ax[0].bar(np.arange(len(cols)) + i * w, g.loc[c, cols].values, w, color=COND_COLOR[c], label=COND_LABEL[c])
        ax[0].set_xticks(np.arange(len(cols)) + w * (len(conds) - 1) / 2)
        ax[0].set_xticklabels([c.replace("_%", "").replace("_", " ") for c in cols], rotation=25, ha="right", fontsize=7)
        ax[0].set_ylabel("% of test outputs"); ax[0].set_title("(a) automatic error flags"); ax[0].legend(fontsize=6)
    if len(f10):
        f10 = f10.copy()
        f10["condition"] = f10["run"].map(lambda r: ctx["recs"][r]["condition"])
        g = f10.groupby("condition")[["sec_per_step", "infer_sent_per_s", "added_params"]].mean()
        conds = [c for c in COND_ORDER if c in g.index]
        ax[1].bar(range(len(conds)), g.loc[conds, "sec_per_step"], color=[COND_COLOR[c] for c in conds])
        ax[1].set_title("(b) training time per optimizer step (s)")
        ax[2].bar(range(len(conds)), g.loc[conds, "infer_sent_per_s"], color=[COND_COLOR[c] for c in conds])
        ax[2].set_title("(c) inference speed (sentences/s, beam 4)")
        for a in ax[1:]:
            a.set_xticks(range(len(conds))); a.set_xticklabels([COND_LABEL[c] for c in conds], rotation=30, ha="right")
        for i, c in enumerate(conds):
            ax[1].annotate(f"+{g.loc[c, 'added_params'] / 1e6:.2f}M", (i, g.loc[c, "sec_per_step"]), ha="center",
                           va="bottom", fontsize=6.5)
    save_fig("FIG15_errors_and_cost")
    _cap("FIG15_errors_and_cost", "(a) Automatic error flags on the test set (heuristics, not a linguistic taxonomy). "
         "(b) Training cost per optimizer step with the number of added parameters. (c) Inference throughput.")


@_safe
def fig16_learning_curves(ctx):
    plt.figure(figsize=(7.5, 4))
    for c in _conds(ctx):
        cs = [x for x in (_curve(r) for r in _runs_of(ctx, c)) if x is not None]
        if not cs:
            continue
        L = min(len(x) for x in cs)
        Mx = np.stack([x["val_nll"].values[:L] for x in cs])
        st = cs[0]["step"].values[:L]
        plt.plot(st, Mx.mean(0), marker="o", ms=3, color=COND_COLOR[c], label=f"{COND_LABEL[c]} (n={len(cs)})")
        if len(cs) > 1:
            plt.fill_between(st, Mx.min(0), Mx.max(0), color=COND_COLOR[c], alpha=.15)
    plt.axvline(2000, ls=":", c="k", lw=.8)
    plt.xlabel("optimizer step (effective batch 64)"); plt.ylabel("validation NLL / token"); plt.legend(ncol=2)
    plt.title("Learning curves (identical step budget; band = range over seeds)")
    save_fig("FIG16_learning_curves")
    _cap("FIG16_learning_curves", "Validation NLL during training for every condition (mean over seeds; band = min–max "
         "over seeds). All runs share the same 3,000-step budget and checkpoint selection by validation NLL.")


@_safe
def fig17_external(ctx):
    tabs = [p for p in (f"{DIRS['tables']}/F13_external_contamination_filtered.csv",
                        f"{DIRS['tables']}/F13b_external_official_full.csv") if os.path.exists(p)]
    if not tabs:
        return
    df = pd.concat([pd.read_csv(p) for p in tabs], ignore_index=True)
    df["condition"] = df["run"].map(lambda r: (ctx["recs"].get(r) or {}).get("condition", parse_run_id(r)[0] if "_s" in r else r))
    sets = df[["set", "version"]].drop_duplicates().values.tolist()
    fig, ax = plt.subplots(1, len(sets), figsize=(4.5 * len(sets), 3.5))
    ax = np.atleast_1d(ax)
    for a, (st, ver) in zip(ax, sets):
        sub = df[(df["set"] == st) & (df["version"] == ver)]
        g = sub.groupby("condition")["chrF++"]
        conds = [c for c in ["zeroshot_mbart50"] + COND_ORDER if c in g.groups]
        a.bar(range(len(conds)), [g.mean()[c] for c in conds], color=[COND_COLOR[c] for c in conds],
              yerr=[g.std().fillna(0)[c] for c in conds], capsize=3)
        a.set_xticks(range(len(conds))); a.set_xticklabels([COND_LABEL[c] for c in conds], rotation=30, ha="right")
        a.set_title(f"{st} ({ver})", fontsize=8); a.set_ylabel("chrF++")
    save_fig("FIG17_external_benchmarks")
    _cap("FIG17_external_benchmarks", "Out-of-domain evaluation on external benchmarks, reported both as the official "
         "full set and after removing sentences that overlap with our corpus.")


@_safe
def qualitative_examples(ctx, n_each=3):
    s0 = ctx["seeds"][0]
    a_, b_ = f"pscab_s{s0}", f"baseline_s{s0}"
    if a_ not in ctx["preds"] or b_ not in ctx["preds"]:
        return
    test = ctx["test"]
    sa, sb = _sent_chrf(ctx, a_), _sent_chrf(ctx, b_)
    d = sa - sb
    has = ctx["counts"] > 0
    cand = np.where(has & (test["en"].str.split().str.len().values <= 25))[0]
    order = cand[np.argsort(d[cand])]
    rng = np.random.default_rng(11)
    pick = [("PS-CAB better", i) for i in order[::-1][:n_each]] + [("baseline better", i) for i in order[:n_each]] + \
           [("random", i) for i in rng.choice(cand, n_each, replace=False)]
    rows = []
    for kind, i in pick:
        preps = ", ".join(w for _, _, w in _SPAN_CACHE[test["en"].iloc[i]])
        rows.append({"type": kind, "prepositions": preps, "source_en": test["en"].iloc[i], "reference_te": test["te"].iloc[i],
                     "baseline": ctx["preds"][b_][i], "pscab": ctx["preds"][a_][i],
                     "chrF++_baseline": round(sb[i], 1), "chrF++_pscab": round(sa[i], 1)})
    t = pd.DataFrame(rows)
    t.to_csv(f"{DIRS['tables']}/T_qualitative_examples.csv", index=False)
    with open(f"{DIRS['tables']}/T_qualitative_examples.md", "w", encoding="utf-8") as f:
        f.write(t.to_markdown(index=False) if hasattr(t, "to_markdown") else t.to_string(index=False))
    try:
        with open(f"{DIRS['tables']}/T_qualitative_examples.tex", "w", encoding="utf-8") as f:
            f.write("% Requires a Telugu-capable font, e.g. \\usepackage{fontspec}\\newfontfamily\\telugufont{Noto Sans Telugu}\n")
            f.write(t.to_latex(index=False, escape=True, caption="Qualitative examples (test, seed %d)" % s0,
                               label="tab:qualitative"))
    except Exception:
        pass
    print(f"\n[qualitative] {len(t)} examples (largest gains, largest losses, random) -> tables/T_qualitative_examples.*")


def write_assets_index():
    figs = sorted(set(os.path.splitext(os.path.basename(p))[0] for p in glob.glob(f"{DIRS['figures']}/*.pdf")))
    tabs = sorted(set(os.path.splitext(os.path.basename(p))[0] for p in glob.glob(f"{DIRS['tables']}/*.csv")))
    extra = {
        "D_fig_lengths": "Sentence-length distributions of the frozen splits (English source, Telugu target).",
        "D_fig_prepositions": "(a) Number of prepositions per source sentence by split; (b) most frequent prepositions.",
        "D_fig_en_te_length_hexbin": "English vs Telugu word counts (train): Telugu is agglutinative.",
        "DIAG_fig_env_drift": "Appendix: output length and BLEU grouped by transformers version (motivates the pinned environment).",
        "V_fig_paired_deltas": "Validation: PS-CAB minus every other condition, per seed, with bootstrap 95% CI.",
    }
    L = ["# Paper assets (auto-generated)", "", "Figures are in `figures/` (PDF = vector for LaTeX, PNG = 300 dpi).",
         "Tables are in `tables/` (CSV + LaTeX).", "", "## Main-paper figures"]
    for f in figs:
        if f.startswith("FIG"):
            L.append(f"- **{f}** — {FIG_CAPTIONS.get(f, '')}")
    L += ["", "## Data / validation / appendix figures"]
    for f in figs:
        if not f.startswith("FIG"):
            L.append(f"- {f} — {extra.get(f, '')}")
    L += ["", "## Tables"] + [f"- {t}" for t in tabs]
    with open(f"{R}/PAPER_ASSETS.md", "w", encoding="utf-8") as fh:
        fh.write("\n".join(L))
    print(f"[assets] {len(figs)} figures, {len(tabs)} tables indexed -> {R}/PAPER_ASSETS.md")


def paper_figures(ctx):
    print(f"\n{'#'*72}\nPAPER FIGURES (test set; built from saved predictions)\n{'#'*72}")
    apply_paper_style()
    for fn in (fig01_architecture, fig02_main_results, fig03_forest, fig04_seed_deltas, fig05_h2, fig06_prep_type_heatmap,
               fig07_density_length, fig08_sentence_level, fig09_output_length, fig10_metric_agreement, fig11_gates,
               fig12_module_magnitude, fig13_attention_robustness, fig14_cv, fig15_errors_cost, fig16_learning_curves,
               fig17_external, qualitative_examples):
        fn(ctx)
    write_assets_index()


# ==============================================================================
# H. DRIVER
# ==============================================================================
build_cv_folds()
if DO_CV:
    run_next_cv_runs(CV_RUNS_THIS_SESSION)
cv_report()
if all(run_done(r) for r in cv_queue()):
    print("[cv] CROSS-VALIDATION COMPLETE.")
else:
    print(f"[cv] {sum(run_done(r) for r in cv_queue())}/{len(cv_queue())} CV runs complete; re-run Part 1 + Part 3 next session.")
final_test_evaluation()
human_eval_agreement()
print(f"\nAll outputs in {R}/ (tables: CSV + LaTeX, figures: PDF + PNG). Session time left {session_hours_left():.1f}h.")

[cv-queue] 0/10 complete; next: ['cv0_baseline_s42', 'cv0_pscab_s42']


HTTP Request: HEAD https://huggingface.co/facebook/mbart-large-50-many-to-many-mmt/resolve/main/tokenizer_config.json "HTTP/1.1 307 Temporary Redirect"
HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/facebook/mbart-large-50-many-to-many-mmt/e30b6cb8eb0d43a0b73cab73c7676b9863223a30/tokenizer_config.json "HTTP/1.1 200 OK"


[cv] cv0_baseline_s42: train=26800 early-stop=2000 held-out=7200

TRAIN cv0_baseline_s42: {'use_pscab': False, 'gate': 'learned', 'gamma': 'learned', 'mask': 'true'}


HTTP Request: GET https://huggingface.co/api/resolve-cache/models/facebook/mbart-large-50-many-to-many-mmt/e30b6cb8eb0d43a0b73cab73c7676b9863223a30/tokenizer_config.json "HTTP/1.1 200 OK"


tokenizer_config.json:   0%|          | 0.00/529 [00:00<?, ?B/s]

HTTP Request: GET https://huggingface.co/api/models/facebook/mbart-large-50-many-to-many-mmt/tree/main/additional_chat_templates?recursive=false&expand=false "HTTP/1.1 404 Not Found"
HTTP Request: GET https://huggingface.co/api/models/facebook/mbart-large-50-many-to-many-mmt/tree/main?recursive=true&expand=false "HTTP/1.1 200 OK"
HTTP Request: HEAD https://huggingface.co/facebook/mbart-large-50-many-to-many-mmt/resolve/main/sentencepiece.bpe.model "HTTP/1.1 302 Found"
HTTP Request: GET https://huggingface.co/api/models/facebook/mbart-large-50-many-to-many-mmt/xet-read-token/e30b6cb8eb0d43a0b73cab73c7676b9863223a30 "HTTP/1.1 200 OK"


... (8 outputs left out)

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/516 [00:00<?, ?it/s]

HTTP Request: HEAD https://huggingface.co/facebook/mbart-large-50-many-to-many-mmt/resolve/main/generation_config.json "HTTP/1.1 307 Temporary Redirect"
HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/facebook/mbart-large-50-many-to-many-mmt/e30b6cb8eb0d43a0b73cab73c7676b9863223a30/generation_config.json "HTTP/1.1 200 OK"
HTTP Request: GET https://huggingface.co/api/resolve-cache/models/facebook/mbart-large-50-many-to-many-mmt/e30b6cb8eb0d43a0b73cab73c7676b9863223a30/generation_config.json "HTTP/1.1 200 OK"


generation_config.json:   0%|          | 0.00/261 [00:00<?, ?B/s]